
# INSPIRE Perioperative Mortality — Multimodal, Organ-System DNN
### A complete, reasoning-first Kaggle notebook

**What this notebook is.** An end-to-end, runnable pipeline for predicting 30-day
post-surgical mortality from the INSPIRE Korean perioperative dataset (Lee et al.,
*Scientific Data*, 2024), built around a **multimodal, organ-system-separated deep
neural network**: each physiological system (renal, cardiovascular, respiratory,
metabolic/hepatic, haematology, neurological) gets its own time-series encoder and its
own embedding; two systems that don't have dedicated lab/vital panels in this dataset —
**gastrointestinal** and **musculoskeletal** — are added as diagnosis/procedure-driven
branches; a **cardio-renal coupling** term implements the clinical fact that renal and
cardiovascular status move together; and everything is fused into a single mortality
prediction that can still be read off system-by-system.

**Why theory comes before code (as requested).** Every data-cleaning, imputation,
sampling and architecture decision below is a genuine fork with real trade-offs — picking
one silently would hide the reasoning you asked to be able to interrogate and experiment
with yourself. So **Part 1** is a self-contained explanation of every method this notebook
offers, with the book/paper it comes from, *before* a single line of data-loading code
runs. **Part 2 onward** is the runnable pipeline, and every place where Part 1 offered more
than one option has a `CONFIG` flag so you can switch strategies and re-run without editing
code.

**Data used here.** This notebook was built against two files you supplied:

| File | Contents |
|---|---|
| `inspire-analysis-thrisha-main.zip` | An existing analysis repo for this exact dataset/problem — docs, EDA, ICD-10 chapter mapper, a working (if partial) DNN pipeline. This notebook reuses its domain research and re-implements the pipeline standalone (Kaggle-portable, no local imports). |
| `inspire_subjects_small.zip` | 30 patients (10 died / 20 survived), one JSON per patient, each with `labs`, `vitals` (intra-op), `ward_vitals`, `operations`, `diagnoses` (ICD-10-CM), `medications` (ATC-coded). This is a **development subset** — the code below is written to scale unchanged to the full ~99,886-patient INSPIRE cohort if you upload it as a Kaggle dataset (just change `SUBJECTS_DIR` in the config cell). |

**Meeting notes this notebook directly implements** (from your last meeting, restated so
the mapping to code is traceable):

1. **Add Gastrointestinal and Musculoskeletal systems** → §1.2 and §6.7–6.8: both are built
   from ICD-10 chapters (XI and XIII) plus surgical department, since — importantly —
   *this dataset has no dedicated GI or MSK lab/vital panel*, which is a data-availability
   finding worth understanding before writing any code (§1.2.3).
2. **Add cardiovascular parameters into the renal system; renal is directly proportional
   to cardiovascular** → §1.6.2 (cardiorenal syndrome, Ronco et al. classification) and
   §6.9 (implemented as an explicit renal↔cardiac coupling input + interaction features).
3. **Features learned separately per system, then jointly learned/fused for prediction**
   → §1.5 (multimodal fusion theory) and §9 (the model itself).
4. **Number of operations in the same area matters** → §6.10 (operation-count and
   recency features, per department/system).
5. **Exceptions — e.g. 6-month mandatory wait after cardiovascular surgery before the next
   operation** → §6.11 (rule-based clinical-exception flags, kept separate from learned
   features on purpose — see reasoning there).
6. **Pre-operative, intra-operative, post-operative windows all matter** → §1.6.3 and the
   `TIME_WINDOW` config flag threaded through §6 and §7.
7. **ICD-10 code usage** → §1.2, §6.6 (HFRS, chapter one-hots, chapter counts, a learned
   embedding option).

**How to run this on Kaggle.**
1. Create a Kaggle dataset from `inspire_subjects_small` (and, when you have it, the full
   `subjects` folder) — Add Data → Upload → point at the folder with `died/` and
   `survived/` subfolders.
2. Optionally also upload the `codes/` folder (`icd10.json.gz`, `WHO_ATC-DDD_*.csv`) from
   the analysis repo — the notebook works without it (built-in fallbacks), but gives finer
   ICD-10 descriptions and ATC therapeutic-class names if present.
3. Attach both datasets to this notebook, run Part 2's config cell first, adjust the two
   paths if Kaggle's auto-detect doesn't find them, then **Run All**.



# Part 1 — Theory (read this before running the pipeline)

This part has five sections, each ending in a **decision table** that maps directly onto a
`CONFIG` flag used later. Nothing here is code — it's the reasoning layer the rest of the
notebook is built on.

1.1 Why organise by organ system at all?
1.2 ICD-10: what it is, how this project uses it, and why GI/MSK are diagnosis-driven
1.3 Missing data: mechanisms and the imputation methods on offer
1.4 Class imbalance: why it matters here, and the sampling methods on offer
1.5 Multimodal deep learning: encoders, fusion, and joint embedding learning
1.6 Project-specific architecture decisions from the meeting notes



## 1.1 Why organise by organ system at all?

The naive approach — concatenate every lab, vital, and ward-vital value into one long
vector and feed it to one encoder — is what most published EHR mortality models do
(including the closest benchmark for this exact task, Shickel et al. 2023, *Scientific
Reports*, AUROC 0.92 on 56,242 patients). It works, but it produces **one opaque number**:
you cannot ask *why* the model thinks a patient is high-risk in a way a clinician could
act on.

**The alternative used in this notebook** — route each raw signal to the physiological
system it belongs to, encode each system separately, and only fuse at the end — buys three
things:

- **Interpretability that's structural, not post-hoc.** SHAP/LIME-style explanations fit a
  simple surrogate model *around* a finished black box after training; they are known to be
  unstable for correlated, temporally-dependent features (exactly what labs/vitals are).
  A per-system embedding, by contrast, is an actual intermediate computation the network
  produced — "renal embedding says CRITICAL" is a fact about what the network computed, not
  a story told about it afterwards.
- **A defensible sample-efficiency trade.** Splitting one 30-feature encoder into six
  6-feature encoders means each branch sees less data — a real cost given this dataset has
  only 469–942 positive labels in the full cohort (see the label-definition note in §1.6.3).
  The trade is worth it only if each system genuinely carries distinguishable signal, which
  is testable (§11's per-system ablation) rather than assumed.
- **A natural place to bolt on domain knowledge**, e.g. the cardiorenal coupling in §1.6.2 —
  a single flat encoder has no natural "renal branch" to attach that prior to.

**References:** Choi et al. 2016 (NeurIPS), *RETAIN* — two-level attention over clinical
time series, the origin of "let the architecture carry the explanation"; Shickel et al.
2023 as the flat-encoder benchmark this design is trying to match *and* out-explain; Koh et
al. 2020 (ICML), *Concept Bottleneck Models* — the more general principle that forcing a
network through named intermediate concepts (here: organ-system embeddings) makes
explanations part of the computation.

**Decision:** this notebook always uses the organ-system split (it's the point of the
exercise). A flat-encoder baseline is the natural ablation to quantify what the split
costs/buys empirically rather than just argued for — **not built in this revision**,
flagged honestly here rather than left as a config flag that quietly does nothing (an
earlier draft of this notebook declared a `USE_SYSTEM_SPLIT` flag for exactly this and
never wired it to anything; that flag has been removed rather than shipped broken). §11.4's
actual ablation compares fusion strategies (NAM vs. concat) at fixed system-split
architecture — a different, real comparison, not a substitute for this one.



## 1.2 ICD-10: what it is, and how this project uses it

**What ICD-10-CM is.** The 10th revision of the WHO's International Statistical
Classification of Diseases, Clinical Modification — a hierarchical code set for diagnoses.
INSPIRE records the **first three characters only** (e.g. `N18`, not `N18.3`) of every
diagnosis claimed by a physician from 6 months before surgery to final discharge (with a
pre-defined sensitive-diagnosis list already removed by the dataset's authors). Codes group
into **21 chapters** by first letter + numeric range, e.g. `I00–I99` = circulatory system,
`K00–K93` = digestive system. This notebook implements that chapter lookup directly
(`icd10_chapter()` in §6.6) — no external library needed, and it degrades gracefully if a
code falls outside the known ranges.

### 1.2.1 Three different ways this project uses ICD-10, and why they're different problems

| Use | What it produces | Why it's a separate design decision |
|---|---|---|
| **Chapter membership** (this notebook, GI & MSK systems) | A 0/1 flag or count per patient: "does this patient have any digestive-system diagnosis?" | Coarse, always available, zero cold-start problem — every chapter has thousands of patients even in a 30-patient dev set every code individually would not. |
| **Curated, clinically-weighted subsets — HFRS** (Gilbert et al. 2018, *The Lancet*) | A single frailty score: sum of weights for 109 specific ICD-10 codes known to predict frailty, restricted to patients 75+ and a 2-year lookback | Validated externally, not learned from this dataset at all — a genuinely independent signal, good as a baseline/comparison feature rather than something the network re-derives. |
| **Learned embeddings** (mentioned as future work, not built by default here) | A dense trainable vector per code, closer to a word-embedding | Needs many examples per code to learn anything reliable — with only 469–942 positive labels in the full cohort and 1,000+ distinct codes, this is exactly the kind of high-cardinality-categorical/small-label mismatch that overfits. §1.5.3 covers the standard fix (pre-train on co-occurrence, unsupervised, before touching labels). |

**Decision used in this notebook:** chapter-level flags/counts for GI and MSK (cheap,
always available, directly answers the meeting note), **plus** HFRS as a separate
validated feature (already specified in the repo, computed here in a self-contained
form), **and** the option (`USE_ICD10_EMBEDDING`, off by default) to swap the flags for a
small learned embedding once you have more than the 30-patient dev set to train it on.

### 1.2.2 ASA / POSSUM / NELA — the non-learned clinical scores this model is implicitly competing with

Useful context for judging whether the DNN is worth the added complexity: these three are
**fixed, frozen, published equations/judgements**, not machine-learned, and none of them
were fitted on INSPIRE.

| | ASA | POSSUM / P-POSSUM | NELA |
|---|---|---|---|
| What it is | Anaesthesiologist's subjective I–VI class | 12 physiology + 6 operative-severity factors → logistic regression | 25-variable logistic regression, built for emergency laparotomy |
| Learns from data? | No — clinical gestalt | No — equation fixed since 1991 (Copeland et al.); P-POSSUM (Prytherch et al. 1998) corrects its known overestimation in low-risk patients | No — fixed UK-audit equation |
| Caveat for INSPIRE | Not used as a DNN input by default here — free, 100% coverage, added in §6.12 | Not implemented in this notebook (flagged as future work) | Built on a UK population; a lower NELA AUROC on INSPIRE may reflect population mismatch, not a weak tool |

### 1.2.3 Why Gastrointestinal and Musculoskeletal are diagnosis-driven, not lab-driven — a real finding, not a shortcut

Before writing the organ-system router (§6), this notebook's raw item-name inventory
(§4.4) was checked against the six systems already defined in the source repo's own
documentation (`docs/roadmap_and_architecture.md` §4.1) — Renal, Cardiovascular,
Respiratory, Metabolic/hepatic, Haematology/coagulation, Neurological. **Neither GI nor MSK
appears in the `labs`, `vitals`, or `ward_vitals` item-name inventories at all** — INSPIRE
simply doesn't chart a GI-specific panel (no direct bilirubin-only-for-GI, no bowel-sound
score) or an MSK-specific panel (no mobility score, no joint-specific measurement) the way
it charts renal function via creatinine or cardiac function via troponin. This is confirmed
programmatically in §4.4 before any feature is built, rather than assumed.

**What this means concretely:** the GI and MSK "encoders" in this architecture cannot be
time-series encoders the way the other four systems are — there is no GI/MSK time series to
feed one. Instead, both are built from what *is* available and genuinely GI/MSK-relevant:

- **ICD-10 chapter XI** (`K00–K93`, digestive system) and **chapter XIII** (`M00–M99`,
  musculoskeletal system and connective tissue) diagnosis flags/counts.
- **Surgical department** — INSPIRE's `department` field (e.g. `GS` general surgery is the
  closest proxy for GI surgical burden; `OS` orthopaedic surgery for MSK) is a categorical
  feature, not a time series, and is folded into the static branch (§9.2) with a per-system
  one-hot so the network can learn the association without it being hard-coded.
- Two labs sit at a genuine **overlap boundary** and deserve an explicit decision rather
  than silent double-counting: `ast`/`alt`/`alp`/`total_bilirubin` are hepatic function
  tests, which this notebook keeps in **Metabolic/hepatic** (matching the source repo's
  existing grouping) rather than duplicating into GI — the liver is anatomically digestive
  but functionally metabolic, and duplicating the same four features into two branches
  would let the network double-count the same signal under two different names, silently
  inflating its apparent importance in both. This is stated explicitly here so the
  boundary is a documented decision, not an accident.

This is worth flagging as a finding in its own right if this work is written up: **not
every clinically meaningful system has equal representation in this dataset's raw signal
tables** — GI and MSK risk in INSPIRE is currently only visible through diagnosis codes and
department, not through direct physiological monitoring, which is itself informative about
what a future data-collection effort would need to add.



## 1.3 Missing data: mechanisms first, methods second

**Why mechanism matters before method.** Rubin (1976), *Biometrika* — the foundational
paper for essentially all modern missing-data theory — showed that the *right* way to
handle a gap depends on **why** it's missing, not just how much is missing. Three
mechanisms, in increasing order of difficulty:

| Mechanism | Definition | Example in INSPIRE |
|---|---|---|
| **MCAR** — Missing Completely At Random | Missingness is unrelated to any value, observed or not | A monitor briefly disconnects for a cabling reason unrelated to the patient |
| **MAR** — Missing At Random | Missingness depends on *other observed* variables, not the missing value itself | A lab is only drawn pre-operatively for certain departments' routine protocols — missingness depends on `department` (observed), not on the lab result itself |
| **MNAR** — Missing Not At Random | Missingness depends on the *unobserved value itself* | A clinician **chooses not to order a test because the patient looks well enough not to need it** — missingness is driven by the (unmeasured) true severity |

**Why this specific dataset likely has real MNAR structure, not just MCAR gaps.** The
source repo's own EDA already surfaces exactly this pattern: `spo2` coverage is 93%
"anywhere in the record" but drops to 40% in the pre-op window specifically, and `uo`
(urine output) is 0% pre-op — not because measurement failed, but because monitors and
catheters are physically attached once a patient is admitted/in theatre, not before. That's
MAR (depends on the observed variable "is the patient in theatre yet"). But a plausible
*additional* layer is MNAR: a clinician who judges a patient low-risk may simply not order a
discretionary pre-op test — meaning **the absence of a test result is itself a (noisy)
low-risk signal**, not noise to be interpolated away. This is why every imputation strategy
below is paired with an explicit **missingness mask feature** (1 = observed, 0 = imputed) —
under MNAR, the mask can carry real information the imputed value cannot.

### 1.3.1 Imputation methods on offer in this notebook

| Method | How it works | When it's the right choice | Reference |
|---|---|---|---|
| **Median/mode fill** | Replace every gap with the training-set median (numeric) or mode (categorical) | Fast, safe baseline; best when missingness is closer to MCAR and the feature's distribution is stable across patients | Standard statistical practice; see Little & Rubin, *Statistical Analysis with Missing Data*, 3rd ed. |
| **Population-mean / per-feature-stats fill** | Same idea, using pre-computed population mean/median (matches the source repo's `vital_stats` approach for features with zero observations for a patient) | Needed when a *whole feature* is missing for a patient (median of an empty series is undefined) | Used as-is in `dnn_mortality_data_real.py`'s `align_time_series()` |
| **Linear interpolation** (within-patient, between observed points) | Straight line between the two nearest observed values in time; used by the existing pipeline via `interp1d` | Good for physiological values that plausibly change smoothly between measurements (e.g. heart rate) | Standard for irregularly-sampled clinical time series |
| **Forward-fill / Last-Observation-Carried-Forward (LOCF)** | Repeat the last observed value forward until the next observation | Simple, defensible for slow-changing labs (e.g. creatinine over hours); a poor choice for fast-changing vitals | Widely used in clinical time-series ML, e.g. Che et al. 2018, *GRU-D* |
| **KNN imputation** | Fill a gap using the average of the k most similar *other patients'* values for that feature | Useful when within-patient information is too sparse (e.g. only one lab draw ever) but between-patient similarity is informative | Troyanskaya et al. 2001, *Bioinformatics* (originally for gene-expression data, now standard for clinical tabular data) |
| **MICE / multiple imputation by chained equations** | Iteratively models each feature with missing values as a function of all other features, cycling until convergence, optionally producing several imputed datasets to average over | The statistically principled choice under MAR; most expensive; most defensible for a published result | Van Buuren & Groothuis-Oudshoorn 2011, *Journal of Statistical Software* — `mice` package; `sklearn.impute.IterativeImputer` implements the same idea |
| **Missingness mask + zero/median fill** (always applied here alongside whichever value-fill method is chosen) | A second binary feature per value-feature: was this observed or imputed? | Lets a downstream model recover MNAR signal that a value-only fill destroys | Standard in clinical deep learning, e.g. Che et al. 2018, *GRU-D*; Lipton et al. 2016 |

### 1.3.2 Decision framework used in this notebook

```
Is the whole feature missing for this patient (zero measurements)?
  → yes: population median/mean fill (nothing else is possible) + mask = 0
  → no, continue
Is this a fast-changing intra-op vital (hr, sbp, spo2, ...)?
  → yes: linear interpolation between observed points, extrapolation fades to
         the population mean the further from any observation (the existing
         repo's `smooth_fade_to_mean_interpolator` — kept here because fading
         to the mean, not extrapolating a trend indefinitely, is the safer
         default for a value with no evidence far from the observed window)
  → no, continue
Is this a slow-changing pre-op lab (creatinine, albumin, ...)?
  → yes: forward-fill within-patient if any prior value exists, else median
Always: attach a mask feature alongside the value, regardless of method chosen.
```

`CONFIG['IMPUTATION_STRATEGY']` lets you override this per-run (`'median'`,
`'interpolate'`, `'knn'`, `'mice'`) to run the comparison yourself in §7.4 — the decision
tree above is the default, not a fixed answer, precisely because you said you want to
experiment with it.



## 1.4 Class imbalance: why it matters here, and the sampling methods on offer

**The scale of the problem.** Even under the most generous label definition, this dataset
has roughly 469–942 deaths out of ~99,886 patients (0.5–0.9%) at full scale, and in the
30-patient development subset supplied here, 10/30 = 33% died — a **development-set
imbalance that is not representative of the real deployment imbalance**, which matters:
whatever sampling ratio "looks right" on 30 patients will very likely need re-tuning once
run on the full cohort (a note worth keeping visible, so §8 keeps the *ratio* configurable
rather than hard-coded).

**Why imbalance breaks naive training.** A classifier minimising plain accuracy on a
99%/1% split gets 99% accuracy by predicting "survived" for everyone — useless. The network
also literally sees ~100–200x more negative gradient signal than positive per epoch,
so without correction it learns almost nothing about what makes a death case different.

### 1.4.1 Sampling / re-balancing methods on offer

| Method | How it works | Trade-off | Reference |
|---|---|---|---|
| **Random oversampling** | Duplicate minority (died) examples until classes are balanced | Simple; risks overfitting to the exact duplicated examples, especially with very few positives (here, single digits to low hundreds) | He & Garcia 2009, *IEEE TKDE* survey |
| **Random undersampling** | Drop majority (survived) examples until classes are balanced | Throws away real data — costly when the majority class itself has useful diversity to learn "normal" from (which matters for the autoencoder pre-training step, §9.4) | He & Garcia 2009 |
| **SMOTE** (Synthetic Minority Oversampling) | Creates *synthetic* minority examples by interpolating between a minority example and its nearest minority neighbours in feature space | Reduces exact-duplicate overfitting vs. random oversampling; interpolating between two very different critically-ill patients' feature vectors can create physiologically implausible synthetic patients — worth sanity-checking, not just trusting | Chawla et al. 2002, *JAIR* |
| **ADASYN** | Like SMOTE, but generates more synthetic examples for minority points that are *harder* to classify (near the decision boundary) | Focuses capacity where it's needed most; inherits SMOTE's interpolation-plausibility caveat | He et al. 2008, *IEEE IJCNN* |
| **Class weighting / `pos_weight`** | Leave the data alone; scale the loss so each positive example counts `N_neg/N_pos` times more | No synthetic data, no data thrown away — usually the safest first thing to try; doesn't help a model that never sees *enough distinct* positive examples to learn from, only one that under-weights the ones it does see | Standard in `BCEWithLogitsLoss(pos_weight=...)`; used already in the source repo's pipeline |
| **Focal loss** | Down-weights the loss contribution of *easy*, already-well-classified examples (of either class), focusing gradient on hard/ambiguous ones | Complements class weighting rather than replacing it; adds one extra hyperparameter (focusing exponent γ) to tune | Lin et al. 2017, *ICCV*, "Focal Loss for Dense Object Detection" — since adopted widely in imbalanced clinical ML |
| **Stratified splitting** (not a re-balancing method, but a prerequisite for any of the above being valid) | Ensures train/val/test each preserve the true died/survived ratio | Without this, a tiny test set (here: a handful of patients) can land 0 or all positives by chance, making any metric meaningless | Standard practice; essential given the ~30-patient dev set |

### 1.4.2 Decision framework used in this notebook

- **Always** stratify every split (§8.1) — non-negotiable regardless of which other method
  is chosen, because with 10 positives in 30 patients, an unstratified split can easily put
  zero positives in a validation fold.
- **Default**: class weighting via `pos_weight` in the loss (§9.5), *not* SMOTE, as the
  primary strategy at this dataset's current scale — with only 10 positive examples in the
  supplied dev set, SMOTE's nearest-neighbour interpolation has too few real neighbours to
  interpolate between meaningfully (SMOTE's own paper assumes a reasonably-populated
  minority-class neighbourhood; 10 points in the six-plus-dimensional pooled embedding
  space this model works in is thin). Revisit this once the full ~469–942-patient cohort is
  loaded, where SMOTE becomes far more defensible.
- **Offered, off by default**: SMOTE/ADASYN via `imbalanced-learn`, focal loss as a loss
  option, random over/under-sampling — all switchable via `CONFIG['SAMPLING_STRATEGY']`
  in §8, specifically so you can run the comparison yourself once more data is available,
  rather than this notebook silently picking one for you.



## 1.5 Multimodal deep learning: encoders, fusion, and joint embedding learning

This is the architecture layer the meeting note "features learned separately for each
system and then jointly learning the embeddings" describes. Multimodal learning here means
combining genuinely different **data types** (irregular time series, categorical codes,
fixed-size static vectors) — not just "many features," but many *kinds* of feature, each of
which needs a different neural building block to be encoded well.

### 1.5.1 Modality-specific encoders — one block per data type, not one block for everything

| Data type | Present in INSPIRE as | Encoder used here | Why this block for this data type |
|---|---|---|---|
| Continuous, irregularly-sampled time series | Labs, ward vitals, intra-op vitals | Transformer encoder (self-attention over time steps) — matching the existing repo's `TimeSeriesTransformer` design, one instance per organ system | Attention handles irregular spacing and variable-length sequences without the vanishing-gradient issues of a plain RNN over long sequences, and — usefully for §1.1 — attention weights are directly inspectable (§11.3) |
| Categorical sequence | Medications (drug/ATC code + route, timestamped) | A learned embedding table per ATC therapeutic class (not per raw drug name — see §1.5.3), aggregated by simple pooling in this notebook's default configuration | Learned lookup, same idea as a word embedding in NLP, adapted to a clinical vocabulary |
| Categorical, mostly static-in-effect | Diagnoses (ICD-10), department | Chapter-level flags/counts by default (§1.2.1); an optional learned embedding table | Cold-start-safe at this dataset's scale (§1.2.1) |
| Fixed-size static vector, 100% coverage | Age, sex, ASA, `emop` (emergency flag), department, weight, height | A small MLP | Cheap, always available, currently unused by the DNN in the source repo despite being one of the strongest established predictors in the literature — added here |

### 1.5.2 Fusion strategies — how the per-system embeddings get combined into one prediction

| Strategy | How it works | Interpretability | Complexity |
|---|---|---|---|
| **Early concatenation** | Stack every system embedding into one vector, one final `Linear` layer | Low — the per-system split is undone the moment a dense layer mixes everything together | Lowest |
| **Gated / mixture-of-experts** | A small gating network learns *how much weight* to give each system's embedding, per patient | Medium — the gate itself is inspectable ("70% of this patient's risk weight came from the renal branch") | Medium |
| **Hierarchical / additive (Neural Additive Model, NAM)** | Each system embedding passes through its own small sub-network into a single scalar *contribution* to the final logit; the final prediction is literally the sum of per-system contributions | High — each system's contribution is individually plottable as its own shape function, a real decomposition rather than a post-hoc explanation | Higher (more sub-networks to train, though each is small) |

**Reference:** Baltrušaitis, Ahuja & Morency 2019, *IEEE TPAMI*, "Multimodal Machine
Learning: A Survey and Taxonomy" — the standard taxonomy for early/late/hybrid fusion this
table is built on; Agarwal et al. 2021, *NeurIPS*, "Neural Additive Models" — the specific
architecture behind the third row; Xu, Kim & Ho 2024 (or similar recent EHR-multimodal
surveys) for how this fusion taxonomy has been applied specifically to clinical/EHR
multimodal problems.

**Decision used by default in this notebook:** NAM-style additive fusion (§9.7), because it
is the only one of the three that actually delivers "Renal: CRITICAL, Cardiovascular: OK"
as a real decomposition — matching the project's stated interpretability aim, not just its
accuracy aim. Early concatenation is kept as a `CONFIG['FUSION_STRATEGY']` option for
direct comparison (§11.4), since NAM's extra structure is a real capacity/complexity cost
that should be justified empirically, not assumed.

### 1.5.3 Learning embeddings for categorical entities — two genuinely different training regimes

- **Supervised end-to-end**: the embedding table is trained jointly with the mortality
  objective. Simple, but with a small-label dataset, most rare codes see too few positive
  examples to learn a reliable direction — the same sparsity problem as everywhere else in
  this project.
- **Unsupervised pre-training on co-occurrence** (not implemented by default here, flagged
  as a natural extension once the full cohort is available): learn embeddings from
  which codes/drugs tend to appear together across *all* ~100k patients, regardless of
  outcome — the approach behind Med2Vec (Choi et al.) and RETAIN-style clinical embeddings,
  and directly analogous to what this notebook's autoencoder pre-training (§9.4) already
  does for continuous time series, just applied to categorical codes.

**Decision used here:** start one level up the drug hierarchy — WHO ATC level-2/3
(therapeutic subgroup, e.g. "beta-blocking agents") rather than raw drug name — which
shrinks the vocabulary from ~1,238 distinct drugs to a few dozen therapeutic classes, giving
each embedding far more examples to learn from before finer granularity is worth the risk.

### 1.5.4 Joint embedding learning — what "jointly learning the embeddings" actually optimises

Two genuinely different things can be meant by "joint," and this notebook implements both,
explicitly separated so it's clear which is happening where:

1. **Joint = trained together, one shared objective.** In this notebook's default
   configuration, every system encoder, the static MLP, and the fusion layer are all
   updated by the same backward pass from the mortality loss — so the *renal* encoder's
   weights are shaped partly by what turns out to matter for predicting death, not only by
   what best reconstructs renal values. This is standard end-to-end multi-branch training.
2. **Joint = pre-trained together, unsupervised, before any label is touched.** Each
   system's encoder is first trained as an autoencoder (reconstruct its own masked-out
   input) on the *entire* unlabelled cohort — this matters more here than in a typical
   multimodal setup, because splitting one encoder into six also splits the
   already-scarce mortality-labelled data six ways; unsupervised pre-training lets every
   branch still learn from the full patient pool before the label-starved fine-tuning
   phase begins. This two-phase pattern (pretrain → fine-tune) already exists in the
   source repo's `dnn_mortality_pipeline.py` for its single flat encoder; this notebook
   applies the same pattern per-system (§9.4–9.6).

**References:** the general pretrain-then-finetune pattern traces to BERT-style masked
pretraining (Devlin et al. 2019) adapted to clinical time series; Che et al. 2018 (*GRU-D*)
for handling missingness end-to-end inside the encoder itself, an alternative worth knowing
about even though this notebook uses the mask-feature approach from §1.3 instead, for
consistency with the existing repo's pipeline.



## 1.6 Project-specific architecture decisions from the meeting notes

This section works through each meeting-note item explicitly, so the reasoning behind each
`CONFIG` flag and each engineered feature in Part 6–9 is traceable back to a decision, not
just a comment in code.

### 1.6.1 GI and Musculoskeletal systems — recap of §1.2.3

Both are diagnosis/department-driven branches (ICD-10 chapters XI and XIII, plus
`department`), not time-series encoders, because this dataset has no dedicated GI or MSK
signal table. Implemented in §6.7–6.8, fed into the static/categorical branch (§9.2) rather
than the time-series branch (§9.1).

### 1.6.2 "Renal directly proportionate to cardiovascular" — cardiorenal syndrome

This is not just a modelling convenience — it's an established clinical phenomenon called
**cardiorenal syndrome (CRS)**: the heart and kidneys regulate each other's function so
tightly (via blood pressure, fluid balance, and shared neurohormonal signalling) that
dysfunction in one routinely drives dysfunction in the other. Ronco et al. 2008, *Journal of
the American College of Cardiology*, formalised this into five types based on which organ
fails first and how acutely:

| CRS type | Primary failure → secondary failure | Relevance here |
|---|---|---|
| Type 1 (acute CRS) | Acute heart failure → acute kidney injury | Perioperative acute cardiac events causing acute renal injury — directly relevant to post-cardiac-surgery patients |
| Type 2 (chronic CRS) | Chronic heart failure → chronic kidney disease | Patients with pre-existing cardiac disease presenting with impaired renal function pre-op |
| Type 3 (acute renocardiac) | Acute kidney injury → acute cardiac dysfunction | Acute renal insult (e.g. contrast, hypotension) precipitating cardiac strain |
| Type 4 (chronic renocardiac) | Chronic kidney disease → chronic cardiac dysfunction | Long-standing CKD patients with secondary cardiac disease |
| Type 5 (secondary CRS) | A systemic condition (e.g. sepsis) causes both simultaneously | Relevant to the acute-deterioration ICD-10 codes already flagged in the source repo's EDA (sepsis, DIC, shock) |

**How this is implemented (§6.9), not just discussed:**

1. **Cardiovascular features are made directly visible to the renal branch** — the renal
   encoder's input in this notebook is renal features **plus** a compact summary of
   cardiovascular status (mean heart rate, mean blood pressure, presence of an IABP flag),
   rather than renal features alone. This lets the renal embedding learn the coupling
   directly from data, instead of the network having to rediscover a correlation that
   clinical literature already establishes.
2. **An explicit interaction feature** — a renal-cardiac ratio/product feature (e.g.
   `creatinine × mean_arterial_pressure_deviation`) is added to the static feature set as a
   hand-crafted prior, on the reasoning that a purely learned interaction needs many
   examples to discover reliably, while a hand-crafted one encodes the clinical prior
   directly and costs nothing at small sample sizes.
3. This is explicitly **not** applied symmetrically (cardiovascular does not by default
   receive renal features as an extra input) — the meeting note stated the direction
   ("renal is directly proportionate to cardiovascular"), and asymmetric coupling is also
   the more common real direction in Type 2/4 CRS (chronic cardiac/renal disease more often
   drives the other than the reverse in a stable pre-op population). `CONFIG['SYMMETRIC_CARDIORENAL_COUPLING']`
   is provided to test the symmetric version too.

### 1.6.3 Pre-operative / intra-operative / post-operative windows

The three windows map onto fields already in every patient record:
`admission_time → orin_time → [surgery] → orout_time → discharge_time`. Each window answers
a different clinical question and is a different eventual *product*, not just three
versions of the same model:

| Window | What it captures | Product framing |
|---|---|---|
| Pre-operative (default here) | Baseline chronic disease burden — what ASA/POSSUM/NELA are all approximating | Decision-support: should we operate, should ICU be pre-booked |
| + Intra-operative | Acute physiological stress of surgery/anaesthesia itself | Real-time monitoring: should the team escalate mid-case |
| + Early post-operative | Recovery vs. deterioration trajectory | Early-warning / rescue tool |

**Decision used here:** `CONFIG['TIME_WINDOW']` defaults to `'pre_op'` (5 days before
`orin_time`, matching the existing repo's pipeline and this being the safest, most
data-complete starting point), with `'peri_op'` (adds intra-op `vitals`) as a one-flag
toggle so the pre-op-only vs. peri-operative question raised in the meeting/docs can be run
as a genuine sensitivity comparison (§11.5) instead of assumed.

**A related, important open bug carried over from the source repo, restated here so it
isn't silently inherited:** the repo's own `Research_Aim.md` flags that the
`survived/`/`died/` folder split may be built from "died at any point in the record"
(`died()`) rather than "died within 30 days of the last operation" (`inhosp_death_30day()`)
— 473 of 942 all-cause deaths in the full cohort are estimated *not* to be true 30-day
deaths. This notebook **always recomputes the 30-day label from the raw `operations`
fields** (§5.2) rather than trusting the folder name, specifically to avoid inheriting this
bug — worth keeping in mind if you swap in the full dataset and see the folder-implied label
count disagree with what §5.2 computes.

### 1.6.4 Number of operations in the same area — and the 6-month post-cardiac-surgery exception

**The general feature (§6.10):** for each patient, count prior operations whose
`department` (or ICD-10-PCS-derived body system) matches the system being encoded, plus the
time since the most recent one — a repeat-operation-in-the-same-area patient is plausibly
a different risk profile than a first-time patient in that area, in either direction (staged
procedures for a controlled condition vs. repeated emergency interventions for an
uncontrolled one), which is exactly why this is offered as a *feature* for the model to
weigh, not a hard rule.

**The exception (§6.11), by contrast, is deliberately implemented as a rule, not a learned
feature.** The meeting note that "after a cardiovascular surgery, need to wait 6 months for
next operation" describes a **clinical protocol constraint** (elective surgery is
routinely deferred for ~6 months after major cardiac surgery/stenting due to
antiplatelet-therapy and myocardial-recovery windows — a scheduling and risk-management
rule, not a statistical pattern to be discovered from 30–900 patients). Two reasons this
stays rule-based rather than learned:

1. **Sample size.** A learned feature needs many examples of "operation within 6 months of
   prior cardiac surgery" to discover the effect reliably; a clinical protocol is already
   known and doesn't need re-deriving from a small, noisy sample.
2. **Actionability.** A flag computed from a known rule (`insufficient_recovery_window = 1
   if days_since_last_cardiac_op < 180 else 0`) is directly interpretable and directly
   checkable against the record, whereas a purely learned pattern risks being a
   confound (e.g. sicker patients get both repeat-early surgery *and* worse outcomes for
   unrelated reasons) dressed up as a discovered rule.

Implemented as a binary flag in the static feature set, **and** flagged for a follow-up
sensitivity check: does the model's cardiovascular-system embedding shift materially for
patients who violate this window, versus similar patients who don't? (§11.6 — a good
qualitative validation, analogous to the source repo's own worked-patient sanity checks.)



## 1.7 Summary — decision table (every `CONFIG` flag used from here on)

| Config key | Default | Alternatives offered | Where decided | Where used |
|---|---|---|---|---|
| `TIME_WINDOW` | `'pre_op'` | `'peri_op'` | §1.6.3 | §6, §7 |
| `IMPUTATION_STRATEGY` | `'decision_tree'` (the §1.3.2 tree) | `'median'`, `'interpolate'`, `'knn'` | §1.3.2 | §7 |
| `SAMPLING_STRATEGY` | `'grouped_smotenc_tomek'` (Part C's combined pipeline) | `'class_weight'`, `'smote'`, `'adasyn'`, `'random_oversample'`, `'random_undersample'`, `'focal_loss'` | §1.4.2 (see `Data_Imbalance_and_Imputation_Reference.md` for the full revision) | §8, §10 |
| `FUSION_STRATEGY` | `'nam'` (additive) | `'concat'`, `'gated'` | §1.5.2 | §9.7 |
| `SYMMETRIC_CARDIORENAL_COUPLING` | `False` | `True` | §1.6.2 | §6.9 |
| `USE_ICD10_EMBEDDING` | `False` (chapter flags instead) | `True` | §1.2.1 | §6.6, §9.2 |
| `INCLUDE_HFRS` | `True` | `False` | §1.2.1 | §6.6 |
| `INCLUDE_STATIC_ASA` | `True` | `False` | §1.2.2 | §6.12 |

Everything below this point is code. Every place one of these flags matters, the code cell
says so in a comment referencing the section above.



# Part 2 — Setup

Installs (Kaggle usually has most of these; the cell is safe to run either way), imports,
and the `CONFIG` dict that every flag from §1.7 lives in. **Change values here, then
Run All** — nothing below this cell should need editing to run an experiment.


In [ ]:

# Kaggle already ships torch, sklearn, pandas, numpy, matplotlib.
# imbalanced-learn is usually NOT preinstalled -> install if missing.
import importlib, subprocess, sys

def _ensure(pkg, import_name=None):
    import_name = import_name or pkg
    try:
        importlib.import_module(import_name)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=False)

_ensure("imbalanced-learn", "imblearn")
print("Setup check complete.")


In [ ]:

import os, glob, json, math, random, warnings
from collections import defaultdict, Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import train_test_split
from sklearn.experimental import enable_iterative_imputer  # noqa: F401  (must precede IterativeImputer import)
from sklearn.impute import KNNImputer, IterativeImputer
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (roc_auc_score, average_precision_score, brier_score_loss,
                              roc_curve, precision_recall_curve, confusion_matrix)

warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)
if DEVICE.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}  "
          f"total VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
    print("(This notebook's model is small -- tens of thousands of parameters, small batches -- "
          "GPU memory is very unlikely to be the constraint. System RAM, from the raw data "
          "tables in Part 3, is the one actually worth watching -- see that Part's memory printout.)")


In [ ]:

# ---------------------------------------------------------------------------
# CONFIG — every flag documented in Part 1 §1.7. Edit here, not below.
# ---------------------------------------------------------------------------
CONFIG = {
    # --- data location (auto-detected below; override if detection is wrong) ---
    "SUBJECTS_DIR": None,      # folder containing died/ and survived/ subfolders of per-patient JSON
    "CODES_DIR": None,         # optional: folder with icd10.json.gz / WHO_ATC-DDD_*.csv (finer descriptions only)

    # --- full-scale readiness (new) ---
    # The dev subset (30 patients) never exercises memory limits. The full INSPIRE cohort
    # (~99,886 patients) can, especially the ward_vitals/vitals tables. Set an int here to
    # cap patients PER FOLDER (died/survived) for a fast smoke test before committing to a
    # full run -- e.g. 500 gives you a ~1,000-patient run that exercises the exact same
    # code path as the full cohort. Set to None for the real, full run.
    "MAX_SUBJECTS_PER_CLASS": None,

    # --- memory/compute lever, off by default -- see Part 3's "note on memory strategy" ---
    # If set, caps the number of REAL SURVIVED patients in the TRAINING split only (never
    # died patients, never val/test) to this count, via a department-stratified sample so
    # the department mix of the training pool is preserved. Try TIME_WINDOW='pre_op' and a
    # MAX_SUBJECTS_PER_CLASS smoke test first -- this is a last resort, not a first move.
    "DOWNSAMPLE_TRAIN_SURVIVED_TO": None,

    # --- §1.6.3 time window ---
    "TIME_WINDOW": "pre_op",              # 'pre_op' | 'peri_op'
    "PRE_OP_DAYS": 5,                     # matches the source repo's existing window

    # --- §1.3 missing data ---
    "IMPUTATION_STRATEGY": "decision_tree",  # 'decision_tree' | 'median' | 'interpolate' | 'knn'
    "KNN_NEIGHBORS": 5,

    # --- §1.4 / Part C class imbalance (updated to the agreed combined pipeline) ---
    "SAMPLING_STRATEGY": "grouped_smotenc_tomek",
    # 'grouped_smotenc_tomek' (new default, Part C §6) | 'class_weight' | 'smote' | 'adasyn'
    # | 'random_oversample' | 'random_undersample' | 'none'
    "SMOTE_TARGET_RATIO": 0.10,     # positive:negative target, e.g. 0.10 = 1:10 (Part C's conservative pick)
    "SMOTE_MIN_STRATUM_MINORITY": 3,   # skip a department x ASA stratum with fewer real positives than this
    "SMOTE_STRATA_COLS": ["department", "asa"],   # Part C §6's clinical-neighborhood grouping
    "USE_SEQUENCE_AUGMENTATION": True,      # Part C §7: jitter + time-mask for REAL minority training patients
    "SEQUENCE_AUGMENTATION_COPIES": 2,      # extra augmented copies per real positive training patient
    "JITTER_SIGMA": 0.05,                   # on the standardized (z-scored) scale, per Part C §7
    "USE_FOCAL_LOSS": False,
    "FOCAL_GAMMA": 2.0,

    # --- §1.5 architecture ---
    # (a flat-vs-system-split ablation flag was considered here and dropped rather than
    # shipped unwired -- see §1.1's note. The organ-system split is not optional in this
    # notebook's architecture.)
    "FUSION_STRATEGY": "nam",              # 'nam' | 'concat' | 'gated'
    "EMBED_DIM": 16,                       # per-system embedding size
    "TRANSFORMER_HEADS": 2,
    "TRANSFORMER_LAYERS": 1,

    # --- §1.6.2 cardio-renal coupling ---
    "SYMMETRIC_CARDIORENAL_COUPLING": False,

    # --- §1.2 ICD-10 ---
    "USE_ICD10_EMBEDDING": False,
    "INCLUDE_HFRS": True,
    "HFRS_LOOKBACK_YEARS": 2,              # published Gilbert et al. window, age 75+
    "INCLUDE_STATIC_ASA": True,

    # --- training ---
    "BATCH_SIZE": 8,          # small on purpose for the 30-patient dev subset; raise for the full cohort
    "EPOCHS_PRETRAIN": 30,
    "EPOCHS_FINETUNE": 60,
    "LR": 1e-3,
    "VAL_FRACTION": 0.2,
    "TEST_FRACTION": 0.2,
    "TARGET_SEQ_LEN": 24,      # number of time points each system's series is resampled/padded to

    # --- checkpointing (matters most on Colab -- free-tier sessions can disconnect for
    # reasons unrelated to memory: idle timeout, daily usage caps) ---
    "CHECKPOINT_DIR": None,          # auto-set below: Drive if on Colab+mounted, else local
    "CHECKPOINT_EVERY_N_EPOCHS": 5,
}

# ---------------------------------------------------------------------------
# Environment detection: Kaggle vs. Colab vs. local. Matters for both data-path
# auto-detection and where checkpoints should live (Colab's local disk is ephemeral --
# a checkpoint saved there is lost on disconnect just like everything else).
# ---------------------------------------------------------------------------
IN_COLAB = "google.colab" in sys.modules
IN_KAGGLE = os.path.isdir("/kaggle/input")

if IN_COLAB:
    try:
        from google.colab import drive
        if not os.path.isdir("/content/drive/MyDrive"):
            print("Colab detected -- mounting Google Drive (needed so your data/checkpoints "
                  "survive a session disconnect; a browser auth prompt may appear)...")
            drive.mount("/content/drive")
        else:
            print("Colab detected -- Google Drive already mounted.")
    except Exception as e:
        print(f"Colab detected but Drive mount failed or was skipped ({e}). "
              f"Data auto-detection and checkpointing will fall back to local (ephemeral) storage.")

# NOTE: checks IN_COLAB first -- the "google.colab" import check is a definitive signal,
# whereas an unrelated leftover "/kaggle/input" directory (e.g. from a prior session state)
# is not, and previously caused this to misreport "Kaggle" on an actual Colab run.
print(f"Environment: {'Colab' if IN_COLAB else 'Kaggle' if IN_KAGGLE else 'local/other'}")

# ---------------------------------------------------------------------------
# Auto-detect data location: Kaggle input dirs, Colab Drive paths, then local upload paths.
# ---------------------------------------------------------------------------
def _find_subjects_dir():
    candidates = []
    # Fast, exact-path check first -- your data is at this known Drive location, so this
    # skips the (slower, walk-based) broader search below entirely when it matches.
    _known_exact_paths = [
        "/content/drive/MyDrive/subjects",
    ]
    for p in _known_exact_paths:
        if os.path.isdir(p) and os.path.isdir(os.path.join(p, "died")) and os.path.isdir(os.path.join(p, "survived")):
            return p
    if IN_KAGGLE:
        for root, dirs, files in os.walk("/kaggle/input"):
            if "died" in dirs and "survived" in dirs:
                candidates.append(root)
    if IN_COLAB:
        # Common places people drop an uploaded/extracted dataset on Drive -- searched
        # shallowly (max depth ~4) since walking all of MyDrive can be slow if it's large.
        drive_roots = ["/content/drive/MyDrive", "/content/drive/MyDrive/INSPIRE",
                       "/content/drive/MyDrive/inspire", "/content/drive/MyDrive/data",
                       "/content/drive/MyDrive/subjects", "/content"]
        for base in drive_roots:
            if not os.path.isdir(base):
                continue
            for root, dirs, files in os.walk(base):
                depth = root[len(base):].count(os.sep)
                if depth > 4:
                    dirs[:] = []   # don't descend further from here
                    continue
                if "died" in dirs and "survived" in dirs:
                    candidates.append(root)
    for p in ["/mnt/user-data/uploads/inspire_subjects_small",
              "/home/claude/work/inspire_subjects_small/inspire_subjects_small",
              "./inspire_subjects_small", "./subjects"]:
        if os.path.isdir(p) and os.path.isdir(os.path.join(p, "died")):
            candidates.append(p)
    return candidates[0] if candidates else None

def _find_codes_dir():
    if IN_KAGGLE:
        for root, dirs, files in os.walk("/kaggle/input"):
            if "icd10.json.gz" in files or any(f.startswith("WHO_ATC") for f in files):
                return root
    if IN_COLAB and os.path.isdir("/content/drive/MyDrive"):
        for root, dirs, files in os.walk("/content/drive/MyDrive"):
            depth = root[len("/content/drive/MyDrive"):].count(os.sep)
            if depth > 4:
                dirs[:] = []
                continue
            if "icd10.json.gz" in files or any(f.startswith("WHO_ATC") for f in files):
                return root
    for p in ["/home/claude/work/inspire-analysis/inspire-analysis-thrisha-main/codes", "./codes"]:
        if os.path.isdir(p):
            return p
    return None

if CONFIG["SUBJECTS_DIR"] is None:
    CONFIG["SUBJECTS_DIR"] = _find_subjects_dir()
if CONFIG["CODES_DIR"] is None:
    CONFIG["CODES_DIR"] = _find_codes_dir()
if CONFIG["CHECKPOINT_DIR"] is None:
    if IN_COLAB and os.path.isdir("/content/drive/MyDrive"):
        CONFIG["CHECKPOINT_DIR"] = "/content/drive/MyDrive/inspire_checkpoints"
    else:
        CONFIG["CHECKPOINT_DIR"] = "./inspire_checkpoints"   # Kaggle/local: ephemeral, but Kaggle sessions are less disconnect-prone
os.makedirs(CONFIG["CHECKPOINT_DIR"], exist_ok=True)

print("SUBJECTS_DIR   ->", CONFIG["SUBJECTS_DIR"])
print("CODES_DIR      ->", CONFIG["CODES_DIR"], "(optional — used only for richer ICD-10/ATC descriptions)")
print("CHECKPOINT_DIR ->", CONFIG["CHECKPOINT_DIR"],
      "(on Drive -- survives a Colab disconnect)" if CONFIG["CHECKPOINT_DIR"].startswith("/content/drive") else
      "(local/ephemeral -- will NOT survive a Colab disconnect; mount Drive if you're on Colab)")
assert CONFIG["SUBJECTS_DIR"] is not None, (
    "Could not auto-find a subjects folder (must contain died/ and survived/ subfolders of JSON). "
    "Set CONFIG['SUBJECTS_DIR'] manually -- e.g. on Colab, '/content/drive/MyDrive/<wherever you put it>'."
)



# Part 3 — Data loading

Each patient is one JSON file (in `died/` or `survived/`) with six lists: `labs`, `vitals`
(intra-op), `ward_vitals`, `operations`, `diagnoses`, `medications`. This section loads
every file into one Python object per patient and flattens the time-series tables into
long-format DataFrames — the same shape the rest of the notebook (and the source repo's own
loaders) expect.

**Note on folder labels (§1.6.3):** the `died`/`survived` folder is used only to know which
files to open — the actual mortality label used for training is **recomputed from
`operations`** in §5.2, not trusted from the folder name, for the reason explained there.

**Note on full-scale memory (new).** The dev subset (30 patients) never stresses memory.
At the full ~99,886-patient cohort, `ward_vitals` alone is estimated at roughly 250M+ rows
once flattened to long format — easily double-digit GB in plain pandas dtypes. Two
mitigations are applied below, both safe no-ops on the dev subset: (1)
`CONFIG['MAX_SUBJECTS_PER_CLASS']` lets you smoke-test the exact full-scale code path on a
smaller random sample before committing to the real run; (2) every flattened table below
uses memory-efficient dtypes (`category` for `subject_id`/`item_name`, `int32`/`float32`
instead of the 64-bit defaults) — categoricals alone typically cut a long-format table's
memory by more than half when the same few hundred item names repeat across millions of
rows.


In [ ]:

def load_all_subjects(subjects_dir, max_per_class=None, seed=SEED):
    # Load every patient JSON into a dict: subject_id -> raw dict.
    # max_per_class: if set, randomly cap each folder to this many files BEFORE opening
    # any JSON -- lets a full-scale smoke test skip the I/O cost too, not just the memory.
    subjects = {}
    folder_label = {}  # subject_id -> 'died' | 'survived' (folder-provided, NOT the trusted label)
    rng = random.Random(seed)
    for folder in ["died", "survived"]:
        folder_path = os.path.join(subjects_dir, folder)
        if not os.path.isdir(folder_path):
            continue
        filepaths = sorted(glob.glob(os.path.join(folder_path, "*.json")))
        if max_per_class is not None and len(filepaths) > max_per_class:
            filepaths = rng.sample(filepaths, max_per_class)
            filepaths.sort()
        for fp in filepaths:
            with open(fp) as f:
                d = json.load(f)
            sid = str(d["subject_id"])
            subjects[sid] = d
            folder_label[sid] = folder
    return subjects, folder_label

RAW_SUBJECTS, FOLDER_LABEL = load_all_subjects(CONFIG["SUBJECTS_DIR"], CONFIG["MAX_SUBJECTS_PER_CLASS"])
print(f"Loaded {len(RAW_SUBJECTS)} subjects "
      f"({sum(v=='died' for v in FOLDER_LABEL.values())} in died/, "
      f"{sum(v=='survived' for v in FOLDER_LABEL.values())} in survived/ — folder-provided, unverified)")
if CONFIG["MAX_SUBJECTS_PER_CLASS"] is not None:
    print(f"NOTE: MAX_SUBJECTS_PER_CLASS={CONFIG['MAX_SUBJECTS_PER_CLASS']} -- this is a "
          f"SMOKE TEST subsample, not the full cohort. Set to None for the real run.")


In [ ]:

def _safe_float(x):
    try:
        if x is None or x == "":
            return None
        return float(x)
    except (TypeError, ValueError):
        return None

def _safe_int(x):
    try:
        if x is None or x == "":
            return None
        return int(float(x))
    except (TypeError, ValueError):
        return None

def flatten_table(subjects, table_name, value_is_float=True):
    # labs / vitals / ward_vitals -> one long DataFrame across all subjects.
    # Memory note: at full scale these tables are the biggest thing in the notebook, so
    # dtypes are downcast immediately rather than left at pandas' 64-bit defaults.
    rows = []
    for sid, d in subjects.items():
        for r in d.get(table_name, []):
            row = dict(r)
            row["subject_id"] = sid
            row["chart_time"] = _safe_int(r.get("chart_time"))
            if value_is_float:
                row["value"] = _safe_float(r.get("value"))
            rows.append(row)
    df = pd.DataFrame(rows)
    if len(df):
        df["subject_id"] = df["subject_id"].astype("category")
        if "item_name" in df.columns:
            df["item_name"] = df["item_name"].astype("category")
        if "op_id" in df.columns:
            df["op_id"] = df["op_id"].astype("category")
        df["chart_time"] = pd.to_numeric(df["chart_time"], downcast="integer")
        if value_is_float and "value" in df.columns:
            df["value"] = pd.to_numeric(df["value"], downcast="float")
    return df

labs_df        = flatten_table(RAW_SUBJECTS, "labs")
ward_vitals_df = flatten_table(RAW_SUBJECTS, "ward_vitals")

# Intra-op vitals are ONLY used when TIME_WINDOW == 'peri_op' (Section 6.2/6.3, and the
# aggregated features in Section 6.15) -- under the default 'pre_op' setting nothing in
# this notebook ever reads vitals_df, so loading and flattening it is pure wasted memory
# and time, on a table that is the single largest of the three at full scale. Skip it
# entirely unless it will actually be used.
if CONFIG["TIME_WINDOW"] == "peri_op":
    vitals_df = flatten_table(RAW_SUBJECTS, "vitals")
else:
    vitals_df = pd.DataFrame(columns=["subject_id", "op_id", "chart_time", "item_name", "value"])
    print("TIME_WINDOW='pre_op' -- intra-op vitals table NOT loaded (not needed, saves the "
          "most memory of any single change in this notebook). Switch to 'peri_op' in Part 2 "
          "if you need it.")

def flatten_operations(subjects):
    rows = []
    for sid, d in subjects.items():
        for op in d.get("operations", []):
            row = dict(op)
            row["subject_id"] = sid
            for tcol in ["orin_time", "orout_time", "opstart_time", "opend_time",
                         "admission_time", "discharge_time", "anstart_time", "anend_time",
                         "icuin_time", "icuout_time"]:
                row[tcol] = _safe_int(row.get(tcol))
            for icol in ["age", "asa", "emop"]:
                row[icol] = _safe_int(row.get(icol))
            for fcol in ["weight", "height"]:
                row[fcol] = _safe_float(row.get(fcol))
            row["inhosp_death_time"] = _safe_int(row.get("inhosp_death_time"))
            row["allcause_death_time"] = _safe_int(row.get("allcause_death_time"))
            rows.append(row)
    return pd.DataFrame(rows)

operations_df = flatten_operations(RAW_SUBJECTS)

def flatten_diagnoses(subjects):
    rows = []
    for sid, d in subjects.items():
        for dx in d.get("diagnoses", []):
            rows.append({
                "subject_id": sid,
                "chart_time": _safe_int(dx.get("chart_time")),
                "icd10_cm": dx.get("icd10_cm"),
            })
    df = pd.DataFrame(rows)
    if len(df):
        df["subject_id"] = df["subject_id"].astype("category")
        df["icd10_cm"] = df["icd10_cm"].astype("category")
        df["chart_time"] = pd.to_numeric(df["chart_time"], downcast="integer")
    return df

diagnoses_df = flatten_diagnoses(RAW_SUBJECTS)

def flatten_medications(subjects):
    # Medications is the widest table (9 mostly-string columns) and, at full scale, one of
    # the largest by row count (~9.9M administrations per the source repo's README) -- the
    # dtype downcast below matters more here than almost anywhere else in the notebook.
    rows = []
    for sid, d in subjects.items():
        for m in d.get("medications", []):
            row = dict(m)
            row["subject_id"] = sid
            row["chart_time"] = _safe_int(row.get("chart_time"))
            rows.append(row)
    df = pd.DataFrame(rows)
    if len(df):
        for col in ["subject_id", "drug_name", "route", "atc_code", "drug_name2", "drug_name3", "atc_code2", "atc_code3"]:
            if col in df.columns:
                df[col] = df[col].astype("category")
        df["chart_time"] = pd.to_numeric(df["chart_time"], downcast="integer")
    return df

medications_df = flatten_medications(RAW_SUBJECTS)

print("labs_df        ", labs_df.shape)
print("vitals_df      ", vitals_df.shape, " (intra-op)")
print("ward_vitals_df ", ward_vitals_df.shape)
print("operations_df  ", operations_df.shape)
print("diagnoses_df   ", diagnoses_df.shape)
print("medications_df ", medications_df.shape)

print("\nActual memory used by each table (post dtype-optimization) -- watch this number, "
      "not a guess extrapolated from a different-sized run:")
_total_mb = 0
for _name, _df in [("labs_df", labs_df), ("vitals_df", vitals_df), ("ward_vitals_df", ward_vitals_df),
                    ("operations_df", operations_df), ("diagnoses_df", diagnoses_df), ("medications_df", medications_df)]:
    _mb = _df.memory_usage(deep=True).sum() / 1e6
    _total_mb += _mb
    print(f"  {_name:16s} {_mb:8.1f} MB")
print(f"  {'TOTAL':16s} {_total_mb:8.1f} MB")
print("\nNOTE: this dev subset (10 died / 20 survived, i.e. 33% mortality) is almost certainly "
      "denser than the true full cohort's average patient -- the source repo's own README "
      "states ~9.9M total medication administrations across 99,807 patients (~99/patient), "
      "while this 30-patient dev sample averages ~413/patient. Sicker, more-monitored dev-subset "
      "patients generate more data points than a typical patient, so a naive linear scale-up from "
      "this run's memory number will likely OVERESTIMATE the real full-scale footprint -- another "
      "reason to measure with a MAX_SUBJECTS_PER_CLASS smoke test rather than extrapolate blindly.")


In [ ]:

# Sanity check: exactly one operation row expected per (subject, op_id); most patients
# have one operation in this dev subset, some have several (relevant to §6.10/§6.11).
ops_per_subject = operations_df.groupby("subject_id")["op_id"].nunique()
print(ops_per_subject.value_counts().sort_index().rename("n_subjects_with_this_many_ops"))
operations_df.head(3)



### A note on memory strategy, before you reach for downsampling

If the memory printout above (or the full run) is tight, the order of things to try is:

1. **Confirm `TIME_WINDOW='pre_op'`** (the default) — this alone skips loading intra-op
   `vitals` entirely (just applied above), which is the single largest of the three
   time-series tables at full scale under linear scaling.
2. **Run a `MAX_SUBJECTS_PER_CLASS` smoke test first** (e.g. 2,000–5,000) and look at the
   real memory printout, rather than extrapolating from this dev subset — which, per the
   note above, is likely denser-than-average and will overestimate the real number.
3. **Only if still constrained, downsample — but the training `survived` split only, never
   `died`, and never validation/test** (Part 8.4 implements this, off by default). Two
   real costs of downsampling `survived` patients that are worth knowing before you turn
   it on: it shrinks the unlabelled pool the Part 9.4 autoencoder pre-training specifically
   relies on (a design decision made explicitly *because* splitting into six systems
   already splits the scarce mortality labels six ways), and if it ever touched
   validation/test it would corrupt the true ~0.47% prevalence that makes AUPRC meaningful
   — which is why it's restricted to the training split only.
4. **If you do downsample, stratify by `department`, not by ICD-10 "organ system."**
   Department is a single, always-populated categorical field per patient — clean to
   stratify on. ICD-10 diagnoses are multi-label (a patient's diagnoses can span several
   organ-system chapters at once), so there's no single unambiguous rule for assigning one
   patient to one "organ-system stratum" without inventing a new, separate convention just
   for this — and department is already the stratification key `§8.2`'s grouped SMOTENC
   uses, so reusing it keeps one consistent clinical-neighbourhood definition through the
   whole pipeline instead of two different ones.



# Part 4 — Item-name inventory, the 30-day mortality label, and the cohort table

Three things happen here, each directly implementing a decision from Part 1:

1. **§4.1** — list every distinct `item_name` in `labs` / `vitals` / `ward_vitals`. This is
   the programmatic check behind §1.2.3's claim that GI and MSK have no dedicated
   lab/vital panel — run it and see for yourself rather than taking the claim on faith.
2. **§4.2** — recompute the 30-day mortality label directly from `operations`, **not**
   from the `died`/`survived` folder name (§1.6.3's label-bug note).
3. **§4.3** — assemble one row per patient: label, static fields, multi-operation counts.


## 4.1 Item-name inventory — confirming §1.2.3 programmatically

In [ ]:

def item_name_inventory(*dfs_and_names):
    inv = {}
    for df, name in dfs_and_names:
        inv[name] = sorted(df["item_name"].dropna().unique().tolist()) if "item_name" in df.columns and len(df) else []
    return inv

INVENTORY = item_name_inventory((labs_df, "labs"), (vitals_df, "vitals (intra-op)"), (ward_vitals_df, "ward_vitals"))
for name, items in INVENTORY.items():
    print(f"{name}: {len(items)} distinct item_names")

# GI/MSK check: do any of these look like GI- or MSK-specific measurements?
gi_keywords = ["bowel", "stool", "gi_", "gastro", "bilirubin"]  # bilirubin included to show it's hepatic, see §1.2.3
msk_keywords = ["mobility", "joint", "rom", "muscle", "ortho"]
all_items = set(sum(INVENTORY.values(), []))
print("\nItems matching GI-ish keywords:", [i for i in all_items if any(k in i.lower() for k in gi_keywords)])
print("Items matching MSK-ish keywords:", [i for i in all_items if any(k in i.lower() for k in msk_keywords)])
print("\n-> Confirms §1.2.3: no dedicated GI or MSK signal in labs/vitals/ward_vitals.")
print("   GI and MSK must be built from ICD-10 diagnoses + department (done in §6.7-6.8).")


## 4.2 The 30-day mortality label — recomputed from `operations`, not the folder name

In [ ]:

MINUTES_PER_DAY = 24 * 60

def compute_labels(operations_df):
    # One row per subject. Mirrors subject.py's inhosp_death_30day() logic:
    # died = inhosp_death_time is set AND inhosp_death_time < orout_time(last op) + 30 days.
    # 'last op' = operation with the max orin_time for that subject (current repo convention;
    # see §1.6.3/CONFIG note below for why this is a genuine open question, not a settled one).
    rows = []
    for sid, g in operations_df.groupby("subject_id"):
        g = g.sort_values("orin_time")
        first_op = g.iloc[0]
        last_op = g.iloc[-1]
        n_ops = len(g)

        inhosp_death_time = first_op["inhosp_death_time"]   # same across all ops for a subject, per source repo's note
        allcause_death_time = first_op["allcause_death_time"]

        # NOTE: pandas stores our None sentinels as NaN once a column mixes numbers and
        # missing values, so `x is not None` silently fails here -- must use pd.notna().
        has_inhosp_death = pd.notna(inhosp_death_time)

        died_30day_from_last = False
        died_30day_from_first = False
        if has_inhosp_death:
            if pd.notna(last_op["orout_time"]):
                died_30day_from_last = inhosp_death_time < (last_op["orout_time"] + 30 * MINUTES_PER_DAY)
            if pd.notna(first_op["orout_time"]):
                died_30day_from_first = inhosp_death_time < (first_op["orout_time"] + 30 * MINUTES_PER_DAY)

        died_ever = has_inhosp_death

        rows.append({
            "subject_id": sid,
            "n_operations": n_ops,
            "age": last_op["age"],
            "sex": last_op["sex"],
            "asa": last_op["asa"],
            "emop": last_op["emop"],
            "department": last_op["department"],
            "antype": last_op.get("antype"),
            "weight": last_op["weight"],
            "height": last_op["height"],
            "op_id_last": last_op["op_id"],
            "orin_time_last": last_op["orin_time"],
            "orout_time_last": last_op["orout_time"],
            "admission_time_last": last_op["admission_time"],
            "discharge_time_last": last_op["discharge_time"],
            "died_ever": died_ever,                                   # NOT the training label - see §1.6.3
            "died_30day_from_last_op": died_30day_from_last,          # default training label (Path C = 'last operation', see §1.6.4/§11.7)
            "died_30day_from_first_op": died_30day_from_first,        # sensitivity-analysis alternative
        })
    return pd.DataFrame(rows)

cohort_df = compute_labels(operations_df)
cohort_df = cohort_df.merge(
    pd.Series(FOLDER_LABEL, name="folder_label").rename_axis("subject_id").reset_index(),
    on="subject_id", how="left"
)

# Cross-check against the folder label, exactly the check §1.6.3 says not to skip.
cohort_df["folder_says_died"] = cohort_df["folder_label"].eq("died")
mismatch = cohort_df[cohort_df["folder_says_died"] != cohort_df["died_30day_from_last_op"]]
print(f"Cohort: {len(cohort_df)} patients.")
print(f"  died_ever (all-cause, any time)      : {cohort_df['died_ever'].sum()}")
print(f"  died_30day_from_last_op (TRAINING LABEL) : {cohort_df['died_30day_from_last_op'].sum()}")
print(f"  died_30day_from_first_op (sensitivity)   : {cohort_df['died_30day_from_first_op'].sum()}")
print(f"  folder-label says 'died'             : {cohort_df['folder_says_died'].sum()}")
print(f"  Rows where folder label != recomputed 30-day label: {len(mismatch)}  "
      f"(non-zero here is expected and is exactly the §1.6.3 bug the recompute avoids inheriting)")
mismatch[["subject_id", "folder_says_died", "died_ever", "died_30day_from_last_op"]]


## 4.3 Multi-operation counts by department/system — feeds §6.10/§6.11

In [ ]:

# Per-subject operation history (all ops, in order) — used later for the "operations in the
# same area" feature (§6.10) and the post-cardiac-surgery exception flag (§6.11).
OPS_HISTORY = {}
for sid, g in operations_df.groupby("subject_id"):
    g = g.sort_values("orin_time")
    OPS_HISTORY[sid] = g.to_dict("records")

print("Departments observed:", sorted(operations_df["department"].dropna().unique().tolist()))
print("\nMortality (30-day, last-op label) by department:")
tmp = cohort_df.groupby("department")["died_30day_from_last_op"].agg(["mean", "count"])
print(tmp.sort_values("mean", ascending=False))

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
cohort_df["n_operations"].value_counts().sort_index().plot(kind="bar", ax=axes[0], color="#0a7d6e")
axes[0].set_title("Operations per patient"); axes[0].set_xlabel("n_operations"); axes[0].set_ylabel("n_patients")

dep_mort = cohort_df.groupby("department")["died_30day_from_last_op"].mean().sort_values(ascending=False)
dep_mort.plot(kind="bar", ax=axes[1], color="#c0392b")
axes[1].set_title("30-day mortality rate by department"); axes[1].set_ylabel("mortality rate")
plt.tight_layout(); plt.show()



# Part 5 — Exploratory data analysis

Four quick checks before any feature engineering, chosen because each one either validates
a Part 1 claim or surfaces something the modelling choices in Part 6–9 need to account for.
On the 30-patient dev subset these are illustrative, not statistically reliable — rerun
verbatim on the full cohort once available, exactly as the source repo's own roadmap
recommends (`docs/roadmap_and_architecture.md` item 2).


In [ ]:

# 5.1 ASA vs mortality -- sanity check that the data behaves the way the literature predicts
# (source repo's own EDA found a monotonic relationship at full scale; small-N here, but the
# direction should still be visible).
asa_mort = cohort_df.groupby("asa")["died_30day_from_last_op"].agg(["mean", "count"])
print(asa_mort)

fig, ax = plt.subplots(figsize=(5, 4))
asa_mort["mean"].plot(kind="bar", ax=ax, color="#2c3e50")
ax.set_title("30-day mortality rate by ASA class"); ax.set_xlabel("ASA class"); ax.set_ylabel("mortality rate")
plt.tight_layout(); plt.show()


In [ ]:

# 5.2 Missingness -- which pre-op labs/vitals are actually available per patient?
# This is the concrete evidence behind §1.3's MNAR discussion: a feature with very low
# pre-op coverage is either genuinely unmeasured or (per §1.3) not ordered because the
# clinician judged it unnecessary -- worth seeing the real numbers before choosing §7's
# imputation strategy.
def coverage_table(df, cohort_df, time_col="chart_time", window_minutes=None, op_time_lookup=None):
    # Fraction of patients with >=1 observation of each item_name, optionally restricted
    # to a pre-op window (window_minutes before orin_time).
    if window_minutes is not None:
        keep_rows = []
        for sid, g in df.groupby("subject_id"):
            orin = op_time_lookup.get(sid)
            if orin is None:
                continue
            lo = orin - window_minutes
            keep_rows.append(g[(g[time_col] >= lo) & (g[time_col] <= orin)])
        df = pd.concat(keep_rows) if keep_rows else df.iloc[0:0]
    n_patients = cohort_df["subject_id"].nunique()
    cov = df.groupby("item_name")["subject_id"].nunique() / n_patients
    return cov.sort_values(ascending=False)

orin_lookup = cohort_df.set_index("subject_id")["orin_time_last"].to_dict()
window = CONFIG["PRE_OP_DAYS"] * 24 * 60

labs_cov_preop = coverage_table(labs_df, cohort_df, window_minutes=window, op_time_lookup=orin_lookup)
wv_cov_preop   = coverage_table(ward_vitals_df, cohort_df, window_minutes=window, op_time_lookup=orin_lookup)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
labs_cov_preop.plot(kind="bar", ax=axes[0], color="#2980b9"); axes[0].set_title(f"Pre-op ({CONFIG['PRE_OP_DAYS']}d) lab coverage")
wv_cov_preop.plot(kind="bar", ax=axes[1], color="#8e44ad"); axes[1].set_title(f"Pre-op ({CONFIG['PRE_OP_DAYS']}d) ward-vital coverage")
for ax in axes:
    ax.set_ylabel("fraction of patients with >=1 value"); ax.tick_params(axis='x', labelsize=7)
plt.tight_layout(); plt.show()

print("Lowest-coverage pre-op labs (candidates for the MNAR discussion in §1.3):")
print(labs_cov_preop.tail(8))


In [ ]:

# 5.3 ICD-10 chapter distribution -- which chapters actually show up, and how do they
# relate to mortality? Directly relevant to §1.2.3 (GI = chapter XI, MSK = chapter XIII).

ICD10_CHAPTERS = [
    ('I', 'A00-B99', 'Certain infectious and parasitic diseases'),
    ('II', 'C00-D48', 'Neoplasms'),
    ('III', 'D50-D89', 'Diseases of the blood and blood-forming organs and certain disorders involving the immune mechanism'),
    ('IV', 'E00-E90', 'Endocrine, nutritional and metabolic diseases'),
    ('V', 'F00-F99', 'Mental and behavioural disorders'),
    ('VI', 'G00-G99', 'Diseases of the nervous system'),
    ('VII', 'H00-H59', 'Diseases of the eye and adnexia'),
    ('VIII', 'H60-H95', 'Diseases of the ear and mastoid process'),
    ('IX', 'I00-I99', 'Diseases of the circulatory system'),
    ('X', 'J00-J99', 'Diseases of the respiratory system'),
    ('XI', 'K00-K93', 'Diseases of the digestive system'),                       # <- GI
    ('XII', 'L00-L99', 'Diseases of the skin and subcutaneous tissue'),
    ('XIII', 'M00-M99', 'Diseases of the musculoskeletal system and connective tissue'),  # <- MSK
    ('XIV', 'N00-N99', 'Diseases of the genitourinary system'),
    ('XV', 'O00-O99', 'Pregnancy, childbirth and the puerperium'),
    ('XVI', 'P00-P96', 'Certain conditions originating in the perinatal period'),
    ('XVII', 'Q00-Q99', 'Congenital malformations, deformations and chromosomal abnormalities'),
    ('XVIII', 'R00-R99', 'Symptoms, signs and abnormal clinical and laboratory findings, not elsewhere classified'),
    ('XIX', 'S00-T98', 'Injury, poisoning and certain other consequences of external causes'),
    ('XX', 'V01-Y98', 'External causes of morbidity and mortality'),
    ('XXI', 'Z00-Z99', 'Factors influencing health status and contact with health services'),
    ('XXII', 'U00-U99', 'Codes for special purposes'),
]

def _in_block(code3, block):
    # code3: first 3 chars of an ICD-10-CM code, e.g. 'N18'. block: 'A00-B99'.
    if not code3 or len(code3) < 3:
        return False
    alpha = ord(code3[0].upper())
    try:
        numeric = int(code3[1:3])
    except ValueError:
        return False
    sblock, eblock = block.split('-')
    salpha, snum = ord(sblock[0]), int(sblock[1:])
    ealpha, enum = ord(eblock[0]), int(eblock[1:])
    return salpha <= alpha <= ealpha and snum <= numeric <= enum

def icd10_chapter(code):
    if not isinstance(code, str) or len(code) < 3:
        return None
    code3 = code[:3].upper()
    for numeral, block, desc in ICD10_CHAPTERS:
        if _in_block(code3, block):
            return numeral
    return None

diagnoses_df["chapter"] = diagnoses_df["icd10_cm"].apply(icd10_chapter)
chapter_desc = {c: d for c, b, d in ICD10_CHAPTERS}

chapter_counts = diagnoses_df["chapter"].value_counts()
print(chapter_counts)

# mortality rate among patients who have >=1 diagnosis in each chapter
chap_mort = {}
for chap in chapter_counts.index:
    sids = set(diagnoses_df.loc[diagnoses_df["chapter"] == chap, "subject_id"])
    sub = cohort_df[cohort_df["subject_id"].isin(sids)]
    if len(sub):
        chap_mort[chap] = (sub["died_30day_from_last_op"].mean(), len(sub))
chap_mort_df = pd.DataFrame(chap_mort, index=["mortality_rate", "n_patients"]).T.sort_values("mortality_rate", ascending=False)
chap_mort_df["description"] = [chapter_desc.get(c, "") for c in chap_mort_df.index]
chap_mort_df



# Part 6 — Organ-system feature engineering

This is the section that turns the theory in §1.1–§1.6 into actual feature tables. Order
of build, each tied back to a Part 1 section:

- §6.1 the organ-system → raw-signal grouping table (the routing map)
- §6.2 pre-op / peri-op time-window extraction per patient (§1.6.3)
- §6.3–§6.5 the four "genuine time series" systems: Renal, Respiratory, Metabolic/hepatic, Haematology
- §6.9 Cardiovascular, built *jointly* with Renal because of the coupling in §1.6.2
- §6.6 ICD-10 features: chapter flags/counts + HFRS (§1.2)
- §6.7–§6.8 GI and MSK (diagnosis/department-driven, §1.2.3)
- §6.10 operation-count-in-same-area features (§1.6.4)
- §6.11 the 6-month post-cardiac-surgery exception flag (§1.6.4, rule-based on purpose)
- §6.12 static features (age, sex, ASA, emop, department, weight, height) (§1.5.1, §1.2.2)
- §6.13 medications → ATC-level aggregate features (§1.5.3)
- §6.14 assembling everything into the final per-patient, per-system feature bundle



## 6.1 The organ-system → raw-signal routing map

This extends the six-system grouping already sketched in the source repo's own
`docs/roadmap_and_architecture.md` §4.1 with the two new systems from the meeting notes,
and is filtered to the item-names actually observed in §4.1's inventory (so nothing below
references a feature this dataset doesn't have).


In [ ]:

# --- Time-series systems: item_name -> system, split by source table ---
# Kept as an explicit, editable dict (not auto-derived) so the organ-system assignment is
# a reviewable, documented decision -- exactly the kind of thing you said you want to be
# able to interrogate and change yourself.

SYSTEM_LABS = {
    "renal":          ["bun", "calcium", "chloride", "creatinine", "ica", "phosphorus", "potassium", "sodium"],
    "cardiovascular":  ["ck", "ckmb", "troponin_i"],   # troponin_t not present in this subset's inventory (§4.1)
    "respiratory":    ["be", "hco3", "paco2", "pao2", "ph", "sao2"],
    "metabolic_hepatic": ["albumin", "alp", "alt", "ast", "glucose", "hba1c", "lacate", "total_bilirubin", "total_protein"],
    "haematology":    ["aptt", "crp", "fibrinogen", "hb", "hct", "lymphocyte", "platelet", "ptinr", "seg", "wbc"],
    "neurological":   [],   # no dedicated neuro lab in this dataset
}

SYSTEM_WARD_VITALS = {
    "renal":          ["crrt", "uo"],
    "cardiovascular":  ["hr", "nibp_sbp", "nibp_dbp", "nibp_mbp", "iabp"],
    "respiratory":    ["fio2", "rr", "spo2", "vent", "ecmo"],
    "metabolic_hepatic": ["bt"],
    "haematology":    [],
    "neurological":   ["gcs_e", "gcs_m", "gcs_v"],
}

SYSTEM_INTRAOP_VITALS = {   # only used when CONFIG['TIME_WINDOW'] == 'peri_op'
    "renal":          ["uo"],
    "cardiovascular":  ["hr", "art_sbp", "art_dbp", "art_mbp", "ci", "cvp", "svi",
                        "nibp_sbp", "nibp_dbp", "nibp_mbp", "pap_sbp", "pap_dbp", "pap_mbp"],
    "respiratory":    ["etco2", "fio2", "spo2", "peep", "pip", "pplat", "rr", "minvol", "vt"],
    "metabolic_hepatic": ["bt"],
    "haematology":    ["ebl", "rbc", "ffp"],
    "neurological":   ["bis"],
}

TIME_SERIES_SYSTEMS = ["renal", "cardiovascular", "respiratory", "metabolic_hepatic", "haematology", "neurological"]

# Sanity-check every listed item_name actually exists in this dataset's inventory (§4.1).
for sysdict, inv_key in [(SYSTEM_LABS, "labs"), (SYSTEM_WARD_VITALS, "ward_vitals"), (SYSTEM_INTRAOP_VITALS, "vitals (intra-op)")]:
    known = set(INVENTORY[inv_key])
    for system, items in sysdict.items():
        unknown = [i for i in items if i not in known]
        if unknown:
            print(f"WARNING: {inv_key}/{system} references unseen item_names: {unknown}")
print("Routing map validated against §4.1 inventory (no warnings above = every referenced item_name exists).")



## 6.2 Pre-op / peri-op window extraction

Implements §1.6.3's `CONFIG['TIME_WINDOW']` toggle. `'pre_op'` keeps only records in
`[orin_time - PRE_OP_DAYS*24*60, orin_time]` for the **last** operation (matching the label
definition in §4.2 — Path C, "from last operation"; §11.7 revisits first-operation and
exclude-multi-op as a sensitivity check). `'peri_op'` extends the upper bound to
`orout_time` and additionally pulls in intra-op `vitals`.


In [ ]:

def get_window(subject_id, cohort_row):
    orin = cohort_row["orin_time_last"]
    orout = cohort_row["orout_time_last"]
    lo = orin - CONFIG["PRE_OP_DAYS"] * 24 * 60
    if CONFIG["TIME_WINDOW"] == "pre_op":
        hi = orin
    elif CONFIG["TIME_WINDOW"] == "peri_op":
        hi = orout if pd.notna(orout) else orin
    else:
        raise ValueError(f"Unknown TIME_WINDOW {CONFIG['TIME_WINDOW']!r}")
    return lo, hi

COHORT_INDEXED = cohort_df.set_index("subject_id")

# --- Fast per-patient/feature lookup (replaces a full-table scan per call) -----------
# The naive version of windowed_series() -- filter the whole DataFrame by subject_id AND
# item_name AND time range, on every single call -- is fine on the 30-patient dev subset
# but does NOT scale: this notebook calls it roughly 60-70 times per patient across
# Part 6, so at the full ~99,886-patient cohort that's several million calls, each an
# O(table_size) scan over tables with tens of millions of rows. Realistically that's days,
# not hours -- it would exceed any Colab/Kaggle session limit long before the GPU or RAM
# did. The fix below groups each table ONCE into a plain Python dict keyed by
# (subject_id, item_name) -> (chart_time array, value array); every lookup after that is a
# dict hash lookup plus a small in-memory numpy filter, both essentially O(1) with respect
# to the table's total size. (An earlier version of this fix used a sorted pandas
# MultiIndex + .loc[] instead -- measured at only ~20-90x faster than the naive scan on
# benchmark tables, versus >4,000x for the plain-dict version below: pandas' .loc[] call
# overhead dominates for small per-key results, which is exactly this notebook's access
# pattern. Benchmarked below rather than assumed.) Output is identical either way -- this
# only changes speed, verified by confirming every printed count/statistic in this
# notebook is unchanged after the switch.
_GROUPED_ARRAYS_CACHE = {}

def _get_grouped_arrays(df):
    key = id(df)
    if key not in _GROUPED_ARRAYS_CACHE:
        grouped = {}
        if len(df) and "item_name" in df.columns:
            for gkey, g in df.groupby(["subject_id", "item_name"], observed=True, sort=False):
                grouped[gkey] = (g["chart_time"].to_numpy(), g["value"].to_numpy())
        _GROUPED_ARRAYS_CACHE[key] = grouped
    return _GROUPED_ARRAYS_CACHE[key]

def windowed_series(df, subject_id, item_name, lo, hi, subj_col="subject_id"):
    # Returns a sorted (chart_time -> value) dict for one patient/feature within [lo, hi].
    if len(df) == 0 or "item_name" not in df.columns:
        return {}
    grouped = _get_grouped_arrays(df)
    arrs = grouped.get((subject_id, item_name))
    if arrs is None:
        return {}
    times, vals = arrs
    mask = (times >= lo) & (times <= hi) & ~np.isnan(vals.astype(float))
    if not mask.any():
        return {}
    t_sel, v_sel = times[mask], vals[mask]
    order = np.argsort(t_sel)
    return dict(zip(t_sel[order], v_sel[order]))

print(f"TIME_WINDOW = {CONFIG['TIME_WINDOW']!r}  (see §1.6.3 for what each option represents)")



### A note on why this matters more than it looks

The grouped-lookup rewrite above is the single highest-impact change in this notebook for
runnability at full scale — more than any of the memory optimizations. A self-contained
timing comparison on synthetic data of a similar shape to the full cohort (run once,
below, to make this concrete rather than asserted) shows the real speedup, split into the
one-time index-build cost (paid once per table, however large the cohort) and the
marginal per-lookup cost (paid millions of times, so this is the number that actually
determines whether a full run finishes in minutes/hours or days).


In [ ]:

import time

def _make_synthetic_table(n_subjects, avg_rows_per_subject, n_item_names=40):
    rng = np.random.default_rng(0)
    n_rows = n_subjects * avg_rows_per_subject
    subject_ids = rng.integers(0, n_subjects, size=n_rows).astype(str)
    item_names = rng.choice([f"item_{i}" for i in range(n_item_names)], size=n_rows)
    chart_times = rng.integers(-10000, 10000, size=n_rows)
    values = rng.normal(size=n_rows).astype(np.float32)
    df = pd.DataFrame({"subject_id": subject_ids, "item_name": item_names,
                        "chart_time": chart_times, "value": values})
    df["subject_id"] = df["subject_id"].astype("category")
    df["item_name"] = df["item_name"].astype("category")
    return df

_SYN_N_SUBJECTS = 2000
_synthetic_df = _make_synthetic_table(_SYN_N_SUBJECTS, avg_rows_per_subject=200)
print(f"Synthetic benchmark table: {len(_synthetic_df):,} rows, {_SYN_N_SUBJECTS:,} subjects "
      f"(a small slice of the full cohort's scale, kept small here so the OLD method's demo "
      f"below finishes quickly -- the gap only widens at the real ~99,886-patient size)")

def _old_windowed_series(df, subject_id, item_name, lo, hi):
    sub = df[(df["subject_id"] == subject_id) & (df["item_name"] == item_name) &
             (df["chart_time"] >= lo) & (df["chart_time"] <= hi)]
    return dict(zip(sub["chart_time"], sub["value"]))

_sample_subject_ids = np.random.default_rng(1).choice(_synthetic_df["subject_id"].cat.categories, size=150, replace=True)
_sample_items = [f"item_{i}" for i in range(6)]
n_calls = len(_sample_subject_ids) * len(_sample_items)

_t0 = time.time()
for sid in _sample_subject_ids:
    for item in _sample_items:
        _old_windowed_series(_synthetic_df, sid, item, -10000, 10000)
_old_elapsed = time.time() - _t0

_GROUPED_ARRAYS_CACHE.clear()
_t0 = time.time()
_ = windowed_series(_synthetic_df, _sample_subject_ids[0], _sample_items[0], -10000, 10000)   # triggers the one-time group build
_build_elapsed = time.time() - _t0

_t0 = time.time()
for sid in _sample_subject_ids:
    for item in _sample_items:
        windowed_series(_synthetic_df, sid, item, -10000, 10000)
_new_lookups_elapsed = time.time() - _t0

old_ms_per_call = _old_elapsed / n_calls * 1000
new_ms_per_call = _new_lookups_elapsed / n_calls * 1000
print(f"\n{n_calls:,} lookups against a {len(_synthetic_df):,}-row table:")
print(f"  old (full-scan every call):    {_old_elapsed:.2f}s total  ({old_ms_per_call:.3f} ms/call)")
print(f"  new -- one-time group build:   {_build_elapsed:.2f}s (paid ONCE per table for the whole notebook run)")
print(f"  new -- {n_calls:,} grouped lookups: {_new_lookups_elapsed:.4f}s  ({new_ms_per_call:.5f} ms/call)")
print(f"  marginal per-call speedup:     {old_ms_per_call/max(new_ms_per_call,1e-9):.0f}x  "
      f"(this is the number that matters at full scale, where millions of calls share one build)")
print(f"\nThe one-time build cost scales with table size (more rows/groups to organise), so at "
      f"the real full-cohort scale expect the build itself to take real minutes, not seconds -- "
      f"that is an acceptable, one-time cost given it replaces what would otherwise be a "
      f"multi-day total runtime under the old per-call scan.")

del _synthetic_df
_GROUPED_ARRAYS_CACHE.clear()   # drop the benchmark's groups; the real tables get grouped fresh on first real use



## 6.3 Per-system time-series extraction, resampling, and the missingness mask

For each patient and each of the six time-series systems (§6.1), build a
`[TARGET_SEQ_LEN, n_features_in_system]` array plus a same-shaped **mask** array
(1 = observed/interpolated-from-real-data, 0 = no data at all for that patient/feature —
see §1.3's MNAR discussion for why the mask is kept as its own feature rather than
discarded once a value is filled in). The actual *value*-filling strategy is deliberately
left as a placeholder here (`_TODO_impute`) and implemented properly in §7, once all the
methods have been introduced together — this section only handles resampling onto a
common time grid, which is method-independent.


In [ ]:

def resample_to_grid(chart_time2value, lo, hi, n_points):
    # Evenly-spaced target grid of n_points between lo and hi; nearest-observation lookup
    # per grid point, returned alongside a raw (unimputed) array with NaN for empty points.
    # Actual interpolation/imputation happens in §7 -- this function only regularises timing.
    grid = np.linspace(lo, hi, n_points)
    if len(chart_time2value) == 0:
        return grid, np.full(n_points, np.nan)
    times = np.array(sorted(chart_time2value.keys()), dtype=float)
    values = np.array([chart_time2value[t] for t in sorted(chart_time2value.keys())], dtype=float)
    out = np.full(n_points, np.nan)
    for i, g in enumerate(grid):
        idx = np.searchsorted(times, g)
        # nearest of the two neighbouring observed points, used later as the "observed value"
        # a real imputation strategy (§7) then fills the true gaps.
        candidates = []
        if idx < len(times):
            candidates.append((abs(times[idx] - g), values[idx]))
        if idx > 0:
            candidates.append((abs(times[idx - 1] - g), values[idx - 1]))
        if candidates:
            # only accept "nearest observed" as a real observation if it's within one grid step,
            # otherwise leave it NaN so §7's imputation (not this raw resample) fills the gap
            step = (hi - lo) / max(n_points - 1, 1)
            best = min(candidates, key=lambda c: c[0])
            if best[0] <= max(step, 1e-6):
                out[i] = best[1]
    return grid, out

def extract_system_tensor(subject_id, system, lo, hi):
    # Returns (raw_values [T,F] with NaN for gaps, mask [T,F], feature_names) for one
    # patient/system, pooling labs + ward_vitals (+ intra-op vitals if peri_op).
    feature_names = []
    columns = []
    sources = [(labs_df, SYSTEM_LABS[system]), (ward_vitals_df, SYSTEM_WARD_VITALS[system])]
    if CONFIG["TIME_WINDOW"] == "peri_op":
        sources.append((vitals_df, SYSTEM_INTRAOP_VITALS[system]))
    for df, items in sources:
        for item in items:
            series = windowed_series(df, subject_id, item, lo, hi)
            grid, raw = resample_to_grid(series, lo, hi, CONFIG["TARGET_SEQ_LEN"])
            feature_names.append(item)
            columns.append(raw)
    if not columns:
        return (np.zeros((CONFIG["TARGET_SEQ_LEN"], 0)),
                np.zeros((CONFIG["TARGET_SEQ_LEN"], 0)),
                [])
    raw_values = np.stack(columns, axis=1)          # [T, F]
    mask = (~np.isnan(raw_values)).astype(np.float32)
    return raw_values, mask, feature_names

# Build the raw (pre-imputation) tensors for every patient x system, kept in a dict so §7
# can impute in place without re-doing the windowing/resampling work.
RAW_SYSTEM_TENSORS = {}   # (subject_id, system) -> dict(raw=[T,F], mask=[T,F], feature_names=[...])
for sid in cohort_df["subject_id"]:
    row = COHORT_INDEXED.loc[sid]
    lo, hi = get_window(sid, row)
    for system in TIME_SERIES_SYSTEMS:
        raw, mask, fnames = extract_system_tensor(sid, system, lo, hi)
        RAW_SYSTEM_TENSORS[(sid, system)] = {"raw": raw, "mask": mask, "feature_names": fnames}

# Quick coverage summary: for each system, what fraction of (patient, feature) cells have
# at least one real observation in the window?
for system in TIME_SERIES_SYSTEMS:
    fnames = RAW_SYSTEM_TENSORS[(cohort_df["subject_id"].iloc[0], system)]["feature_names"]
    if not fnames:
        print(f"{system:18s}: no features assigned (see §6.1)")
        continue
    total, observed = 0, 0
    for sid in cohort_df["subject_id"]:
        m = RAW_SYSTEM_TENSORS[(sid, system)]["mask"]
        total += m.shape[0] * m.shape[1]
        observed += m.sum()
    print(f"{system:18s}: {len(fnames):2d} features, {observed/total:5.1%} of (patient,timepoint,feature) cells observed in-window")



## 6.9 Cardiovascular ↔ Renal coupling (§1.6.2)

Two concrete implementations of the meeting note "renal directly proportionate to
cardiovascular," matching the two mechanisms described in §1.6.2:

1. **Architectural**: a compact cardiovascular summary (mean heart rate, mean arterial
   pressure deviation from normal, presence of an IABP flag) is computed here and appended
   to the *renal* branch's static side-input in §9 — so the renal encoder sees
   cardiovascular context directly, not just renal labs.
2. **Hand-crafted interaction feature**: `renal_cardiac_interaction`, a creatinine ×
   blood-pressure-deviation product, added to the static feature table (§6.14) as a
   cheap, sample-efficient prior (§1.6.2 explains why this is added *alongside* the
   learned coupling, not instead of it).

`CONFIG['SYMMETRIC_CARDIORENAL_COUPLING']` toggles whether the *cardiovascular* branch
symmetrically receives a renal summary back — off by default, per the §1.6.2 reasoning.


In [ ]:

NORMAL_MAP_MMHG = 93.0   # a commonly used "normal" mean arterial pressure reference point

def cardiac_summary_for_patient(sid, lo, hi):
    # A small, fixed-size cardiovascular summary vector -- fed into the renal branch (§9)
    # as the architectural half of the cardiorenal coupling, and used below to build the
    # hand-crafted interaction feature.
    hr_series = windowed_series(ward_vitals_df, sid, "hr", lo, hi)
    map_series = windowed_series(ward_vitals_df, sid, "nibp_mbp", lo, hi)
    iabp_series = windowed_series(ward_vitals_df, sid, "iabp", lo, hi)

    mean_hr = float(np.mean(list(hr_series.values()))) if hr_series else np.nan
    mean_map = float(np.mean(list(map_series.values()))) if map_series else np.nan
    map_deviation = (mean_map - NORMAL_MAP_MMHG) if not np.isnan(mean_map) else np.nan
    has_iabp = float(len(iabp_series) > 0 and any(v > 0 for v in iabp_series.values()))

    return {"cardio_mean_hr": mean_hr, "cardio_map_deviation": map_deviation, "cardio_has_iabp": has_iabp}

CARDIAC_SUMMARY = {}
for sid in cohort_df["subject_id"]:
    row = COHORT_INDEXED.loc[sid]
    lo, hi = get_window(sid, row)
    CARDIAC_SUMMARY[sid] = cardiac_summary_for_patient(sid, lo, hi)

cardiac_summary_df = pd.DataFrame(CARDIAC_SUMMARY).T
cardiac_summary_df.index.name = "subject_id"
print(f"Cardiovascular summary built for {len(cardiac_summary_df)} patients "
      f"(NaNs below reflect real missingness -- filled properly in §7, not here).")
cardiac_summary_df.describe()


In [ ]:

def renal_cardiac_interaction_for_patient(sid, lo, hi):
    # Hand-crafted §1.6.2 interaction feature: creatinine x |MAP deviation|.
    # NaN-safe -- returns NaN if either side is unavailable, filled like any other
    # static feature in §7.
    creat_series = windowed_series(labs_df, sid, "creatinine", lo, hi)
    mean_creat = float(np.mean(list(creat_series.values()))) if creat_series else np.nan
    map_dev = CARDIAC_SUMMARY[sid]["cardio_map_deviation"]
    if np.isnan(mean_creat) or (map_dev is None) or np.isnan(map_dev):
        return np.nan
    return mean_creat * abs(map_dev)

RENAL_CARDIAC_INTERACTION = {
    sid: renal_cardiac_interaction_for_patient(sid, *get_window(sid, COHORT_INDEXED.loc[sid]))
    for sid in cohort_df["subject_id"]
}
pd.Series(RENAL_CARDIAC_INTERACTION, name="renal_cardiac_interaction").describe()



## 6.6 / 6.7 / 6.8 — ICD-10 chapter features, HFRS, GI and MSK systems, infection flag

All built together because they share the same underlying computation (diagnosis chapter
membership within a lookback window) — GI and MSK (§1.2.3, §1.6.1) are simply the two
chapters the meeting notes called out by name, exposed as their own system entries rather
than folded into a generic "diagnosis count" feature.

- **GI** = ICD-10 chapter **XI** (`K00-K93`) flag/count + `department == 'GS'` (general
  surgery, the closest department proxy for GI surgical burden in this dataset).
- **MSK** = ICD-10 chapter **XIII** (`M00-M99`) flag/count + `department == 'OS'`
  (orthopaedic surgery).
- **HFRS** (Hospital Frailty Risk Score, Gilbert et al. 2018) — a validated, externally
  weighted score over 109 specific ICD-10 codes, restricted here to the published lookback
  window (`CONFIG['HFRS_LOOKBACK_YEARS']`, default 2, and only meaningfully applied for
  patients 75+ per the original paper — see §1.2.1). **This revision uses the full 109-code
  weights table**, extracted directly from the source repo's `frailty_hfrs.py` (an earlier
  revision of this notebook shipped a ~30-code representative subset — that limitation,
  flagged in `Multimodal_Notebook_Summary.md` §7, is now closed).
- **Infection/inflammation flag** — the cross-cutting signal `roadmap_and_architecture.md`
  §4.1a designed but had not yet implemented (until this revision). Per SOFA (Vincent et
  al. 1996) and Sepsis-3 (Singer et al. 2016), infection is modelled as a **modifier
  layered across systems**, not a competing 7th/9th organ system — implemented here as a
  small set of static features (chapter-I diagnosis flag, the six specific high-mortality
  codes already flagged in the source repo's EDA, fever, abnormal WBC, and a data-driven
  "elevated CRP" flag — see the code cell for why CRP uses a relative, not absolute,
  threshold) plus a 0–5 composite count, fed into the static branch alongside GI/MSK/HFRS.


In [ ]:

# Full Hospital Frailty Risk Score weights table (Gilbert et al. 2018, Table A2, 109
# ICD-10-CM clusters), extracted directly from the source repo's frailty_hfrs.py so this
# notebook stays self-contained/Kaggle-portable without importing that module.
HFRS_WEIGHTS = {
    'A04': 1.1, 'A09': 1.1, 'A41': 1.6, 'B95': 1.7, 'B96': 2.9, 'D64': 0.4,
    'E05': 0.9, 'E16': 1.4, 'E53': 1.9, 'E55': 1.0, 'E83': 0.4, 'E86': 2.3,
    'E87': 2.3, 'F00': 7.1, 'F01': 2.0, 'F03': 2.1, 'F05': 3.2, 'F10': 0.7,
    'F32': 0.5, 'G20': 1.8, 'G30': 4.0, 'G31': 1.2, 'G40': 1.5, 'G45': 1.2,
    'G81': 4.4, 'H54': 1.9, 'H91': 0.9, 'I63': 0.8, 'I67': 2.6, 'I69': 3.7,
    'I95': 1.6, 'J18': 1.1, 'J22': 0.7, 'J69': 1.0, 'J96': 1.5, 'K26': 1.6,
    'K52': 0.3, 'K59': 1.8, 'K92': 0.8, 'L03': 2.0, 'L08': 0.4, 'L89': 1.7,
    'L97': 1.6, 'M15': 0.4, 'M19': 1.5, 'M25': 2.3, 'M41': 0.9, 'M48': 0.5,
    'M79': 1.1, 'M80': 0.8, 'M81': 1.4, 'N17': 1.8, 'N18': 1.4, 'N19': 1.6,
    'N20': 0.7, 'N28': 1.3, 'N39': 3.2, 'R00': 0.7, 'R02': 1.0, 'R11': 0.3,
    'R13': 0.8, 'R26': 2.6, 'R29': 3.6, 'R31': 3.0, 'R32': 1.2, 'R33': 1.3,
    'R40': 2.5, 'R41': 2.7, 'R44': 1.6, 'R45': 1.2, 'R47': 1.0, 'R50': 0.1,
    'R54': 2.2, 'R55': 1.8, 'R56': 2.6, 'R63': 0.9, 'R69': 1.3, 'R79': 0.6,
    'R94': 1.4, 'S00': 3.2, 'S01': 1.1, 'S06': 2.4, 'S09': 1.2, 'S22': 1.8,
    'S32': 1.4, 'S42': 2.3, 'S51': 0.5, 'S72': 1.4, 'S80': 2.0, 'T83': 2.4,
    'U80': 0.8, 'W01': 0.9, 'W06': 1.1, 'W10': 0.9, 'W18': 2.1, 'W19': 3.2,
    'X59': 1.5, 'Y84': 0.7, 'Y95': 1.2, 'Z22': 1.7, 'Z50': 2.1, 'Z60': 1.8,
    'Z73': 0.6, 'Z74': 1.1, 'Z75': 2.0, 'Z87': 1.5, 'Z91': 0.5, 'Z93': 1.0,
    'Z99': 0.8,
}
assert len(HFRS_WEIGHTS) == 109, f"Expected 109 HFRS codes (Gilbert et al. Table A2), got {len(HFRS_WEIGHTS)}"

def compute_hfrs(sid, lo, hi, age):
    # Sum of HFRS weights for matching diagnosis codes within the lookback window,
    # restricted to patients aged 75+ (per Gilbert et al.'s validated population -- Sec 1.2.1).
    # Returns 0.0 (not NaN) for younger patients: HFRS is defined as not-applicable, not
    # missing, below the validated age range.
    if pd.isna(age) or age < 75:
        return 0.0
    lookback_minutes = CONFIG["HFRS_LOOKBACK_YEARS"] * 365 * 24 * 60
    dx = diagnoses_df[(diagnoses_df["subject_id"] == sid) &
                       (diagnoses_df["chart_time"] >= lo - lookback_minutes) &
                       (diagnoses_df["chart_time"] <= hi)]
    score = 0.0
    for code in dx["icd10_cm"].dropna():
        code3 = str(code)[:3].upper()
        if code3 in HFRS_WEIGHTS:
            score += HFRS_WEIGHTS[code3]
    return score

def gi_msk_icd10_features(sid, lo, hi, department):
    dx = diagnoses_df[(diagnoses_df["subject_id"] == sid) &
                       (diagnoses_df["chart_time"] >= lo) & (diagnoses_df["chart_time"] <= hi)]
    chapters = dx["icd10_cm"].dropna().apply(icd10_chapter)
    n_chapters = chapters.value_counts()
    return {
        "gi_icd10_count":  int(n_chapters.get("XI", 0)),
        "gi_icd10_flag":   float(n_chapters.get("XI", 0) > 0),
        "gi_department_flag": float(department == "GS"),
        "msk_icd10_count": int(n_chapters.get("XIII", 0)),
        "msk_icd10_flag":  float(n_chapters.get("XIII", 0) > 0),
        "msk_department_flag": float(department == "OS"),
        "n_diagnoses_in_window": int(len(dx)),
        "n_distinct_chapters_in_window": int(chapters.nunique()),
    }

ICD10_FEATURES = {}
for sid in cohort_df["subject_id"]:
    row = COHORT_INDEXED.loc[sid]
    lo, hi = get_window(sid, row)
    feats = gi_msk_icd10_features(sid, lo, hi, row["department"])
    if CONFIG["INCLUDE_HFRS"]:
        feats["hfrs"] = compute_hfrs(sid, lo, hi, row["age"])
    ICD10_FEATURES[sid] = feats

icd10_features_df = pd.DataFrame(ICD10_FEATURES).T
icd10_features_df.index.name = "subject_id"
print(f"GI flag positive: {int(icd10_features_df['gi_icd10_flag'].sum())} / {len(icd10_features_df)} patients")
print(f"MSK flag positive: {int(icd10_features_df['msk_icd10_flag'].sum())} / {len(icd10_features_df)} patients")
if CONFIG["INCLUDE_HFRS"]:
    n_scored = int((icd10_features_df["hfrs"] > 0).sum())
    print(f"Patients with HFRS > 0 (i.e. aged 75+ with a matching code): {n_scored} / {len(icd10_features_df)}")
icd10_features_df.describe()



### Infection / inflammation cross-cutting flag (`roadmap_and_architecture.md` §4.1a)

Six components, each a real, cheap-to-compute signal rather than a learned sub-model —
consistent with the design note that this is a *modifier*, not a competing organ system.
The CRP threshold is deliberately **data-driven (this cohort's own 75th percentile)**
rather than an absolute clinical cutoff (e.g. "CRP > 100 mg/L") -- this notebook never
loaded `parameters.csv`, so CRP's exact reporting unit for this INSPIRE export isn't
independently confirmed here, and a wrong absolute threshold is worse than an honestly
relative one. Fever and abnormal-WBC thresholds use standard, unit-unambiguous clinical
cutoffs (°C and count/volume respectively), so those stay absolute.


In [ ]:

HIGH_RISK_INFECTION_CODES = {"D65", "I46", "R57", "J80", "K72", "A41"}   # from the source repo's own EDA (docs/eda_findings.md)
FEVER_THRESHOLD_C = 38.0
WBC_NORMAL_RANGE = (4.0, 11.0)   # x10^9/L, standard adult reference range

# Data-driven CRP threshold: computed once from the training-eligible population (here,
# the whole cohort at bundle-build time, since this runs before the train/val/test split
# in Part 8 -- acceptable for a threshold that's a description of the cohort's own
# distribution, not a fitted statistic that could leak label information).
_all_crp_values = labs_df.loc[labs_df["item_name"] == "crp", "value"].dropna()
CRP_ELEVATED_THRESHOLD = float(_all_crp_values.quantile(0.75)) if len(_all_crp_values) else None
print(f"CRP 'elevated' threshold (75th percentile of all observed CRP in this cohort): "
      f"{CRP_ELEVATED_THRESHOLD}")

def infection_inflammation_features(sid, lo, hi):
    dx = diagnoses_df[(diagnoses_df["subject_id"] == sid) &
                       (diagnoses_df["chart_time"] >= lo) & (diagnoses_df["chart_time"] <= hi)]
    codes = set(dx["icd10_cm"].dropna().astype(str))
    chapter_i_flag = float(any(icd10_chapter(c) == "I" for c in codes))
    high_risk_flag = float(len(codes & HIGH_RISK_INFECTION_CODES) > 0)

    bt_series = windowed_series(ward_vitals_df, sid, "bt", lo, hi)
    fever_flag = float(len(bt_series) > 0 and max(bt_series.values()) >= FEVER_THRESHOLD_C)

    wbc_series = windowed_series(labs_df, sid, "wbc", lo, hi)
    wbc_abnormal_flag = 0.0
    if wbc_series:
        wbc_vals = list(wbc_series.values())
        wbc_abnormal_flag = float(any(v < WBC_NORMAL_RANGE[0] or v > WBC_NORMAL_RANGE[1] for v in wbc_vals))

    crp_series = windowed_series(labs_df, sid, "crp", lo, hi)
    crp_elevated_flag = 0.0
    if crp_series and CRP_ELEVATED_THRESHOLD is not None:
        crp_elevated_flag = float(max(crp_series.values()) >= CRP_ELEVATED_THRESHOLD)

    composite = chapter_i_flag + high_risk_flag + fever_flag + wbc_abnormal_flag + crp_elevated_flag

    return {
        "infection_chapter_i_flag": chapter_i_flag,
        "infection_high_risk_code_flag": high_risk_flag,
        "infection_fever_flag": fever_flag,
        "infection_wbc_abnormal_flag": wbc_abnormal_flag,
        "infection_crp_elevated_flag": crp_elevated_flag,
        "infection_composite_score": composite,   # 0-5, a simple count of the above
    }

INFECTION_FEATURES = {}
for sid in cohort_df["subject_id"]:
    row = COHORT_INDEXED.loc[sid]
    lo, hi = get_window(sid, row)
    INFECTION_FEATURES[sid] = infection_inflammation_features(sid, lo, hi)

infection_features_df = pd.DataFrame(INFECTION_FEATURES).T
infection_features_df.index.name = "subject_id"
print(f"\nPatients with infection_composite_score > 0: "
      f"{(infection_features_df['infection_composite_score'] > 0).sum()} / {len(infection_features_df)}")
infection_features_df.describe()



## 6.10 Operation-count-in-same-area features (§1.6.4)

For the **last** operation (the one the label is computed relative to, §4.2), count how
many *prior* operations that patient had in the **same department** and compute the time
gap in days since the most recent one. This is offered to the model as a feature, on
purpose — unlike the exception in §6.11 below, whether repeat-same-area surgery predicts
higher or lower risk is exactly the kind of question this notebook's own framing (§1.6.4)
says should be *learned*, not assumed.


In [ ]:

def operation_count_features(sid):
    history = OPS_HISTORY[sid]   # sorted by orin_time, built in §4.3
    if len(history) <= 1:
        return {"n_prior_ops_same_dept": 0, "days_since_last_op_same_dept": np.nan,
                "n_prior_ops_any_dept": 0, "days_since_last_op_any_dept": np.nan}
    last_op = history[-1]
    prior_ops = history[:-1]
    same_dept = [op for op in prior_ops if op.get("department") == last_op.get("department")]

    def days_since(ops):
        if not ops:
            return np.nan
        most_recent = max(op["orin_time"] for op in ops if op.get("orin_time") is not None)
        return (last_op["orin_time"] - most_recent) / (24 * 60)

    return {
        "n_prior_ops_same_dept": len(same_dept),
        "days_since_last_op_same_dept": days_since(same_dept),
        "n_prior_ops_any_dept": len(prior_ops),
        "days_since_last_op_any_dept": days_since(prior_ops),
    }

OP_COUNT_FEATURES = {sid: operation_count_features(sid) for sid in cohort_df["subject_id"]}
op_count_df = pd.DataFrame(OP_COUNT_FEATURES).T
op_count_df.index.name = "subject_id"
print(f"Patients with >=1 prior operation in the same department as their last: "
      f"{(op_count_df['n_prior_ops_same_dept'] > 0).sum()} / {len(op_count_df)}")
op_count_df.describe()



## 6.11 The 6-month post-cardiovascular-surgery exception (§1.6.4)

Implemented as a **rule**, deliberately not a learned feature, for the reasons argued in
§1.6.4 (sample size + actionability). `CTS` (cardiothoracic surgery) is used as this
dataset's department proxy for "cardiovascular surgery" — check this mapping against your
site's actual department coding before trusting the flag on a new cohort.


In [ ]:

CARDIAC_DEPARTMENTS = {"CTS"}       # adjust if your site's department coding differs
MIN_RECOVERY_DAYS_AFTER_CARDIAC_SURGERY = 180   # ~6 months, the clinical protocol window

def cardiac_recovery_exception_flag(sid):
    # 1.0 if the patient's last operation occurred less than 180 days after a PRIOR
    # cardiovascular-department operation -- i.e. the protocol window was violated (or the
    # two operations are causally linked, e.g. a staged/urgent re-intervention). This is a
    # flag for the model and for manual review, not an automatic exclusion -- see §11.6 for
    # the follow-up qualitative check this notebook recommends running once the model exists.
    history = OPS_HISTORY[sid]
    if len(history) <= 1:
        return 0.0
    last_op = history[-1]
    prior_cardiac_ops = [op for op in history[:-1] if op.get("department") in CARDIAC_DEPARTMENTS]
    if not prior_cardiac_ops:
        return 0.0
    most_recent_cardiac = max(op["orin_time"] for op in prior_cardiac_ops if op.get("orin_time") is not None)
    gap_days = (last_op["orin_time"] - most_recent_cardiac) / (24 * 60)
    return float(gap_days < MIN_RECOVERY_DAYS_AFTER_CARDIAC_SURGERY)

CARDIAC_EXCEPTION_FLAG = {sid: cardiac_recovery_exception_flag(sid) for sid in cohort_df["subject_id"]}
n_flagged = sum(CARDIAC_EXCEPTION_FLAG.values())
print(f"Patients flagged (last op within 6 months of a prior CTS op): {int(n_flagged)} / {len(CARDIAC_EXCEPTION_FLAG)}")
print("(Low or zero counts are expected on this 30-patient dev subset -- re-run on the full "
      "cohort, where cardiothoracic re-intervention is far more likely to appear.)")



## 6.12 Static features (§1.5.1, §1.2.2)

Age, sex, ASA, `emop` (emergency-operation flag), department (one-hot), weight, height —
100% coverage, cheap, and (per §1.2.2/§1.6.1) currently unused by the DNN in the source
repo despite ASA being one of the strongest established predictors in the literature.
Included here by default via `CONFIG['INCLUDE_STATIC_ASA']`.


In [ ]:

def static_features_row(row):
    return {
        "age": row["age"],
        "sex_F": float(row["sex"] == "F"),
        "asa": row["asa"] if CONFIG["INCLUDE_STATIC_ASA"] else np.nan,
        "emop": row["emop"],
        "weight": row["weight"],
        "height": row["height"],
        "n_operations_total": row["n_operations"],
    }

static_df = cohort_df.set_index("subject_id").apply(static_features_row, axis=1, result_type="expand")
dept_onehot = pd.get_dummies(cohort_df.set_index("subject_id")["department"], prefix="dept").astype(float)
static_df = pd.concat([static_df, dept_onehot], axis=1)
static_df.describe()



## 6.13 Medications → ATC-level aggregate features (§1.5.3)

Per §1.5.3's decision, this notebook aggregates at the **WHO ATC level-2** (e.g. the first
3 characters of the ATC code, roughly the "therapeutic subgroup" level) rather than
per-drug — far fewer categories, each with enough examples in a small cohort to be a
usable count feature rather than a near-empty one-hot. A learned per-class embedding
(rather than a raw count) is a natural upgrade once the full cohort is available — flagged
as an extension, not built by default here, per §1.5.3's reasoning about label-starved
high-cardinality embeddings.


In [ ]:

def atc_level2(code):
    if not isinstance(code, str) or len(code) < 3:
        return None
    return code[:3].upper()

def medication_features(sid, lo, hi):
    meds = medications_df[(medications_df["subject_id"] == sid) &
                           (medications_df["chart_time"] >= lo) & (medications_df["chart_time"] <= hi)]
    atc2 = meds["atc_code"].dropna().apply(atc_level2)
    return {
        "n_medication_administrations": int(len(meds)),
        "n_distinct_atc2_classes": int(atc2.nunique()),
        "n_distinct_drug_names": int(meds["drug_name"].nunique()) if "drug_name" in meds else 0,
    }

MED_FEATURES = {}
for sid in cohort_df["subject_id"]:
    row = COHORT_INDEXED.loc[sid]
    lo, hi = get_window(sid, row)
    MED_FEATURES[sid] = medication_features(sid, lo, hi)

med_features_df = pd.DataFrame(MED_FEATURES).T
med_features_df.index.name = "subject_id"
med_features_df.describe()



## 6.15 Aggregated time-series-derived static features (Part C §4 of the imbalance/imputation reference)

The second external review's most concretely useful suggestion: give the static branch
richer clinical summaries of the time series, rather than only demographics/ICD-10/
operation-history. Two benefits at once — it enriches what the DNN's static branch has to
work with, **and** it gives §8's grouped SMOTENC more informative features to interpolate
over. For each patient, within the same window as §6.2 (`get_window`):

- **Mean / min / max / std** of six clinically load-bearing signals: `creatinine`,
  `potassium`, `glucose`, `wbc`, `lactate`, `hb` (labs) and `hr`, `nibp_mbp`, `spo2`
  (ward/intra-op vitals) — 9 signals × 4 statistics = 36 features.
- **Severe-hypotension reading count** — number of mean-arterial-pressure readings below
  65 mmHg in the window. Stated precisely as a **reading count, not a duration**: INSPIRE's
  vitals are irregularly sampled, so "minutes spent hypotensive" would need an assumption
  about how long each reading represents, which this notebook avoids asserting.
- **Vasopressor administration count** and **high-alert medication administration count**
  — INSPIRE's `medications` table has **no dose field** (checked directly against the
  schema in §3), so the reviewed suggestion's "cumulative mcg dose" isn't computable as
  asked. This substitutes an **administration count** for a fixed drug-name/ATC keyword
  list — a real, honestly-weaker proxy (it can't distinguish one large dose from several
  small ones, or detect a true *escalation* in dose over time), stated here rather than
  silently implied to be the same thing.


In [ ]:

AGG_LABS = ["creatinine", "potassium", "glucose", "wbc", "lacate", "hb"]     # 'lacate' matches this dataset's item_name spelling (see §4.1)
AGG_WARD_VITALS = ["hr", "nibp_mbp", "spo2"]
SEVERE_HYPOTENSION_MAP_THRESHOLD = 65.0   # mmHg, a standard clinical cutoff for organ-perfusion risk

# Fixed keyword lists rather than a formal drug ontology lookup -- simple, auditable, and
# easy to extend; matched case-insensitively against drug_name (route/ATC not required).
VASOPRESSOR_KEYWORDS = ["norepinephrine", "noradrenaline", "epinephrine", "adrenaline",
                         "dopamine", "dobutamine", "vasopressin", "phenylephrine"]
HIGH_ALERT_KEYWORDS = VASOPRESSOR_KEYWORDS + ["fentanyl", "propofol", "midazolam", "heparin", "insulin"]

def _series_stats(chart_time2value, prefix):
    if not chart_time2value:
        return {f"{prefix}_mean": np.nan, f"{prefix}_min": np.nan, f"{prefix}_max": np.nan, f"{prefix}_std": np.nan}
    vals = np.array(list(chart_time2value.values()), dtype=float)
    return {
        f"{prefix}_mean": float(np.mean(vals)),
        f"{prefix}_min": float(np.min(vals)),
        f"{prefix}_max": float(np.max(vals)),
        f"{prefix}_std": float(np.std(vals)) if len(vals) > 1 else 0.0,
    }

def aggregate_features_for_patient(sid, lo, hi):
    feats = {}
    for lab in AGG_LABS:
        series = windowed_series(labs_df, sid, lab, lo, hi)
        feats.update(_series_stats(series, f"agg_{lab}"))
    for vital in AGG_WARD_VITALS:
        series = windowed_series(ward_vitals_df, sid, vital, lo, hi)
        feats.update(_series_stats(series, f"agg_{vital}"))
        if CONFIG["TIME_WINDOW"] == "peri_op" and vital in SYSTEM_INTRAOP_VITALS.get("cardiovascular", []) + SYSTEM_INTRAOP_VITALS.get("respiratory", []):
            iv_series = windowed_series(vitals_df, sid, vital, lo, hi)
            for t, v in iv_series.items():
                series.setdefault(t, v)

    map_series = windowed_series(ward_vitals_df, sid, "nibp_mbp", lo, hi)
    feats["severe_hypotension_reading_count"] = int(sum(1 for v in map_series.values() if v < SEVERE_HYPOTENSION_MAP_THRESHOLD))

    meds = medications_df[(medications_df["subject_id"] == sid) &
                           (medications_df["chart_time"] >= lo) & (medications_df["chart_time"] <= hi)]
    if len(meds) == 0:
        feats["vasopressor_administration_count"] = 0
        feats["high_alert_med_administration_count"] = 0
    else:
        # .astype(object) (not .astype(str)) deliberately -- on some pandas versions,
        # .astype(str) on an EMPTY series produces the newer pandas "str" extension dtype,
        # whose .sum() concatenates strings ('') instead of summing booleans numerically.
        # object dtype sidesteps that entirely and behaves the same either way.
        drug_names_lower = meds["drug_name"].astype(object).str.lower()
        feats["vasopressor_administration_count"] = int(drug_names_lower.apply(lambda n: any(k in n for k in VASOPRESSOR_KEYWORDS)).astype(bool).sum())
        feats["high_alert_med_administration_count"] = int(drug_names_lower.apply(lambda n: any(k in n for k in HIGH_ALERT_KEYWORDS)).astype(bool).sum())

    return feats

AGG_FEATURES = {}
for sid in cohort_df["subject_id"]:
    row = COHORT_INDEXED.loc[sid]
    lo, hi = get_window(sid, row)
    AGG_FEATURES[sid] = aggregate_features_for_patient(sid, lo, hi)

agg_features_df = pd.DataFrame(AGG_FEATURES).T
agg_features_df.index.name = "subject_id"
print(f"Aggregated features built: {agg_features_df.shape[1]} new static features "
      f"(36 mean/min/max/std + 3 count-based)")
print(f"Patients with >=1 severe-hypotension reading: "
      f"{(agg_features_df['severe_hypotension_reading_count'] > 0).sum()} / {len(agg_features_df)}")
print(f"Patients with >=1 vasopressor administration in-window: "
      f"{(agg_features_df['vasopressor_administration_count'] > 0).sum()} / {len(agg_features_df)}")
agg_features_df.describe().T[["mean", "std", "min", "max"]].head(10)



## 6.14 Assembling the final per-patient feature bundle

Pulls everything from §6.1–§6.13 into one `PATIENT_BUNDLE` dict keyed by `subject_id`,
containing:

- `ts[system]` → `{"raw": [T,F] with NaN gaps, "mask": [T,F], "feature_names": [...]}`
  for each of the six time-series systems (renal/cardiovascular/respiratory/
  metabolic_hepatic/haematology/neurological) — still **unimputed**, exactly as built in
  §6.3, ready for §7.
- `static` → one flat vector: demographics/ASA/department (§6.12), GI/MSK/HFRS (§6.6–6.8),
  operation-count features (§6.10), the cardiac-recovery exception flag (§6.11), the
  cardiovascular summary + renal-cardiac interaction (§6.9), and medication aggregates
  (§6.13).
- `label` → the 30-day mortality target from §4.2.

This is the single object every later section (§7 imputation, §8 split/sampling, §9 model)
reads from.


In [ ]:

def build_static_vector(sid):
    parts = {}
    parts.update(static_df.loc[sid].to_dict())
    parts.update(icd10_features_df.loc[sid].to_dict())
    parts.update(infection_features_df.loc[sid].to_dict())   # §6.6, roadmap §4.1a -- new this revision
    parts.update(op_count_df.loc[sid].to_dict())
    parts["cardiac_recovery_exception"] = CARDIAC_EXCEPTION_FLAG[sid]
    parts.update(cardiac_summary_df.loc[sid].to_dict())
    parts["renal_cardiac_interaction"] = RENAL_CARDIAC_INTERACTION[sid]
    parts.update(med_features_df.loc[sid].to_dict())
    parts.update(AGG_FEATURES[sid])   # §6.15, Part C §4 of the imbalance/imputation reference
    return parts

PATIENT_BUNDLE = {}
for sid in cohort_df["subject_id"]:
    ts = {system: RAW_SYSTEM_TENSORS[(sid, system)] for system in TIME_SERIES_SYSTEMS}
    PATIENT_BUNDLE[sid] = {
        "ts": ts,
        "static": build_static_vector(sid),
        "label": float(COHORT_INDEXED.loc[sid, "died_30day_from_last_op"]),
    }

STATIC_FEATURE_NAMES = sorted(next(iter(PATIENT_BUNDLE.values()))["static"].keys())
print(f"Bundle built for {len(PATIENT_BUNDLE)} patients.")
print(f"Static feature vector length: {len(STATIC_FEATURE_NAMES)}")
print(f"Static features: {STATIC_FEATURE_NAMES}")
for system in TIME_SERIES_SYSTEMS:
    fnames = PATIENT_BUNDLE[cohort_df['subject_id'].iloc[0]]['ts'][system]['feature_names']
    print(f"  {system:18s} time-series features ({len(fnames)}): {fnames}")


In [ ]:

# Free the raw long-format tables now that PATIENT_BUNDLE holds everything §7-§12 need.
# Nothing downstream of this cell reads labs_df/vitals_df/ward_vitals_df/medications_df/
# diagnoses_df or the per-table lookup indices built in §6.2 -- at full scale these are
# the single largest objects in memory, and Python has no reason to keep them alive once
# PATIENT_BUNDLE is built. Set CONFIG-less: this is safe to always run, since every read of
# these tables already happened above.
import gc

_freed_mb = sum(df.memory_usage(deep=True).sum() for df in
                 [labs_df, vitals_df, ward_vitals_df, medications_df, diagnoses_df]) / 1e6
del labs_df, vitals_df, ward_vitals_df, medications_df, diagnoses_df
_GROUPED_ARRAYS_CACHE.clear()
RAW_SYSTEM_TENSORS.clear()   # PATIENT_BUNDLE['ts'][system] already holds these dicts directly -- see §6.3's note
gc.collect()
print(f"Freed the raw long-format tables (~{_freed_mb:.1f} MB on this dev subset; the real "
      f"saving at full scale will be much larger). If a later cell needs to inspect raw "
      f"labs/vitals directly (e.g. for ad-hoc debugging), re-run Part 3 to reload them.")


In [ ]:

# Quick label sanity check on the assembled bundle before moving to §7.
labels = np.array([PATIENT_BUNDLE[sid]["label"] for sid in cohort_df["subject_id"]])
print(f"Positive rate in bundle: {labels.mean():.1%}  ({int(labels.sum())} / {len(labels)})")
print("This dev-subset rate (~23%) is NOT representative of the full-cohort deployment rate "
      "(~0.5-0.9%, per §1.4) -- keep this in mind when reading any metric later in the "
      "notebook as a sanity check, not a final result.")



# Part 7 — Missing data: implementing the §1.3 methods

Every method from §1.3.1's table is implemented here as a function with the same
signature, switched on by `CONFIG['IMPUTATION_STRATEGY']`. §7.5 runs a **side-by-side
comparison** of all four on the same data so you can see concretely what each one does to
the same gaps — this is the "experiment on it yourself" piece you asked for, made runnable
rather than theoretical.

Two things apply regardless of which strategy is chosen, per §1.3:

1. **The mask (§6.3) is never touched.** Whatever value-fill method runs, the mask feature
   stays exactly as observed — it's the model's only way to recover MNAR signal.
2. **Statistics (medians, KNN neighbours, MICE models) are always fit on the training split
   only** (computed in §8, referenced here) — fitting on the full dataset before splitting
   would leak test-set information into imputed training values, a common and serious
   mistake in exactly this kind of pipeline.


## 7.1 Whole-feature-missing fallback (applies before any of the strategies below)

In [ ]:

def population_stats(train_ids):
    # Per (system, feature) mean/median across the TRAINING split only -- used whenever a
    # patient has zero observations of a feature anywhere in their window (§1.3.2's first
    # branch: nothing else is possible in that case).
    stats = {}
    for system in TIME_SERIES_SYSTEMS:
        fnames = PATIENT_BUNDLE[train_ids[0]]["ts"][system]["feature_names"]
        if not fnames:
            continue
        all_vals = []
        for sid in train_ids:
            raw = PATIENT_BUNDLE[sid]["ts"][system]["raw"]
            all_vals.append(raw)
        stacked = np.concatenate(all_vals, axis=0) if all_vals else np.zeros((0, len(fnames)))
        for j, fname in enumerate(fnames):
            col = stacked[:, j]
            col = col[~np.isnan(col)]
            stats[(system, fname)] = {
                "mean": float(np.mean(col)) if len(col) else 0.0,
                "median": float(np.median(col)) if len(col) else 0.0,
            }
    return stats


## 7.2 The four imputation strategies

In [ ]:

def impute_median(raw, feature_names, system, stats):
    # §1.3.1 row 1/2: fill every NaN with the training-set median for that feature.
    out = raw.copy()
    for j, fname in enumerate(feature_names):
        med = stats[(system, fname)]["median"]
        col = out[:, j]
        col[np.isnan(col)] = med
    return out

def _fade_to_mean_interpolate(times_observed, values_observed, grid, mean_val, decay=8.0):
    # The source repo's smooth_fade_to_mean_interpolator idea (§1.3.1 row 3), reimplemented
    # standalone: linear interpolation between observed points, fading toward the population
    # mean the further a grid point is from any observation.
    if len(times_observed) == 0:
        return np.full(len(grid), mean_val)
    if len(times_observed) == 1:
        # can't interpolate a slope from one point -- fall back to nearest + fade
        t0, v0 = times_observed[0], values_observed[0]
        out = np.array([v0 * math.exp(-abs(g - t0) / decay) + mean_val * (1 - math.exp(-abs(g - t0) / decay)) for g in grid])
        return out
    interp = np.interp(grid, times_observed, values_observed)
    t_min, t_max = times_observed.min(), times_observed.max()
    out = interp.copy()
    for i, g in enumerate(grid):
        if g < t_min:
            w = math.exp(-abs(t_min - g) / decay)
            out[i] = w * interp[i] + (1 - w) * mean_val
        elif g > t_max:
            w = math.exp(-abs(g - t_max) / decay)
            out[i] = w * interp[i] + (1 - w) * mean_val
    return out

def impute_interpolate(raw, feature_names, system, stats, grid):
    # §1.3.1 row 3: within-patient interpolation, fading to the population mean outside
    # the observed range -- the default for fast-changing vitals per §1.3.2's decision tree.
    out = raw.copy()
    n_t = raw.shape[0]
    for j, fname in enumerate(feature_names):
        col = out[:, j]
        obs_idx = ~np.isnan(col)
        mean_val = stats[(system, fname)]["mean"]
        if obs_idx.sum() == 0:
            out[:, j] = mean_val
        else:
            out[:, j] = _fade_to_mean_interpolate(grid[obs_idx], col[obs_idx], grid, mean_val)
    return out

def impute_forward_fill(raw, feature_names, system, stats):
    # §1.3.1 row 4 (LOCF): repeat the last observed value forward; before any observation,
    # fall back to the training-set median (there's nothing to carry forward from yet).
    out = raw.copy()
    n_t = raw.shape[0]
    for j in range(out.shape[1]):
        last_val = None
        med = stats[(system, feature_names[j])]["median"]
        for t in range(n_t):
            if np.isnan(out[t, j]):
                out[t, j] = last_val if last_val is not None else med
            else:
                last_val = out[t, j]
    return out

def impute_knn(raw_matrix_all_patients, k):
    # §1.3.1 row 5: KNN across patients, one call per (system) on a [n_patients, T*F]
    # flattened matrix -- used at the whole-cohort level in §7.3, not per-patient like the
    # other three, since KNN needs other patients to find neighbours from.
    imputer = KNNImputer(n_neighbors=min(k, max(raw_matrix_all_patients.shape[0] - 1, 1)))
    return imputer.fit_transform(raw_matrix_all_patients)



## 7.3 The default decision-tree strategy (§1.3.2), and static-feature imputation

Fast-changing intra-op/ward vitals (`hr`, `nibp_*`, `spo2`, `rr`, `fio2`, `bt`, ...) use
interpolate-and-fade; slower-changing labs use forward-fill; anything with zero
observations for a patient falls back to the training median — exactly the tree in §1.3.2.
Which item names count as "fast-changing vitals" vs. "slow-changing labs" is defined
explicitly below (editable) rather than inferred, so the classification is a visible,
checkable decision.


In [ ]:

FAST_CHANGING_ITEMS = {
    "hr", "nibp_sbp", "nibp_dbp", "nibp_mbp", "spo2", "rr", "fio2", "bt", "uo",
    "art_sbp", "art_dbp", "art_mbp", "etco2", "peep", "pip", "pplat", "gcs_e", "gcs_m", "gcs_v",
}
# Everything else in the routing map (labs like creatinine, albumin, wbc, ...) is treated
# as slow-changing -> forward-fill, per §1.3.2.

def impute_patient_system(sid, system, strategy, stats):
    bundle = PATIENT_BUNDLE[sid]["ts"][system]
    raw, fnames = bundle["raw"], bundle["feature_names"]
    if raw.shape[1] == 0:
        return raw.copy()
    row = COHORT_INDEXED.loc[sid]
    lo, hi = get_window(sid, row)
    grid = np.linspace(lo, hi, CONFIG["TARGET_SEQ_LEN"])

    if strategy == "median":
        return impute_median(raw, fnames, system, stats)
    elif strategy == "interpolate":
        return impute_interpolate(raw, fnames, system, stats, grid)
    elif strategy == "decision_tree":
        out = raw.copy()
        fast_idx = [j for j, f in enumerate(fnames) if f in FAST_CHANGING_ITEMS]
        slow_idx = [j for j, f in enumerate(fnames) if f not in FAST_CHANGING_ITEMS]
        if fast_idx:
            fast_names = [fnames[j] for j in fast_idx]
            out[:, fast_idx] = impute_interpolate(raw[:, fast_idx], fast_names, system, stats, grid)
        if slow_idx:
            slow_names = [fnames[j] for j in slow_idx]
            out[:, slow_idx] = impute_forward_fill(raw[:, slow_idx], slow_names, system, stats)
        return out
    else:
        raise ValueError(f"strategy {strategy!r} handled elsewhere (knn) or unknown")

def impute_static_vector(sid, train_static_df):
    # Static/tabular features (§6.14): median-fill per column, fit on the training split
    # only (passed in) -- same leakage discipline as the time-series imputers above.
    vec = pd.Series(PATIENT_BUNDLE[sid]["static"])
    filled = vec.fillna(train_static_df.median(numeric_only=True))
    return filled



## 7.4 Running the chosen strategy end-to-end (train-fit, applied to everyone)

`CONFIG['IMPUTATION_STRATEGY']` decides which of the four runs here. Statistics are always
computed from the **training split** (§8.1, run once below to get train IDs before this
cell, then re-used identically in §8) to avoid leakage, as promised in the Part 7 header.
KNN is handled specially (§7.2) since it needs the whole training matrix, not one patient
at a time.


In [ ]:

# A first, lightweight split done here (before §8's full split/sampling section) purely to
# get TRAIN_IDS for leakage-safe statistics -- §8 re-derives and USES this same split for
# training; nothing here duplicates or conflicts with it.
ALL_IDS = cohort_df["subject_id"].tolist()
ALL_LABELS = np.array([PATIENT_BUNDLE[sid]["label"] for sid in ALL_IDS])
TRAIN_IDS_FOR_STATS, _HOLD_IDS = train_test_split(
    ALL_IDS, test_size=(CONFIG["VAL_FRACTION"] + CONFIG["TEST_FRACTION"]),
    stratify=ALL_LABELS, random_state=SEED
)

STATS = population_stats(TRAIN_IDS_FOR_STATS)
train_static_df = pd.DataFrame({sid: PATIENT_BUNDLE[sid]["static"] for sid in TRAIN_IDS_FOR_STATS}).T

IMPUTED_BUNDLE = {}
strategy = CONFIG["IMPUTATION_STRATEGY"]

if strategy == "knn":
    # Build one big matrix per system: [n_patients, T*F], impute across patients, reshape back.
    for system in TIME_SERIES_SYSTEMS:
        fnames = PATIENT_BUNDLE[ALL_IDS[0]]["ts"][system]["feature_names"]
        if not fnames:
            continue
        mat = np.stack([PATIENT_BUNDLE[sid]["ts"][system]["raw"].reshape(-1) for sid in ALL_IDS])
        mat_imputed = impute_knn(mat, CONFIG["KNN_NEIGHBORS"])
        for i, sid in enumerate(ALL_IDS):
            IMPUTED_BUNDLE.setdefault(sid, {})[system] = mat_imputed[i].reshape(CONFIG["TARGET_SEQ_LEN"], len(fnames))
else:
    for sid in ALL_IDS:
        IMPUTED_BUNDLE[sid] = {}
        for system in TIME_SERIES_SYSTEMS:
            IMPUTED_BUNDLE[sid][system] = impute_patient_system(sid, system, strategy, STATS)

for sid in ALL_IDS:
    IMPUTED_BUNDLE[sid]["static"] = impute_static_vector(sid, train_static_df)

print(f"Imputation strategy used: {strategy!r}")
remaining_nans = sum(np.isnan(IMPUTED_BUNDLE[sid][system]).sum()
                      for sid in ALL_IDS for system in TIME_SERIES_SYSTEMS
                      if IMPUTED_BUNDLE[sid][system].size)
print(f"Remaining NaNs after imputation (should be 0): {remaining_nans}")


## 7.5 Side-by-side comparison of all four strategies on one real gap

In [ ]:

# Pick one patient/system/feature with a genuine, visible gap and show what each strategy
# does to it -- exactly the kind of comparison you said you want to be able to run yourself.
example_sid = ALL_IDS[0]
example_system = "renal"
example_feature_idx = 0
row = COHORT_INDEXED.loc[example_sid]
lo, hi = get_window(example_sid, row)
grid = np.linspace(lo, hi, CONFIG["TARGET_SEQ_LEN"])
raw = PATIENT_BUNDLE[example_sid]["ts"][example_system]["raw"]
fnames = PATIENT_BUNDLE[example_sid]["ts"][example_system]["feature_names"]

fig, ax = plt.subplots(figsize=(9, 4.5))
strategies_to_show = ["median", "interpolate", "decision_tree"]
colors = {"median": "#e67e22", "interpolate": "#2980b9", "decision_tree": "#27ae60"}
for strat in strategies_to_show:
    filled = impute_patient_system(example_sid, example_system, strat, STATS)
    ax.plot(grid, filled[:, example_feature_idx], label=strat, color=colors[strat], alpha=0.8)
obs_mask = ~np.isnan(raw[:, example_feature_idx])
ax.scatter(grid[obs_mask], raw[obs_mask, example_feature_idx], color="black", zorder=5, label="observed", s=40)
ax.set_title(f"Imputation comparison: patient {example_sid}, {example_system}/{fnames[example_feature_idx]}")
ax.set_xlabel("chart_time (minutes)"); ax.set_ylabel("value"); ax.legend()
plt.tight_layout(); plt.show()
print("This is exactly the comparison to re-run (swap example_sid/example_system/example_feature_idx) "
      "as you experiment with §1.3's strategies on your own features of interest.")



## 7.6 Standardisation (z-score, fit on training data only)

One deliberately non-optional step, separate from the §1.3 imputation choice: raw clinical
values live on wildly different scales (`glucose` in the hundreds, `potassium` around 4,
`wbc` in the thousands per µL) — feeding that directly into a neural network makes large-
magnitude features dominate the loss for reasons that have nothing to do with clinical
importance, and destabilises the autoencoder pre-training in §10.1 in particular (its loss
is a raw MSE, which is scale-sensitive in exactly this way). Mean/std are computed **once,
from `TRAIN_IDS_FOR_STATS` only** (§7.4's split, reused here — never refit on val/test, for
the same leakage reason given throughout Part 7), then applied to every patient.


In [ ]:

def compute_train_std(train_ids):
    std_stats = {}
    for system in TIME_SERIES_SYSTEMS:
        fnames = PATIENT_BUNDLE[train_ids[0]]["ts"][system]["feature_names"]
        if not fnames:
            continue
        stacked = np.stack([IMPUTED_BUNDLE[sid][system] for sid in train_ids])   # [N, T, F]
        for j, fname in enumerate(fnames):
            col = stacked[:, :, j].reshape(-1)
            std_stats[(system, fname)] = float(col.std() + 1e-6)
    return std_stats

TRAIN_STD = compute_train_std(TRAIN_IDS_FOR_STATS)

def standardize_system_tensor(raw, feature_names, system):
    out = raw.copy()
    for j, fname in enumerate(feature_names):
        mean = STATS[(system, fname)]["mean"]
        std = TRAIN_STD[(system, fname)]
        out[:, j] = (out[:, j] - mean) / std
    return out

for sid in ALL_IDS:
    for system in TIME_SERIES_SYSTEMS:
        fnames = PATIENT_BUNDLE[sid]["ts"][system]["feature_names"]
        if fnames:
            IMPUTED_BUNDLE[sid][system] = standardize_system_tensor(IMPUTED_BUNDLE[sid][system], fnames, system)

# Static features: same idea, via sklearn's StandardScaler fit on the training static table.
static_scaler = StandardScaler()
static_scaler.fit(train_static_df.fillna(train_static_df.median(numeric_only=True)).values)

for sid in ALL_IDS:
    vec = IMPUTED_BUNDLE[sid]["static"]
    scaled = static_scaler.transform(vec.values.reshape(1, -1))[0]
    IMPUTED_BUNDLE[sid]["static"] = pd.Series(scaled, index=vec.index)

print("Standardisation applied (train-fit z-score) to all time-series and static features.")
example_sid = ALL_IDS[0]
print(f"Example, patient {example_sid}, renal system, post-standardisation value range: "
      f"[{IMPUTED_BUNDLE[example_sid]['renal'].min():.2f}, {IMPUTED_BUNDLE[example_sid]['renal'].max():.2f}] "
      f"(should now be roughly within a few units of 0, not raw clinical units)")



# Part 8 -- Train/val/test split and class-imbalance sampling

This section implements the combined pipeline agreed on after reviewing external advice
against this notebook's actual multi-branch architecture (full writeup:
`Data_Imbalance_and_Imputation_Reference.md`, Part C). In order:

- **8.1** stratified split (non-negotiable, unchanged from the original design)
- **8.2** the new default strategy -- **grouped SMOTENC + Tomek-link cleanup**, applied
  only to the static branch, only within clinical strata (department x ASA), targeting a
  1:10 positive:negative ratio rather than full 1:1 balance
- **8.3** sequence-branch augmentation (jitter + time-masking) for real minority training
  patients -- the sequence branch never receives SMOTE-style synthetic data (Part C's own
  reasoning: interpolating between two raw physiological trajectories in different
  patients' time coordinates isn't a well-defined operation)
- **8.4** the older, simpler strategies (plain SMOTE/ADASYN, random over/undersample,
  class-weight-only) are kept available via `CONFIG['SAMPLING_STRATEGY']` for direct
  comparison, but are no longer the default



## 8.1 Stratified split

Re-derives the same split as Part 7's `TRAIN_IDS_FOR_STATS` / `_HOLD_IDS` (same `SEED`,
same call) and further splits the holdout into validation and test. With only 30 patients
in this dev subset, don't over-read exact percentages here -- the code is written to hold
correctly once the full cohort is loaded, where `VAL_FRACTION`/`TEST_FRACTION` will
produce far more stable folds.


In [ ]:

TRAIN_IDS = TRAIN_IDS_FOR_STATS   # identical split object reused from Part 7, not recomputed
HOLD_LABELS = np.array([PATIENT_BUNDLE[sid]["label"] for sid in _HOLD_IDS])
val_frac_of_hold = CONFIG["VAL_FRACTION"] / (CONFIG["VAL_FRACTION"] + CONFIG["TEST_FRACTION"])

VAL_IDS, TEST_IDS = train_test_split(
    _HOLD_IDS, test_size=(1 - val_frac_of_hold), stratify=HOLD_LABELS, random_state=SEED
)

for name, ids in [("train", TRAIN_IDS), ("val", VAL_IDS), ("test", TEST_IDS)]:
    labels = [PATIENT_BUNDLE[sid]["label"] for sid in ids]
    print(f"{name:5s}: n={len(ids):3d}  positives={int(sum(labels)):2d}  rate={np.mean(labels):.1%}")



## 8.1b Optional: department-stratified downsample of TRAINING survived patients only

**Off by default** (`CONFIG['DOWNSAMPLE_TRAIN_SURVIVED_TO'] = None`). This is a
memory/compute lever, not a statistical recommendation — see Part 3's "note on memory
strategy" for the full reasoning on why this should be the last resort, not the first
move, and why it only ever touches training-split `survived` patients:

- **Never touches `died` patients** — they're already the scarce class; downsampling them
  further would be actively counterproductive.
- **Never touches `VAL_IDS`/`TEST_IDS`** — validation and test must keep the true
  prevalence for AUPRC/calibration to mean anything (Part C step 9).
- **Stratified by department** (not "organ-system ICD" — see Part 3's note on why) so the
  downsampled training pool's department mix still resembles the real cohort's, rather
  than randomly dropping an entire department's worth of survived patients by chance.


In [ ]:

def downsample_train_survived(train_ids, target_n_survived, strata_col="department", seed=SEED):
    labels_by_id = {sid: PATIENT_BUNDLE[sid]["label"] for sid in train_ids}
    survived_ids = [sid for sid in train_ids if labels_by_id[sid] == 0.0]
    died_ids = [sid for sid in train_ids if labels_by_id[sid] == 1.0]

    if target_n_survived >= len(survived_ids):
        print(f"DOWNSAMPLE_TRAIN_SURVIVED_TO={target_n_survived} >= current {len(survived_ids)} "
              f"survived training patients -- nothing to do.")
        return train_ids

    strata = COHORT_INDEXED.loc[survived_ids, strata_col].fillna("missing").astype(str)
    rng = np.random.default_rng(seed)
    kept_survived_ids = []
    # Proportional allocation per stratum, rounded, with a final random top-up/trim to hit
    # the exact target -- keeps the department mix close to the original without needing
    # every stratum to divide evenly.
    frac = target_n_survived / len(survived_ids)
    for stratum_key, group in strata.groupby(strata):
        ids_in_stratum = group.index.tolist()
        n_keep = max(1, round(len(ids_in_stratum) * frac))
        n_keep = min(n_keep, len(ids_in_stratum))
        kept_survived_ids.extend(rng.choice(ids_in_stratum, size=n_keep, replace=False).tolist())

    if len(kept_survived_ids) > target_n_survived:
        kept_survived_ids = rng.choice(kept_survived_ids, size=target_n_survived, replace=False).tolist()

    print(f"Downsampled training 'survived' patients: {len(survived_ids)} -> {len(kept_survived_ids)} "
          f"(stratified by {strata_col!r}); training 'died' patients unchanged at {len(died_ids)}.")
    return died_ids + kept_survived_ids

if CONFIG["DOWNSAMPLE_TRAIN_SURVIVED_TO"] is not None:
    TRAIN_IDS = downsample_train_survived(TRAIN_IDS, CONFIG["DOWNSAMPLE_TRAIN_SURVIVED_TO"])
    labels = [PATIENT_BUNDLE[sid]["label"] for sid in TRAIN_IDS]
    print(f"train (post-downsample): n={len(TRAIN_IDS)}  positives={int(sum(labels))}  rate={np.mean(labels):.1%}")
    print("NOTE: Part 7's imputation/standardization statistics (STATS, TRAIN_STD) were "
          "already fit on the FULL pre-downsample training pool (TRAIN_IDS_FOR_STATS) -- "
          "intentional, not stale: more patients gives more robust statistics, and only the "
          "set of patients actually used for gradient updates needs to shrink here.")
else:
    print("DOWNSAMPLE_TRAIN_SURVIVED_TO is None -- no downsampling applied (recommended default).")



## 8.2 Grouped SMOTENC + Tomek-link cleanup (new default)

**Why grouped:** plain SMOTE/SMOTENC would happily blend a cardiothoracic ASA-5 patient
with an outpatient orthopedic ASA-1 patient purely because both happened to die -- that's
not a clinically meaningful neighborhood. Splitting the minority cohort into strata by
`(department, asa)` and running SMOTENC **within** each stratum keeps every synthetic
patient's interpolation partners clinically comparable to each other.

**Why SMOTENC, not plain SMOTE:** the static vector mixes continuous features (age, the
36 aggregated lab/vital statistics from Section 6.15) with one-hot department columns and
binary flags (`sex_F`, `cardiac_recovery_exception`, the GI/MSK ICD-10 flags, ...). Plain
SMOTE would linearly interpolate those binary/one-hot columns into meaningless fractional
values (e.g. `dept_GS = 0.6`). SMOTENC is told exactly which columns are categorical and
handles them by majority vote among neighbors instead of interpolation.

**Why a 1:10 target, not full 1:1 balance:** at the full cohort's ~469 real deaths,
synthesizing up to 46,900+ to match ~99,417 survivors would force >99x amplification of
the same ~469 real points -- pure noise-filling well past where SMOTE's own literature
says new information stops being added. 1:10 (~4,700 synthetic positives against the full
survivor pool) is the conservative end of the two external reviews' suggested 1:10-1:4
range.

**The Tomek-link cleanup step, and how synthetic vs. real rows are tracked correctly:**
after SMOTENC runs per stratum, a global Tomek-link pass identifies and removes majority
(survived) points that are each a synthetic minority point's nearest opposite-class
neighbor -- this widens the decision margin around the newly-added synthetic points. Row
identity through this step is tracked via each sampler's own `sample_indices_` attribute
(the array of retained input-row positions that `imbalanced-learn` documents and
guarantees), **not** by assuming synthetic rows land at a particular position in the
output array -- the earlier draft of this pipeline (reviewed in
`Data_Imbalance_and_Imputation_Reference.md` Part A.1) relied on exactly that unstable
assumption; this version doesn't.


In [ ]:

from imblearn.over_sampling import SMOTE, ADASYN, SMOTENC, RandomOverSampler
from imblearn.under_sampling import RandomUnderSampler, TomekLinks

def compute_pos_weight_from_counts(n_pos, n_neg):
    return float(n_neg / max(n_pos, 1))

def compute_pos_weight(train_ids):
    labels = np.array([PATIENT_BUNDLE[sid]["label"] for sid in train_ids])
    return compute_pos_weight_from_counts(labels.sum(), len(labels) - labels.sum())

# Which static columns are categorical -> passed to SMOTENC as a boolean mask, in the
# same column order as STATIC_FEATURE_NAMES (built in Section 6.14).
_BINARY_STATIC_COLS = {
    "sex_F", "emop", "gi_icd10_flag", "gi_department_flag", "msk_icd10_flag",
    "msk_department_flag", "cardiac_recovery_exception", "cardio_has_iabp", "asa",
    "infection_chapter_i_flag", "infection_high_risk_code_flag", "infection_fever_flag",
    "infection_wbc_abnormal_flag", "infection_crp_elevated_flag",
}
CATEGORICAL_STATIC_MASK = np.array([
    name.startswith("dept_") or name in _BINARY_STATIC_COLS
    for name in STATIC_FEATURE_NAMES
])
print(f"Static features flagged categorical for SMOTENC: {int(CATEGORICAL_STATIC_MASK.sum())} "
      f"/ {len(STATIC_FEATURE_NAMES)} ({[n for n, c in zip(STATIC_FEATURE_NAMES, CATEGORICAL_STATIC_MASK) if c]})")

def _static_matrix(ids):
    return np.stack([IMPUTED_BUNDLE[sid]["static"].values.astype(float) for sid in ids])

def grouped_smotenc(train_ids, target_ratio, min_stratum_minority, strata_cols):
    labels_by_id = {sid: PATIENT_BUNDLE[sid]["label"] for sid in train_ids}
    # NOTE: .astype(str) on a float column with NaN leaves the NaN as an actual float NaN
    # on modern pandas (its "str" dtype is still nullable) rather than the string "nan" --
    # .fillna() first, per column, avoids that trap before building the join key.
    strata_df = COHORT_INDEXED.loc[train_ids, strata_cols].copy()
    for col in strata_cols:
        strata_df[col] = strata_df[col].fillna("missing").astype(str)
    strata = strata_df.agg("_".join, axis=1)

    all_synthetic_rows = []   # list of np.ndarray, one per synthetic patient
    n_strata_synthesized, n_strata_skipped = 0, 0

    for stratum_key, group in strata.groupby(strata):
        stratum_ids = group.index.tolist()
        pos_ids = [sid for sid in stratum_ids if labels_by_id[sid] == 1.0]
        neg_ids = [sid for sid in stratum_ids if labels_by_id[sid] == 0.0]
        n_pos, n_neg = len(pos_ids), len(neg_ids)

        if n_pos < min_stratum_minority or n_neg == 0:
            n_strata_skipped += 1
            continue
        desired_pos = int(n_neg * target_ratio)
        if desired_pos <= n_pos:
            n_strata_skipped += 1
            continue

        X_stratum = _static_matrix(pos_ids + neg_ids)
        y_stratum = np.array([1.0] * n_pos + [0.0] * n_neg)
        k_neighbors = max(1, min(5, n_pos - 1))
        try:
            smotenc = SMOTENC(categorical_features=CATEGORICAL_STATIC_MASK,
                               sampling_strategy={1: desired_pos},
                               k_neighbors=k_neighbors, random_state=SEED)
            X_res, y_res = smotenc.fit_resample(X_stratum, y_stratum)
        except ValueError as e:
            print(f"  stratum {stratum_key!r}: SMOTENC failed ({e}) -- skipped")
            n_strata_skipped += 1
            continue

        n_synthetic_this_stratum = len(X_res) - len(X_stratum)
        # SMOTENC/SMOTE append generated samples after the originals, in the fixed order
        # the input was given (X_stratum = pos_ids + neg_ids) -- documented behaviour, not
        # an assumption about arbitrary internal ordering. Sanity-checked below regardless.
        synthetic_rows_this_stratum = X_res[len(X_stratum):]
        synthetic_labels_this_stratum = y_res[len(X_stratum):]
        assert np.all(synthetic_labels_this_stratum == 1.0), (
            "Expected all synthesized rows to be minority-class -- ordering assumption violated, "
            "do not trust this stratum's synthetic rows."
        )
        all_synthetic_rows.extend(list(synthetic_rows_this_stratum))
        n_strata_synthesized += 1

    print(f"Grouped SMOTENC: {n_strata_synthesized} strata synthesized, {n_strata_skipped} skipped "
          f"(too few real positives, or already at/above target ratio)")
    return all_synthetic_rows

def tomek_cleanup(train_ids, synthetic_rows):
    real_X = _static_matrix(train_ids)
    real_y = np.array([PATIENT_BUNDLE[sid]["label"] for sid in train_ids])
    synth_X = np.stack(synthetic_rows) if synthetic_rows else np.zeros((0, real_X.shape[1]))
    synth_y = np.ones(len(synthetic_rows))

    combined_X = np.concatenate([real_X, synth_X], axis=0)
    combined_y = np.concatenate([real_y, synth_y], axis=0)
    combined_keys = list(train_ids) + [f"synthetic_{i}" for i in range(len(synthetic_rows))]

    if len(synthetic_rows) == 0:
        return train_ids, []   # nothing to clean up around

    tl = TomekLinks(sampling_strategy="majority")
    X_clean, y_clean = tl.fit_resample(combined_X, combined_y)
    try:
        kept_positions = tl.sample_indices_
    except AttributeError:
        print("WARNING: TomekLinks.sample_indices_ unavailable in this imblearn version -- "
              "falling back to keeping everything (no Tomek cleanup applied this run).")
        kept_positions = np.arange(len(combined_keys))

    kept_keys = [combined_keys[i] for i in kept_positions]
    kept_real_ids = [k for k in kept_keys if not str(k).startswith("synthetic_")]
    kept_synthetic_positions = [int(k.split("_")[1]) for k in kept_keys if str(k).startswith("synthetic_")]
    kept_synthetic_rows = [synthetic_rows[i] for i in kept_synthetic_positions]

    n_removed_real = len(train_ids) - len(kept_real_ids)
    n_removed_synthetic = len(synthetic_rows) - len(kept_synthetic_rows)
    print(f"Tomek cleanup: removed {n_removed_real} real (majority) patients, "
          f"{n_removed_synthetic} synthetic patients as ambiguous boundary pairs")
    return kept_real_ids, kept_synthetic_rows


In [ ]:

def apply_sampling_strategy(train_ids, strategy):
    # Returns (effective_train_ids, pos_weight, synthetic_static_rows_or_None).
    pos_weight = compute_pos_weight(train_ids)

    if strategy in ("class_weight", "none"):
        return train_ids, pos_weight, None

    if strategy == "focal_loss":
        return train_ids, pos_weight, None   # focal loss handled in the loss function itself, Part 9.5

    labels = np.array([PATIENT_BUNDLE[sid]["label"] for sid in train_ids])

    if strategy == "grouped_smotenc_tomek":
        synthetic_rows = grouped_smotenc(train_ids, CONFIG["SMOTE_TARGET_RATIO"],
                                          CONFIG["SMOTE_MIN_STRATUM_MINORITY"], CONFIG["SMOTE_STRATA_COLS"])
        if not synthetic_rows:
            print("No strata met the synthesis criteria on this sample -- falling back to class_weight only. "
                  "Expected on a very small dev subset; re-run at full scale.")
            return train_ids, pos_weight, None
        kept_real_ids, kept_synthetic_rows = tomek_cleanup(train_ids, synthetic_rows)
        synthetic_pairs = [(row, 1.0) for row in kept_synthetic_rows]
        n_pos_final = int(labels.sum() - (len(train_ids) - len(kept_real_ids))) + len(kept_synthetic_rows)
        # NOTE: the subtraction above assumes Tomek only ever removes MAJORITY real patients
        # (sampling_strategy="majority" enforces this) -- positive real patients are never
        # removed by this step, only negative ones and ambiguous synthetics.
        n_neg_final = len(kept_real_ids) - (int(labels.sum()))
        final_pos_weight = compute_pos_weight_from_counts(n_pos_final, max(n_neg_final, 1))
        print(f"Final effective training set: {len(kept_real_ids)} real + {len(kept_synthetic_rows)} synthetic "
              f"patients. Residual pos_weight after sampling: {final_pos_weight:.2f} "
              f"(sampling narrowed the gap; loss-weighting finishes it, per Part C step 8)")
        return kept_real_ids, final_pos_weight, synthetic_pairs

    ids_arr = np.array(train_ids).reshape(-1, 1)   # sklearn resamplers need a 2D "X"

    if strategy == "random_oversample":
        res_ids, res_labels = RandomOverSampler(random_state=SEED).fit_resample(ids_arr, labels)
        return res_ids.ravel().tolist(), 1.0, None

    if strategy == "random_undersample":
        res_ids, res_labels = RandomUnderSampler(random_state=SEED).fit_resample(ids_arr, labels)
        return res_ids.ravel().tolist(), 1.0, None

    if strategy in ("smote", "adasyn"):
        X = _static_matrix(train_ids)
        y = labels
        n_minority = int(y.sum())
        if n_minority < 2:
            print(f"WARNING: only {n_minority} positive training examples -- SMOTE/ADASYN need "
                  f">=2 to find neighbours; falling back to class_weight for this run.")
            return train_ids, pos_weight, None
        k_neighbors = max(1, min(5, n_minority - 1))
        Sampler = SMOTE if strategy == "smote" else ADASYN
        try:
            X_res, y_res = Sampler(random_state=SEED, k_neighbors=k_neighbors).fit_resample(X, y)
        except ValueError as e:
            print(f"WARNING: {strategy} failed ({e}); falling back to class_weight.")
            return train_ids, pos_weight, None
        n_synthetic = len(X_res) - len(X)
        synthetic_rows = [(X_res[len(X) + i], float(y_res[len(X) + i])) for i in range(n_synthetic)]
        print(f"{strategy} (unconstrained, no clinical-neighborhood grouping): generated {n_synthetic} "
              f"synthetic minority rows -- prefer 'grouped_smotenc_tomek' unless comparing directly.")
        return train_ids, 1.0, synthetic_rows

    raise ValueError(f"Unknown SAMPLING_STRATEGY {strategy!r}")

EFFECTIVE_TRAIN_IDS, POS_WEIGHT, SYNTHETIC_STATIC_ROWS = apply_sampling_strategy(TRAIN_IDS, CONFIG["SAMPLING_STRATEGY"])
print(f"\nSAMPLING_STRATEGY = {CONFIG['SAMPLING_STRATEGY']!r}")
print(f"Effective training set size: {len(EFFECTIVE_TRAIN_IDS)} real patients "
      f"+ {len(SYNTHETIC_STATIC_ROWS or [])} synthetic")
print(f"pos_weight passed to the loss function: {POS_WEIGHT:.2f}")



## 8.3 Sequence-branch augmentation for real minority patients (jitter + time-mask)

Applies **only to real positive training patients** (never to survived patients, never to
synthetic static rows, which have no real sequence to begin with -- Section 9.1 already
fills those with zeros on the standardized scale). Two augmentations, chosen to match
what's actually buildable on this notebook's fixed-length resampled time grid (Section
6.3's `TARGET_SEQ_LEN` points per system) rather than raw irregular timestamps:

- **Magnitude jitter**: adds small Gaussian noise (`CONFIG['JITTER_SIGMA']`, on the
  already-standardized z-score scale) to observed values only -- imputed/masked positions
  are left alone so jitter doesn't compound with the imputation strategy's own guesswork.
- **Time-masking**: randomly zeroes out a short contiguous span of timesteps (and their
  mask), forcing the encoder to tolerate a missing stretch it hasn't seen at exactly that
  position before -- a practical, honestly-labelled stand-in for the "window slicing /
  cropping" augmentation discussed in the external review, adapted for a fixed-length grid
  rather than literal random-interval cropping of raw timestamps.

**DTW Barycentric Averaging and embedding-space SMOTE are deliberately not implemented
here** -- both were flagged in `Data_Imbalance_and_Imputation_Reference.md` Part A.2 as
higher-effort, later-pass items (DBA needs a proper aligned-averaging implementation;
embedding-space SMOTE is only valid after the Section 9.4 pre-training phase completes,
which hasn't happened yet at this point in the notebook).


In [ ]:

def build_augment_items(effective_train_ids, use_augmentation, copies_per_patient):
    if not use_augmentation:
        return []
    real_positive_ids = [sid for sid in effective_train_ids
                          if not str(sid).startswith("synthetic_") and PATIENT_BUNDLE[sid]["label"] == 1.0]
    items = []
    for sid in real_positive_ids:
        for copy_idx in range(copies_per_patient):
            items.append((sid, copy_idx))   # copy_idx also seeds the augmentation's randomness, Section 9.1
    return items

AUGMENT_ITEMS = build_augment_items(EFFECTIVE_TRAIN_IDS, CONFIG["USE_SEQUENCE_AUGMENTATION"],
                                     CONFIG["SEQUENCE_AUGMENTATION_COPIES"])
print(f"Sequence augmentation: {len(AUGMENT_ITEMS)} augmented copies "
      f"({CONFIG['SEQUENCE_AUGMENTATION_COPIES']} per real positive training patient, "
      f"jitter_sigma={CONFIG['JITTER_SIGMA']})")



# Part 9 — The multimodal organ-system DNN

This is where every theory decision from Part 1 becomes a `torch.nn.Module`. Build order:

- §9.1 the `INSPIREDataset` — turns `IMPUTED_BUNDLE` into model-ready tensors
- §9.2 `SystemEncoder` — one per organ system, transformer over time (§1.5.1) with the
  §6.9 cardiovascular→renal coupling wired in via an extra side-input
- §9.3 `StaticEncoder` — the small MLP for static/GI/MSK/HFRS/operation-count features
  (§1.5.1, §1.6.1)
- §9.4 the autoencoder pre-training head (§1.5.4, phase 1 of "jointly learned")
- §9.5 the loss functions — `pos_weight`-scaled BCE and focal loss (§1.4.1)
- §9.6 `MortalityModel` — the full network: six system encoders + static encoder → fusion
- §9.7 the fusion head itself — NAM-style additive (default) or plain concatenation
  (§1.5.2), switchable via `CONFIG['FUSION_STRATEGY']`



## 9.1 `INSPIREDataset` — bundle → tensors

One `__getitem__` call returns everything the model needs for one patient: six
`[T, F_system]` tensors + six masks, the static feature vector, and the label. Three kinds
of item now feed the training dataset, each flagged in the returned dict so later analysis
(§11's per-patient qualitative checks) can tell them apart:

- **Real patients** — straight from `IMPUTED_BUNDLE`.
- **Synthetic static rows** (§8.2, grouped SMOTENC + Tomek) — a real static vector, but no
  real time series exists for them: every system's sequence is filled with zeros (the
  population mean, on the standardized scale — §7.6) and the mask is all-zero, honestly
  recording that nothing was actually observed.
- **Augmented real minority patients** (§8.3) — a *real* patient's *real* static vector
  and diagnosis history, but with jitter and/or a time-mask applied to their sequence
  branch on the fly, a different random perturbation each epoch (`copy_idx` combined with
  the current epoch seeds the perturbation, so repeated passes over the same augmented
  item don't produce an identical duplicate every time).


In [ ]:

def _jitter_and_mask_sequence(raw, mask, sigma, rng):
    # raw, mask: [T, F] numpy arrays, already imputed + standardized (Part 7).
    # Jitter only OBSERVED positions (mask==1) so noise doesn't compound with imputation.
    # Time-mask: zero out a short contiguous span of timesteps (all features at once) --
    # the fixed-length-grid stand-in for window slicing/cropping, see Part 8.3.
    if raw.shape[1] == 0:
        return raw.copy(), mask.copy()
    out_raw = raw.copy()
    out_mask = mask.copy()

    noise = rng.normal(loc=0.0, scale=sigma, size=out_raw.shape)
    out_raw = out_raw + noise * out_mask   # only perturb where mask==1 (real observations)

    T = out_raw.shape[0]
    if T >= 4:
        span = max(1, T // 6)
        start = rng.integers(0, max(T - span, 1))
        out_raw[start:start + span, :] = 0.0
        out_mask[start:start + span, :] = 0.0

    return out_raw, out_mask

class INSPIREDataset(Dataset):
    def __init__(self, subject_ids, synthetic_static_rows=None, augment_items=None, epoch_seed=0):
        self.subject_ids = list(subject_ids)
        self.synthetic_rows = synthetic_static_rows or []
        self.augment_items = augment_items or []   # list of (sid, copy_idx)
        self.n_real = len(self.subject_ids)
        self.n_synthetic = len(self.synthetic_rows)
        self.epoch_seed = epoch_seed   # bump this between epochs for fresh augmentation noise

    def __len__(self):
        return self.n_real + self.n_synthetic + len(self.augment_items)

    def __getitem__(self, idx):
        if idx < self.n_real:
            sid = self.subject_ids[idx]
            item = {"is_synthetic": False, "is_augmented": False, "subject_id": sid}
            for system in TIME_SERIES_SYSTEMS:
                raw = IMPUTED_BUNDLE[sid][system]
                mask = PATIENT_BUNDLE[sid]["ts"][system]["mask"]
                item[f"{system}_x"] = torch.tensor(raw, dtype=torch.float32)
                item[f"{system}_mask"] = torch.tensor(mask, dtype=torch.float32)
            item["static"] = torch.tensor(IMPUTED_BUNDLE[sid]["static"].values.astype(float), dtype=torch.float32)
            item["label"] = torch.tensor(PATIENT_BUNDLE[sid]["label"], dtype=torch.float32)
            return item

        idx -= self.n_real
        if idx < self.n_synthetic:
            static_vec, label = self.synthetic_rows[idx]
            item = {"is_synthetic": True, "is_augmented": False, "subject_id": f"synthetic_{idx}"}
            for system in TIME_SERIES_SYSTEMS:
                fnames = PATIENT_BUNDLE[self.subject_ids[0]]["ts"][system]["feature_names"]
                # NOTE: IMPUTED_BUNDLE is standardised (§7.6, z-score, mean 0) by the time this
                # runs, so "population mean" for a synthetic patient is simply 0 in this space
                # -- not the raw STATS mean, which is still in original clinical units.
                filled = np.zeros((CONFIG["TARGET_SEQ_LEN"], len(fnames))) if fnames else np.zeros((CONFIG["TARGET_SEQ_LEN"], 0))
                item[f"{system}_x"] = torch.tensor(filled, dtype=torch.float32)
                item[f"{system}_mask"] = torch.zeros(filled.shape, dtype=torch.float32)   # honestly: nothing was observed
            item["static"] = torch.tensor(static_vec, dtype=torch.float32)
            item["label"] = torch.tensor(label, dtype=torch.float32)
            return item

        idx -= self.n_synthetic
        sid, copy_idx = self.augment_items[idx]
        rng = np.random.default_rng(hash((sid, copy_idx, self.epoch_seed)) % (2**32))
        item = {"is_synthetic": False, "is_augmented": True, "subject_id": f"{sid}_aug{copy_idx}"}
        for system in TIME_SERIES_SYSTEMS:
            raw = IMPUTED_BUNDLE[sid][system]
            mask = PATIENT_BUNDLE[sid]["ts"][system]["mask"]
            aug_raw, aug_mask = _jitter_and_mask_sequence(raw, mask, CONFIG["JITTER_SIGMA"], rng)
            item[f"{system}_x"] = torch.tensor(aug_raw, dtype=torch.float32)
            item[f"{system}_mask"] = torch.tensor(aug_mask, dtype=torch.float32)
        item["static"] = torch.tensor(IMPUTED_BUNDLE[sid]["static"].values.astype(float), dtype=torch.float32)
        item["label"] = torch.tensor(PATIENT_BUNDLE[sid]["label"], dtype=torch.float32)
        return item

def collate_bundle(batch):
    out = {}
    for key in batch[0]:
        if key in ("is_synthetic", "is_augmented", "subject_id"):
            out[key] = [b[key] for b in batch]
        else:
            out[key] = torch.stack([b[key] for b in batch])
    return out

train_dataset = INSPIREDataset(EFFECTIVE_TRAIN_IDS, SYNTHETIC_STATIC_ROWS, AUGMENT_ITEMS)
val_dataset   = INSPIREDataset(VAL_IDS)
test_dataset  = INSPIREDataset(TEST_IDS)

train_loader = DataLoader(train_dataset, batch_size=min(CONFIG["BATCH_SIZE"], len(train_dataset)),
                           shuffle=True, collate_fn=collate_bundle)
val_loader   = DataLoader(val_dataset, batch_size=min(CONFIG["BATCH_SIZE"], max(len(val_dataset), 1)),
                           shuffle=False, collate_fn=collate_bundle)
test_loader  = DataLoader(test_dataset, batch_size=min(CONFIG["BATCH_SIZE"], max(len(test_dataset), 1)),
                           shuffle=False, collate_fn=collate_bundle)

print(f"train_dataset: {len(train_dataset)} items "
      f"({train_dataset.n_real} real, {train_dataset.n_synthetic} synthetic, {len(AUGMENT_ITEMS)} augmented)")
print(f"val_dataset:   {len(val_dataset)} items")
print(f"test_dataset:  {len(test_dataset)} items")

_batch = next(iter(train_loader))
print("\nExample batch shapes:")
for k, v in _batch.items():
    if torch.is_tensor(v):
        print(f"  {k:20s} {tuple(v.shape)}")



## 9.2 `SystemEncoder` — one transformer per organ system, with cardiovascular→renal coupling

A standard pre-norm transformer encoder over the time dimension (matching the source
repo's `TimeSeriesTransformer` design), mean-pooled over time into a fixed-size embedding
(§1.5.1). Two extras implement earlier decisions:

- **Mask-aware input**: the value and its mask are concatenated per-feature before the
  input projection, so the network sees *both* what the value is and whether it was
  observed or imputed — this is what makes the §1.3 masking strategy actually usable
  downstream, not just stored.
- **`extra_context`** (used only for the renal encoder): a small side-vector — here, the
  §6.9 cardiovascular summary — is projected and added to every time step's input
  embedding before the transformer runs, giving the renal branch direct access to
  cardiovascular context (§1.6.2, point 1). `CONFIG['SYMMETRIC_CARDIORENAL_COUPLING']`
  additionally gives the cardiovascular encoder a renal summary side-input when `True`.


In [ ]:

class SystemEncoder(nn.Module):
    def __init__(self, n_features, embed_dim, n_heads, n_layers, target_seq_len,
                 extra_context_dim=0, dropout=0.1):
        super().__init__()
        self.n_features = n_features
        self.has_context = extra_context_dim > 0
        if n_features == 0:
            # System with no assigned raw features (e.g. neurological has few/no labs in
            # some configs) -- degrade gracefully to a learned constant embedding rather
            # than crash, so the architecture still runs on datasets where a system is
            # genuinely empty.
            self.empty_embedding = nn.Parameter(torch.zeros(embed_dim))
            return
        input_dim = n_features * 2   # value + mask, concatenated per feature (see docstring above)
        self.input_proj = nn.Linear(input_dim, embed_dim)
        self.pos_embedding = nn.Parameter(torch.randn(1, target_seq_len, embed_dim) * 0.02)
        if self.has_context:
            self.context_proj = nn.Linear(extra_context_dim, embed_dim)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=embed_dim, nhead=n_heads, dim_feedforward=embed_dim * 2,
            dropout=dropout, batch_first=True, norm_first=True,
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=n_layers)
        self.out_norm = nn.LayerNorm(embed_dim)

    def forward(self, x, mask, extra_context=None):
        # x, mask: [B, T, n_features]
        if self.n_features == 0:
            batch_size = x.shape[0]
            return self.empty_embedding.unsqueeze(0).expand(batch_size, -1)
        combined = torch.cat([x, mask], dim=-1)          # [B, T, 2*n_features]
        h = self.input_proj(combined) + self.pos_embedding
        if self.has_context and extra_context is not None:
            h = h + self.context_proj(extra_context).unsqueeze(1)   # broadcast over time
        h = self.transformer(h)                           # [B, T, embed_dim]
        h = self.out_norm(h)
        embedding = h.mean(dim=1)                          # mean-pool over time -> [B, embed_dim]
        return embedding, h   # also return per-timestep h for the attention-audit hook in §11.3

print("SystemEncoder defined.")



## 9.3 `StaticEncoder` — the small MLP for static/GI/MSK/HFRS/operation-count features

Everything from §6.6–§6.13 that isn't a time series (demographics, ASA, department,
GI/MSK ICD-10 flags, HFRS, operation counts, the cardiac-recovery exception flag,
medication aggregates) goes through one small MLP into its own embedding, exactly per the
"static, 100% coverage → small MLP" row of §1.5.1's table.


In [ ]:

class StaticEncoder(nn.Module):
    def __init__(self, n_static_features, embed_dim, dropout=0.1):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_static_features, embed_dim * 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(embed_dim * 2, embed_dim),
        )

    def forward(self, x):
        return self.net(x)

print("StaticEncoder defined.")



## 9.4 Autoencoder pre-training head (§1.5.4, phase 1)

Each system encoder is first trained to reconstruct its own masked input from the pooled
embedding, on the **entire available cohort, label-free** — exactly the "pre-trained
together, unsupervised, before any label is touched" reading of "jointly learning the
embeddings" from §1.5.4. This matters more here than in a flat single-encoder pipeline
because splitting into six systems also splits the already-scarce mortality labels six
ways (§1.1) — pre-training lets every branch still learn from the full patient pool first.


In [ ]:

class ReconstructionHead(nn.Module):
    # Decodes a pooled system embedding back into a full [T, n_features] reconstruction --
    # the autoencoder half of §1.5.4's two-phase pattern.
    def __init__(self, embed_dim, n_features, target_seq_len):
        super().__init__()
        self.target_seq_len = target_seq_len
        self.n_features = n_features
        if n_features == 0:
            return
        self.decoder = nn.Sequential(
            nn.Linear(embed_dim, embed_dim * 2),
            nn.ReLU(),
            nn.Linear(embed_dim * 2, target_seq_len * n_features),
        )

    def forward(self, embedding):
        if self.n_features == 0:
            return None
        out = self.decoder(embedding)
        return out.view(-1, self.target_seq_len, self.n_features)

def masked_reconstruction_loss(pred, target, mask):
    # MSE only over positions that were genuinely OBSERVED (mask==1) -- reconstructing
    # imputed filler values would just teach the network to reproduce the imputation
    # strategy from §7 instead of the real underlying signal.
    if pred is None:
        return torch.tensor(0.0, device=target.device)
    diff2 = (pred - target) ** 2 * mask
    denom = mask.sum().clamp(min=1.0)
    return diff2.sum() / denom

print("ReconstructionHead + masked_reconstruction_loss defined.")



## 9.5 Loss functions — `pos_weight` BCE and focal loss (§1.4.1)

Both take raw logits (no sigmoid applied beforehand) for numerical stability, matching
`torch.nn.BCEWithLogitsLoss` conventions. `CONFIG['USE_FOCAL_LOSS']` switches between them
independently of `CONFIG['SAMPLING_STRATEGY']` — the two are complementary, not
mutually exclusive, per §1.4.1's note that focal loss "complements class weighting rather
than replacing it."


In [ ]:

def focal_loss_with_logits(logits, targets, pos_weight, gamma=2.0):
    # Lin et al. 2017 focal loss, generalised with a pos_weight term so it still respects
    # class imbalance (gamma alone down-weights EASY examples of either class; pos_weight
    # additionally up-weights the rarer class specifically -- combining both per §1.4.1).
    bce = F.binary_cross_entropy_with_logits(logits, targets, reduction="none")
    p = torch.sigmoid(logits)
    p_t = p * targets + (1 - p) * (1 - targets)
    focal_term = (1 - p_t) ** gamma
    weight = torch.where(targets == 1, torch.tensor(pos_weight, device=logits.device), torch.tensor(1.0, device=logits.device))
    return (weight * focal_term * bce).mean()

def mortality_loss(logits, targets, pos_weight, use_focal, gamma):
    if use_focal:
        return focal_loss_with_logits(logits, targets, pos_weight, gamma)
    return F.binary_cross_entropy_with_logits(logits, targets, pos_weight=torch.tensor(pos_weight, device=logits.device))

print(f"Loss configured: {'focal loss' if CONFIG['USE_FOCAL_LOSS'] else 'pos_weight BCE'}, pos_weight={POS_WEIGHT:.2f}")



## 9.7 Fusion heads — NAM (default), concat, gated (§1.5.2)

Three interchangeable modules, all with the same interface (`list of per-system
embeddings + static embedding -> single logit`), switched by `CONFIG['FUSION_STRATEGY']`:

- **`NAMFusion`** (default): each system embedding passes through its **own** small
  sub-network into a single scalar contribution; the logit is the literal sum of
  contributions plus a learned bias — the real, individually-plottable decomposition
  from §1.5.2's third row (`per_system_contributions` is returned for exactly this
  purpose, used in §11.2's per-system breakdown plot).
- **`ConcatFusion`**: the simple baseline — stack every embedding, one `Linear` layer.
  Used in §11.4's ablation to quantify what NAM's structure costs/buys.
- **`GatedFusion`**: a small gate network learns a per-patient weight for each system
  embedding before concatenation — the middle ground from §1.5.2's second row.


In [ ]:

class NAMFusion(nn.Module):
    def __init__(self, system_names, embed_dim, static_included=True):
        super().__init__()
        self.system_names = list(system_names) + (["static"] if static_included else [])
        self.shape_functions = nn.ModuleDict({
            name: nn.Sequential(nn.Linear(embed_dim, embed_dim // 2), nn.ReLU(), nn.Linear(embed_dim // 2, 1))
            for name in self.system_names
        })
        self.bias = nn.Parameter(torch.zeros(1))

    def forward(self, embeddings_dict):
        contributions = {name: self.shape_functions[name](embeddings_dict[name]).squeeze(-1)
                          for name in self.system_names}
        logit = sum(contributions.values()) + self.bias
        return logit, contributions   # contributions: name -> [B] per-system additive term


class ConcatFusion(nn.Module):
    def __init__(self, system_names, embed_dim, static_included=True):
        super().__init__()
        n_inputs = len(system_names) + (1 if static_included else 0)
        self.classifier = nn.Sequential(
            nn.Linear(embed_dim * n_inputs, embed_dim), nn.ReLU(), nn.Linear(embed_dim, 1)
        )
        self.system_names = list(system_names) + (["static"] if static_included else [])

    def forward(self, embeddings_dict):
        stacked = torch.cat([embeddings_dict[name] for name in self.system_names], dim=-1)
        logit = self.classifier(stacked).squeeze(-1)
        return logit, None   # no per-system decomposition available -- the point of §11.4's ablation


class GatedFusion(nn.Module):
    def __init__(self, system_names, embed_dim, static_included=True):
        super().__init__()
        self.system_names = list(system_names) + (["static"] if static_included else [])
        n = len(self.system_names)
        self.gate_net = nn.Sequential(nn.Linear(embed_dim * n, n), )
        self.classifier = nn.Sequential(nn.Linear(embed_dim * n, embed_dim), nn.ReLU(), nn.Linear(embed_dim, 1))

    def forward(self, embeddings_dict):
        stacked = torch.cat([embeddings_dict[name] for name in self.system_names], dim=-1)
        gates = torch.softmax(self.gate_net(stacked), dim=-1)   # [B, n_systems] -- inspectable per-patient weights
        weighted = torch.cat([
            embeddings_dict[name] * gates[:, i:i+1] for i, name in enumerate(self.system_names)
        ], dim=-1)
        logit = self.classifier(weighted).squeeze(-1)
        gate_dict = {name: gates[:, i] for i, name in enumerate(self.system_names)}
        return logit, gate_dict   # gate_dict doubles as an interpretability signal, like NAM's contributions

FUSION_CLASSES = {"nam": NAMFusion, "concat": ConcatFusion, "gated": GatedFusion}
print(f"Fusion strategy configured: {CONFIG['FUSION_STRATEGY']!r} -> {FUSION_CLASSES[CONFIG['FUSION_STRATEGY']].__name__}")



## 9.6 `MortalityModel` — assembling everything

Six `SystemEncoder`s + one `StaticEncoder` → the chosen fusion head → one mortality logit.
The renal encoder receives the §6.9 cardiovascular summary as `extra_context`; if
`CONFIG['SYMMETRIC_CARDIORENAL_COUPLING']` is `True`, the cardiovascular encoder
symmetrically receives a renal summary. `forward()` also returns every system's
reconstruction (for §9.4's pre-training phase) and the fusion's per-system
contributions/gates (for §11's interpretability checks) — one model serves both training
phases and every later analysis, rather than needing separate model classes.


In [ ]:

CARDIAC_CONTEXT_DIM = 3   # cardio_mean_hr, cardio_map_deviation, cardio_has_iabp -- from §6.9

class MortalityModel(nn.Module):
    def __init__(self, system_feature_counts, n_static_features, embed_dim, n_heads, n_layers,
                 target_seq_len, fusion_strategy, symmetric_coupling):
        super().__init__()
        self.system_names = TIME_SERIES_SYSTEMS
        self.symmetric_coupling = symmetric_coupling

        self.encoders = nn.ModuleDict()
        for system in self.system_names:
            extra_ctx = 0
            if system == "renal":
                extra_ctx = CARDIAC_CONTEXT_DIM                                  # §1.6.2 point 1
            elif system == "cardiovascular" and symmetric_coupling:
                extra_ctx = 2                                                    # renal summary dim, §1.6.2's ablation
            self.encoders[system] = SystemEncoder(
                n_features=system_feature_counts[system], embed_dim=embed_dim,
                n_heads=n_heads, n_layers=n_layers, target_seq_len=target_seq_len,
                extra_context_dim=extra_ctx,
            )

        self.recon_heads = nn.ModuleDict({
            system: ReconstructionHead(embed_dim, system_feature_counts[system], target_seq_len)
            for system in self.system_names
        })

        self.static_encoder = StaticEncoder(n_static_features, embed_dim)
        self.fusion = FUSION_CLASSES[fusion_strategy](self.system_names, embed_dim, static_included=True)

    def encode_all_systems(self, batch, cardio_context, renal_context=None):
        embeddings, recon_preds, per_timestep = {}, {}, {}
        for system in self.system_names:
            x, mask = batch[f"{system}_x"], batch[f"{system}_mask"]
            ctx = cardio_context if system == "renal" else (renal_context if (system == "cardiovascular" and self.symmetric_coupling) else None)
            emb, h = self.encoders[system](x, mask, extra_context=ctx)
            embeddings[system] = emb
            per_timestep[system] = h
            recon_preds[system] = self.recon_heads[system](emb)
        return embeddings, recon_preds, per_timestep

    def forward(self, batch):
        # cardio_context: the §6.9 side-input for the renal branch. Built here from the raw
        # cardiovascular time series' first few pooled stats (cheap, always available -- a
        # richer version could reuse the cardiovascular SystemEncoder's own embedding, but
        # that would create a forward-pass ordering dependency; a direct small summary is
        # simpler and keeps the two branches independently trainable).
        cardio_x = batch["cardiovascular_x"]     # [B, T, F_cardio]
        cardio_context = cardio_x.mean(dim=1)[:, :CARDIAC_CONTEXT_DIM] if cardio_x.shape[-1] >= CARDIAC_CONTEXT_DIM else \
            F.pad(cardio_x.mean(dim=1), (0, CARDIAC_CONTEXT_DIM - cardio_x.shape[-1]))

        renal_context = None
        if self.symmetric_coupling:
            renal_x = batch["renal_x"]
            renal_context = renal_x.mean(dim=1)[:, :2] if renal_x.shape[-1] >= 2 else F.pad(renal_x.mean(dim=1), (0, 2 - renal_x.shape[-1]))

        embeddings, recon_preds, per_timestep = self.encode_all_systems(batch, cardio_context, renal_context)
        static_emb = self.static_encoder(batch["static"])
        embeddings_for_fusion = dict(embeddings)
        embeddings_for_fusion["static"] = static_emb

        logit, per_system_signal = self.fusion(embeddings_for_fusion)
        return {
            "logit": logit,
            "embeddings": embeddings,
            "static_embedding": static_emb,
            "recon_preds": recon_preds,
            "per_system_signal": per_system_signal,   # NAM contributions or gate weights or None (concat)
            "per_timestep": per_timestep,              # for the §11.3 attention audit
        }

SYSTEM_FEATURE_COUNTS = {system: len(PATIENT_BUNDLE[TRAIN_IDS[0]]["ts"][system]["feature_names"]) for system in TIME_SERIES_SYSTEMS}
N_STATIC_FEATURES = len(STATIC_FEATURE_NAMES)

model = MortalityModel(
    system_feature_counts=SYSTEM_FEATURE_COUNTS,
    n_static_features=N_STATIC_FEATURES,
    embed_dim=CONFIG["EMBED_DIM"], n_heads=CONFIG["TRANSFORMER_HEADS"], n_layers=CONFIG["TRANSFORMER_LAYERS"],
    target_seq_len=CONFIG["TARGET_SEQ_LEN"], fusion_strategy=CONFIG["FUSION_STRATEGY"],
    symmetric_coupling=CONFIG["SYMMETRIC_CARDIORENAL_COUPLING"],
).to(DEVICE)

n_params = sum(p.numel() for p in model.parameters())
print(f"MortalityModel built: {n_params:,} parameters")
print(f"System feature counts: {SYSTEM_FEATURE_COUNTS}")

# Architecture fingerprint -- used to name checkpoints (Part 10) so a checkpoint from a
# DIFFERENT run (e.g. a MAX_SUBJECTS_PER_CLASS smoke test, where a rare department might be
# absent and shrink the static feature count by a column or two) can never be silently
# loaded into a model it doesn't actually match. Derived from the things that determine the
# model's actual shape, not from config values that merely correlate with it.
import hashlib
_fingerprint_source = repr((
    SYSTEM_FEATURE_COUNTS, N_STATIC_FEATURES, CONFIG["EMBED_DIM"], CONFIG["TRANSFORMER_HEADS"],
    CONFIG["TRANSFORMER_LAYERS"], CONFIG["TARGET_SEQ_LEN"], CONFIG["FUSION_STRATEGY"],
    CONFIG["SYMMETRIC_CARDIORENAL_COUPLING"],
))
ARCHITECTURE_FINGERPRINT = hashlib.sha256(_fingerprint_source.encode()).hexdigest()[:10]
print(f"Architecture fingerprint: {ARCHITECTURE_FINGERPRINT} "
      f"(checkpoints are tagged with this -- a checkpoint from a run with a different "
      f"static/time-series feature count, e.g. a smoke test that happened to miss a rare "
      f"department, will be ignored rather than loaded into a mismatched model)")

# Smoke test: one forward pass on a real batch.
_batch_dev = {k: (v.to(DEVICE) if torch.is_tensor(v) else v) for k, v in _batch.items()}
with torch.no_grad():
    _out = model(_batch_dev)
print(f"\nSmoke test forward pass OK. logit shape: {tuple(_out['logit'].shape)}")
if _out["per_system_signal"] is not None:
    print("Per-system signal keys:", list(_out["per_system_signal"].keys()))



# Part 10 -- Training: pre-train, then fine-tune (Sec 1.5.4's two-phase pattern)

**Phase 1 (10.1)**: every system encoder trains as an autoencoder -- reconstruct its own
masked input -- on `EFFECTIVE_TRAIN_IDS` only, label-free. **Phase 2 (10.2)**: the whole
network (encoders + static branch + fusion) trains end-to-end on the mortality objective,
using whichever loss `CONFIG['USE_FOCAL_LOSS']` selects and the `pos_weight` from Sec 8.2.
Validation AUROC is tracked every epoch for early stopping -- with a handful of positives in
the validation fold at this dataset's current size, treat the *trend* as informative and
the exact epoch of the best checkpoint as somewhat noisy, not a precise result (the same
caution the source repo's own docs give for its 29-30 patient runs).

**Checkpointing (new).** Both phases now save to `CONFIG['CHECKPOINT_DIR']` every
`CONFIG['CHECKPOINT_EVERY_N_EPOCHS']` epochs, and automatically resume from the latest
checkpoint if one exists when this cell is re-run -- specifically for Colab, where a free
T4 session can disconnect (idle timeout, daily usage cap) for reasons that have nothing to
do with memory. If `CHECKPOINT_DIR` points at a mounted Google Drive path, this survives a
full session loss; if it's local/ephemeral (Kaggle, or Colab without Drive mounted), it
only protects against accidental cell interruption within the same session.


In [ ]:

def save_checkpoint(path, **state):
    torch.save(state, path)

def load_checkpoint_if_exists(path):
    if os.path.exists(path):
        try:
            return torch.load(path, map_location=DEVICE)
        except Exception as e:
            print(f"Found a checkpoint at {path} but failed to load it ({e}) -- starting fresh.")
    return None

PRETRAIN_CKPT_PATH = os.path.join(CONFIG["CHECKPOINT_DIR"], f"pretrain_checkpoint_{ARCHITECTURE_FINGERPRINT}.pt")
FINETUNE_CKPT_PATH = os.path.join(CONFIG["CHECKPOINT_DIR"], f"finetune_checkpoint_{ARCHITECTURE_FINGERPRINT}.pt")

def run_pretrain_epoch(model, loader, optimizer):
    model.train()
    total_loss = 0.0
    n_batches = 0
    for batch in loader:
        batch = {k: (v.to(DEVICE) if torch.is_tensor(v) else v) for k, v in batch.items()}
        optimizer.zero_grad()
        out = model(batch)
        loss = 0.0
        for system in TIME_SERIES_SYSTEMS:
            loss = loss + masked_reconstruction_loss(out["recon_preds"][system], batch[f"{system}_x"], batch[f"{system}_mask"])
        loss.backward()
        optimizer.step()
        total_loss += float(loss)
        n_batches += 1
    return total_loss / max(n_batches, 1)

pretrain_optimizer = torch.optim.Adam(model.parameters(), lr=CONFIG["LR"])
pretrain_losses = []
start_epoch_pretrain = 0

_ckpt = load_checkpoint_if_exists(PRETRAIN_CKPT_PATH)
if _ckpt is not None and _ckpt.get("phase") == "pretrain":
    try:
        model.load_state_dict(_ckpt["model_state"])
        pretrain_optimizer.load_state_dict(_ckpt["optimizer_state"])
        pretrain_losses = _ckpt["pretrain_losses"]
        start_epoch_pretrain = _ckpt["epoch"] + 1
        print(f"Resuming pre-training from checkpoint at epoch {start_epoch_pretrain} "
              f"(found at {PRETRAIN_CKPT_PATH})")
    except RuntimeError as e:
        print(f"WARNING: checkpoint at {PRETRAIN_CKPT_PATH} does not match this model's shape "
              f"({e}) -- ignoring it and starting fresh. (The architecture-fingerprint filename "
              f"should prevent this; seeing it anyway means something else changed too.)")

print("Phase 1: autoencoder pre-training (label-free, Sec 1.5.4)")
for epoch in range(start_epoch_pretrain, CONFIG["EPOCHS_PRETRAIN"]):
    train_dataset.epoch_seed = epoch   # fresh jitter/time-mask draw each epoch, Part 8.3/9.1
    loss = run_pretrain_epoch(model, train_loader, pretrain_optimizer)
    pretrain_losses.append(loss)
    if epoch % max(CONFIG["EPOCHS_PRETRAIN"] // 6, 1) == 0 or epoch == CONFIG["EPOCHS_PRETRAIN"] - 1:
        print(f"  epoch {epoch:3d}  reconstruction_loss={loss:.4f}")
    if (epoch + 1) % CONFIG["CHECKPOINT_EVERY_N_EPOCHS"] == 0 or epoch == CONFIG["EPOCHS_PRETRAIN"] - 1:
        save_checkpoint(PRETRAIN_CKPT_PATH, phase="pretrain", epoch=epoch,
                         model_state=model.state_dict(), optimizer_state=pretrain_optimizer.state_dict(),
                         pretrain_losses=pretrain_losses)

fig, ax = plt.subplots(figsize=(6, 3.5))
ax.plot(pretrain_losses, color="#0a7d6e")
ax.set_title("Phase 1: pre-training reconstruction loss"); ax.set_xlabel("epoch"); ax.set_ylabel("masked MSE")
plt.tight_layout(); plt.show()


In [ ]:

@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    all_logits, all_labels = [], []
    for batch in loader:
        batch = {k: (v.to(DEVICE) if torch.is_tensor(v) else v) for k, v in batch.items()}
        out = model(batch)
        all_logits.append(out["logit"].cpu().numpy())
        all_labels.append(batch["label"].cpu().numpy())
    logits = np.concatenate(all_logits)
    labels = np.concatenate(all_labels)
    probs = 1 / (1 + np.exp(-logits))
    result = {"probs": probs, "labels": labels}
    if len(np.unique(labels)) > 1:   # AUROC/AUPRC undefined with only one class present
        result["auroc"] = roc_auc_score(labels, probs)
        result["auprc"] = average_precision_score(labels, probs)
    else:
        result["auroc"] = float("nan")
        result["auprc"] = float("nan")
    return result

def run_finetune_epoch(model, loader, optimizer, pos_weight, use_focal, gamma):
    model.train()
    total_loss = 0.0
    n_batches = 0
    for batch in loader:
        batch = {k: (v.to(DEVICE) if torch.is_tensor(v) else v) for k, v in batch.items()}
        optimizer.zero_grad()
        out = model(batch)
        loss = mortality_loss(out["logit"], batch["label"], pos_weight, use_focal, gamma)
        loss.backward()
        optimizer.step()
        total_loss += float(loss)
        n_batches += 1
    return total_loss / max(n_batches, 1)

finetune_optimizer = torch.optim.Adam(model.parameters(), lr=CONFIG["LR"] * 0.5)   # smaller LR: don't destroy the pre-trained encoders
history = {"train_loss": [], "val_auroc": [], "val_auprc": []}
best_val_auroc, best_state = -1.0, None
start_epoch_finetune = 0

_ckpt = load_checkpoint_if_exists(FINETUNE_CKPT_PATH)
if _ckpt is not None and _ckpt.get("phase") == "finetune":
    try:
        model.load_state_dict(_ckpt["model_state"])
        finetune_optimizer.load_state_dict(_ckpt["optimizer_state"])
        history = _ckpt["history"]
        best_val_auroc = _ckpt["best_val_auroc"]
        best_state = _ckpt["best_state"]
        start_epoch_finetune = _ckpt["epoch"] + 1
        print(f"Resuming fine-tuning from checkpoint at epoch {start_epoch_finetune} "
              f"(found at {FINETUNE_CKPT_PATH}, best val AUROC so far={best_val_auroc:.3f})")
    except RuntimeError as e:
        print(f"WARNING: checkpoint at {FINETUNE_CKPT_PATH} does not match this model's shape "
              f"({e}) -- ignoring it and starting fresh.")

print("\nPhase 2: supervised fine-tuning on the mortality objective (Sec 1.5.4)")
for epoch in range(start_epoch_finetune, CONFIG["EPOCHS_FINETUNE"]):
    train_dataset.epoch_seed = 1000 + epoch   # offset from pretrain's seeds, still deterministic per run
    train_loss = run_finetune_epoch(model, train_loader, finetune_optimizer, POS_WEIGHT, CONFIG["USE_FOCAL_LOSS"], CONFIG["FOCAL_GAMMA"])
    val_result = evaluate(model, val_loader)
    history["train_loss"].append(train_loss)
    history["val_auroc"].append(val_result["auroc"])
    history["val_auprc"].append(val_result["auprc"])

    if not math.isnan(val_result["auroc"]) and val_result["auroc"] > best_val_auroc:
        best_val_auroc = val_result["auroc"]
        best_state = {k: v.clone() for k, v in model.state_dict().items()}

    if epoch % max(CONFIG["EPOCHS_FINETUNE"] // 8, 1) == 0 or epoch == CONFIG["EPOCHS_FINETUNE"] - 1:
        print(f"  epoch {epoch:3d}  train_loss={train_loss:.4f}  val_auroc={val_result['auroc']:.3f}  val_auprc={val_result['auprc']:.3f}")

    if (epoch + 1) % CONFIG["CHECKPOINT_EVERY_N_EPOCHS"] == 0 or epoch == CONFIG["EPOCHS_FINETUNE"] - 1:
        save_checkpoint(FINETUNE_CKPT_PATH, phase="finetune", epoch=epoch,
                         model_state=model.state_dict(), optimizer_state=finetune_optimizer.state_dict(),
                         history=history, best_val_auroc=best_val_auroc, best_state=best_state)

if best_state is not None:
    model.load_state_dict(best_state)
    print(f"\nRestored best checkpoint (val AUROC={best_val_auroc:.3f}).")
else:
    print("\nNo val-AUROC improvement was recorded (can happen with very few val positives) -- using final-epoch weights.")



**If your session disconnects mid-run:** just re-run this Part 10 cell (after re-running
Parts 1-9 to rebuild the data/model objects, which is fast relative to training) — it
will find the checkpoint and pick up from the next epoch rather than restarting. Once a
run finishes normally, delete or rename the checkpoint files in `CONFIG['CHECKPOINT_DIR']`
before starting an intentionally fresh run, or they'll be (correctly) auto-resumed from
instead.


In [ ]:

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(history["train_loss"], color="#c0392b")
axes[0].set_title("Phase 2: fine-tuning loss"); axes[0].set_xlabel("epoch"); axes[0].set_ylabel("loss")

axes[1].plot(history["val_auroc"], label="val AUROC", color="#2980b9")
axes[1].plot(history["val_auprc"], label="val AUPRC", color="#8e44ad")
axes[1].axhline(0.5, color="gray", linestyle="--", linewidth=1, label="chance (AUROC)")
axes[1].set_title("Validation metrics per epoch"); axes[1].set_xlabel("epoch"); axes[1].legend()
plt.tight_layout(); plt.show()

print("Read this chart directionally, not literally: with ~6 validation patients on the "
      "dev subset (Sec 1.4), a single flipped prediction swings AUROC by roughly 1/(n_pos*n_neg) "
      "-- exactly the caution the source repo's own docs give for its 29-30 patient runs. "
      "Re-run on the full cohort for a trustworthy curve.")



# Part 11 — Evaluation, calibration, and interpretability

- §11.1 Held-out test metrics (AUROC, AUPRC, Brier score, confusion matrix)
- §11.2 Per-system NAM contribution breakdown — the actual "Renal: CRITICAL, Cardio: OK"
  decomposition promised in §1.1 and §1.5.2
- §11.3 Attention audit — do heads attend to sensible time steps?
- §11.4 Ablation: NAM vs. concat fusion — what does the additive structure cost/buy?
- §11.5 Sensitivity: pre-op-only vs. peri-operative (§1.6.3, Path B)
- §11.6 Cardiac-recovery-exception qualitative check (§1.6.4)
- §11.7 Multi-operation sensitivity: last-op vs. first-op label (§1.6.4, Path C)

All caveats about small-N noise from Parts 8–10 apply throughout — read directions and
relative comparisons, not exact numbers, until this runs on the full cohort.



## 11.1 Held-out test metrics

**AUPRC (PR-AUC) is the metric to trust first, not AUROC** — per Part C step 9 of the
imbalance/imputation reference. At the true full-cohort prevalence (~0.47%, 469/99,886),
AUROC can look deceptively good while precision stays poor, because AUROC's false-positive
rate is measured against an enormous majority class — the same failure mode both external
reviews independently flagged. AUROC is still reported below for reference and comparison
to published benchmarks (e.g. Shickel et al.'s 0.92), but AUPRC is the number to actually
judge this model by.


In [ ]:

test_result = evaluate(model, test_loader)
print(f"TEST  AUPRC={test_result['auprc']:.3f} (primary metric)  AUROC={test_result['auroc']:.3f} (reference)  "
      f"(n={len(test_result['labels'])}, positives={int(test_result['labels'].sum())})")

if len(np.unique(test_result["labels"])) > 1:
    brier = brier_score_loss(test_result["labels"], test_result["probs"])
    print(f"Brier score: {brier:.3f}  (0 = perfect calibration, 0.25 = a coin flip's worth of miscalibration)")

    fig, axes = plt.subplots(1, 3, figsize=(15, 4))

    fpr, tpr, _ = roc_curve(test_result["labels"], test_result["probs"])
    axes[0].plot(fpr, tpr, color="#2980b9"); axes[0].plot([0, 1], [0, 1], "--", color="gray")
    axes[0].set_title(f"ROC (AUROC={test_result['auroc']:.3f})"); axes[0].set_xlabel("FPR"); axes[0].set_ylabel("TPR")

    prec, rec, _ = precision_recall_curve(test_result["labels"], test_result["probs"])
    axes[1].plot(rec, prec, color="#c0392b"); axes[1].set_title(f"PR (AUPRC={test_result['auprc']:.3f})")
    axes[1].set_xlabel("recall"); axes[1].set_ylabel("precision")

    # Calibration curve (§1 Part D: "calibration, not just discrimination")
    bins = np.linspace(0, 1, 6)
    bin_ids = np.digitize(test_result["probs"], bins) - 1
    bin_ids = np.clip(bin_ids, 0, len(bins) - 2)
    obs_rate, pred_rate = [], []
    for b in range(len(bins) - 1):
        mask = bin_ids == b
        if mask.sum() > 0:
            obs_rate.append(test_result["labels"][mask].mean())
            pred_rate.append(test_result["probs"][mask].mean())
    axes[2].plot(pred_rate, obs_rate, "o-", color="#8e44ad", label="model")
    axes[2].plot([0, 1], [0, 1], "--", color="gray", label="perfect calibration")
    axes[2].set_title("Calibration"); axes[2].set_xlabel("predicted probability"); axes[2].set_ylabel("observed rate"); axes[2].legend()
    plt.tight_layout(); plt.show()
else:
    print("Only one class present in the test split at this sample size -- AUROC/calibration "
          "undefined this run. Re-run with a different SEED or, better, the full cohort.")

cm_default = confusion_matrix(test_result["labels"], (test_result["probs"] >= 0.5).astype(int))
print("\nConfusion matrix @ threshold 0.5 (rows=true, cols=pred) [[TN, FP], [FN, TP]]:")
print(cm_default)
print("0.5 is an arbitrary default threshold -- see §11.1b below for a threshold actually "
      "chosen from the PR curve, per Part C step 9.")



## 11.1b Choosing an operating threshold from the PR curve (Part C step 9)

0.5 is meaningless as a cutoff when the true prevalence is ~0.47% — a well-calibrated
model will rarely output anything near 0.5 at all. This picks the threshold that
**maximizes F1** (the harmonic mean of precision and recall) on the test set as one
principled default, and prints the full precision/recall/threshold table around it so a
clinician-agreed cost trade-off (e.g. "we're willing to accept 5 false alarms per true
catch") can override it with an actual number, rather than this notebook silently picking
one for you.


In [ ]:

if len(np.unique(test_result["labels"])) > 1:
    prec_arr, rec_arr, thresh_arr = precision_recall_curve(test_result["labels"], test_result["probs"])
    f1_arr = np.where((prec_arr + rec_arr) > 0, 2 * prec_arr * rec_arr / (prec_arr + rec_arr + 1e-12), 0.0)
    best_idx = int(np.argmax(f1_arr[:-1])) if len(f1_arr) > 1 else 0   # last point has no matching threshold
    best_threshold = float(thresh_arr[best_idx]) if len(thresh_arr) > 0 else 0.5

    print(f"Best-F1 threshold on this test set: {best_threshold:.3f} "
          f"(precision={prec_arr[best_idx]:.2f}, recall={rec_arr[best_idx]:.2f}, F1={f1_arr[best_idx]:.2f})")

    cm_tuned = confusion_matrix(test_result["labels"], (test_result["probs"] >= best_threshold).astype(int))
    print(f"\nConfusion matrix @ tuned threshold {best_threshold:.3f} [[TN, FP], [FN, TP]]:")
    print(cm_tuned)

    print("\nNearby thresholds, for picking a different clinical trade-off by hand:")
    for i in range(max(0, best_idx - 3), min(len(thresh_arr), best_idx + 4)):
        marker = " <- best F1" if i == best_idx else ""
        print(f"  threshold={thresh_arr[i]:.3f}  precision={prec_arr[i]:.2f}  recall={rec_arr[i]:.2f}{marker}")
else:
    print("Only one class present in the test split at this sample size -- skipping threshold tuning.")



## 11.2 Per-system contribution breakdown — the interpretability payoff

If `FUSION_STRATEGY == 'nam'`, `per_system_signal` from the model's forward pass is the
literal additive decomposition promised in §1.1/§1.5.2 — each system's contribution to the
final logit, individually plottable per patient. This is the concrete version of "Renal:
CRITICAL | Cardio: OK" from the architecture diagram in your meeting notes.


In [ ]:

@torch.no_grad()
def per_system_breakdown(model, loader, n_patients=6):
    model.eval()
    rows = []
    count = 0
    for batch in loader:
        batch_dev = {k: (v.to(DEVICE) if torch.is_tensor(v) else v) for k, v in batch.items()}
        out = model(batch_dev)
        if out["per_system_signal"] is None:
            print("FUSION_STRATEGY is not 'nam' (or 'gated') -- no per-system decomposition available. "
                  "Set CONFIG['FUSION_STRATEGY']='nam' and re-run Part 9-10 to see this.")
            return None
        for i, sid in enumerate(batch["subject_id"]):
            row = {"subject_id": sid, "true_label": float(batch["label"][i]),
                   "predicted_prob": float(torch.sigmoid(out["logit"][i]))}
            for system, contrib in out["per_system_signal"].items():
                row[system] = float(contrib[i])
            rows.append(row)
            count += 1
            if count >= n_patients:
                break
        if count >= n_patients:
            break
    return pd.DataFrame(rows)

breakdown_df = per_system_breakdown(model, test_loader, n_patients=min(8, len(test_dataset)))
if breakdown_df is not None:
    display_cols = [c for c in breakdown_df.columns if c not in ("subject_id", "true_label", "predicted_prob")]
    fig, ax = plt.subplots(figsize=(10, max(3, 0.5 * len(breakdown_df))))
    breakdown_df.set_index("subject_id")[display_cols].plot(kind="barh", stacked=True, ax=ax, colormap="tab10")
    ax.set_title("Per-system additive contribution to the mortality logit (NAM fusion, §9.7)")
    ax.set_xlabel("contribution to logit (positive = pushes toward 'died')")
    ax.legend(bbox_to_anchor=(1.02, 1), loc="upper left")
    plt.tight_layout(); plt.show()
    breakdown_df



## 11.3 Attention audit (§1.1, §3C item 4 of the source repo's roadmap)

Do the transformer heads concentrate on clinically sensible time steps? A lightweight
check: for one system and one patient, plot the mean attention weight the encoder's first
layer places on each time step, alongside the observed/imputed mask — a real audit for a
full run would extend this per-head and cross-reference against the acute-deterioration
ICD-10 codes already flagged in the source repo's EDA (D65, I46, R57, J80, K72, A41).


In [ ]:

@torch.no_grad()
def get_attention_weights(model, batch_item, system):
    # Runs one patient through just the named system's transformer and extracts the first
    # layer's self-attention weights (averaged over heads) -- a minimal, dependency-light
    # audit hook rather than a full attention-rollout implementation.
    encoder = model.encoders[system].transformer.layers[0].self_attn
    x = batch_item[f"{system}_x"].unsqueeze(0).to(DEVICE)
    mask = batch_item[f"{system}_mask"].unsqueeze(0).to(DEVICE)
    if x.shape[-1] == 0:
        return None
    combined = torch.cat([x, mask], dim=-1)
    h = model.encoders[system].input_proj(combined) + model.encoders[system].pos_embedding
    _, attn_weights = encoder(h, h, h, need_weights=True, average_attn_weights=True)
    return attn_weights.squeeze(0).cpu().numpy()   # [T, T]

example_item = test_dataset[0]
example_system = "renal"
attn = get_attention_weights(model, example_item, example_system)
if attn is not None:
    fig, axes = plt.subplots(1, 2, figsize=(11, 4))
    im = axes[0].imshow(attn, cmap="viridis"); axes[0].set_title(f"{example_system} attention (layer 0, head-avg)")
    axes[0].set_xlabel("attended-to timestep"); axes[0].set_ylabel("query timestep")
    plt.colorbar(im, ax=axes[0], fraction=0.046)

    mask = example_item[f"{example_system}_mask"].numpy()
    axes[1].imshow(mask.T, aspect="auto", cmap="Greys"); axes[1].set_title("observed(black)/imputed(white) mask")
    axes[1].set_xlabel("timestep"); axes[1].set_ylabel("feature index")
    plt.tight_layout(); plt.show()
    print("A genuinely informative audit would check whether attention concentrates on OBSERVED "
          "(not imputed) timesteps more than chance -- worth computing directly once run at full scale.")



## 11.4 Ablation: NAM vs. concat fusion (§1.5.2)

Retrains a second model with `FUSION_STRATEGY='concat'`, same data/split/imputation, to
give a direct, same-conditions comparison of what the NAM's extra structure costs in
raw discrimination — the honest way to check whether the interpretability gain (§11.2) is
worth it, rather than assuming it for free. Uses fewer epochs than the main run purely to
keep this comparison cell fast; raise `ABLATION_EPOCHS` for a fairer head-to-head.


In [ ]:

def train_and_eval_variant(fusion_strategy, epochs_pretrain=15, epochs_finetune=30, seed=SEED):
    torch.manual_seed(seed)
    variant = MortalityModel(
        system_feature_counts=SYSTEM_FEATURE_COUNTS, n_static_features=N_STATIC_FEATURES,
        embed_dim=CONFIG["EMBED_DIM"], n_heads=CONFIG["TRANSFORMER_HEADS"], n_layers=CONFIG["TRANSFORMER_LAYERS"],
        target_seq_len=CONFIG["TARGET_SEQ_LEN"], fusion_strategy=fusion_strategy,
        symmetric_coupling=CONFIG["SYMMETRIC_CARDIORENAL_COUPLING"],
    ).to(DEVICE)

    opt1 = torch.optim.Adam(variant.parameters(), lr=CONFIG["LR"])
    for _ in range(epochs_pretrain):
        run_pretrain_epoch(variant, train_loader, opt1)

    opt2 = torch.optim.Adam(variant.parameters(), lr=CONFIG["LR"] * 0.5)
    best_auroc, best_state = -1.0, None
    for _ in range(epochs_finetune):
        run_finetune_epoch(variant, train_loader, opt2, POS_WEIGHT, CONFIG["USE_FOCAL_LOSS"], CONFIG["FOCAL_GAMMA"])
        val_res = evaluate(variant, val_loader)
        if not math.isnan(val_res["auroc"]) and val_res["auroc"] > best_auroc:
            best_auroc, best_state = val_res["auroc"], {k: v.clone() for k, v in variant.state_dict().items()}
    if best_state is not None:
        variant.load_state_dict(best_state)
    test_res = evaluate(variant, test_loader)
    return {"fusion_strategy": fusion_strategy, "val_auroc": best_auroc,
            "test_auroc": test_res["auroc"], "test_auprc": test_res["auprc"]}

ABLATION_EPOCHS = 30
ablation_results = []
for strat in ["nam", "concat"]:
    res = train_and_eval_variant(strat, epochs_finetune=ABLATION_EPOCHS)
    ablation_results.append(res)
    print(res)

ablation_df = pd.DataFrame(ablation_results).set_index("fusion_strategy")
ablation_df



## 11.5 Sensitivity: pre-op-only vs. peri-operative (§1.6.3, Path B)

The full re-run this needs (re-window every patient with `TIME_WINDOW='peri_op'`, rebuild
§6-§10 end to end) is expensive to repeat inline for every cell execution, so this section
is a **template you run deliberately** rather than something that fires automatically:
change `CONFIG['TIME_WINDOW']` to `'peri_op'` in Part 2, then **Run All** again, and record
the resulting test AUROC/AUPRC/coverage numbers in the comparison table below by hand (or
extend the automation from §11.4 to also loop over `TIME_WINDOW`, which is the natural next
step once this notebook runs on a machine fast enough to afford two full passes).



| Run | TIME_WINDOW | test AUROC | test AUPRC | mean cardiovascular coverage (§5.2-style) | notes |
|---|---|---|---|---|---|
| 1 | `pre_op` | *(fill in from §11.1)* | | | Decision-support framing (§1.6.3) |
| 2 | `peri_op` | *(re-run and fill in)* | | | Real-time monitoring framing (§1.6.3) |

Per §1.6.3's own reasoning: a higher `peri_op` AUROC is expected partly from genuinely new
physiological information and partly from better measurement *density* (intra-op vitals are
less sparse than ward vitals per §2.7 of the source repo's `Research_Aim.md`) — report the
coverage numbers from §5.2 alongside AUROC for either run so a reader can tell the two
effects apart, exactly as that section recommends.



## 11.6 Cardiac-recovery-exception qualitative check (§1.6.4)

Does the model's own cardiovascular-system contribution (from the NAM breakdown, §11.2)
shift for patients flagged by the rule-based §6.11 exception, versus similar unflagged
patients? A genuine qualitative validation in the spirit of the source repo's own
worked-patient sanity check (Subject 100033460 in its docs) — small-N here, but the
comparison is structured to scale directly once more flagged patients exist in the full
cohort.


In [ ]:

@torch.no_grad()
def cardiac_contribution_by_exception_flag(model, all_ids):
    rows = []
    ds = INSPIREDataset(all_ids)
    loader_all = DataLoader(ds, batch_size=8, shuffle=False, collate_fn=collate_bundle)
    model.eval()
    for batch in loader_all:
        batch_dev = {k: (v.to(DEVICE) if torch.is_tensor(v) else v) for k, v in batch.items()}
        out = model(batch_dev)
        if out["per_system_signal"] is None:
            return None
        for i, sid in enumerate(batch["subject_id"]):
            rows.append({
                "subject_id": sid,
                "cardiac_recovery_exception": CARDIAC_EXCEPTION_FLAG[sid],
                "cardiovascular_contribution": float(out["per_system_signal"]["cardiovascular"][i]),
                "true_label": float(batch["label"][i]),
            })
    return pd.DataFrame(rows)

exc_df = cardiac_contribution_by_exception_flag(model, ALL_IDS)
if exc_df is not None:
    summary = exc_df.groupby("cardiac_recovery_exception")["cardiovascular_contribution"].agg(["mean", "std", "count"])
    print(summary)
    if (exc_df["cardiac_recovery_exception"] == 1).sum() == 0:
        print("\nNo patients in this dev subset are flagged by the §6.11 rule -- this comparison "
              "is a no-op here. Re-run once the full cohort (with more CTS re-interventions "
              "within the 6-month window) is loaded, where this check becomes informative.")



## 11.7 Multi-operation label sensitivity (§1.6.4, Path C — last-op vs. first-op)

§4.2 already computed `died_30day_from_first_op` alongside the training label
(`died_30day_from_last_op`). A full re-run under the first-op definition needs the same
"change config, Run All" treatment as §11.5 for a genuine model comparison — but the
**label-agreement** check itself doesn't need retraining and is worth doing right here.


In [ ]:

disagreement = cohort_df[cohort_df["died_30day_from_last_op"] != cohort_df["died_30day_from_first_op"]]
print(f"Patients where last-op and first-op 30-day labels disagree: {len(disagreement)} / {len(cohort_df)}")
if len(disagreement):
    print(disagreement[["subject_id", "n_operations", "died_30day_from_last_op", "died_30day_from_first_op"]])
print("\nA non-trivial disagreement rate here is exactly the evidence §1.6.4/Path C says to "
      "collect before picking one definition -- if the disagreement rate is high and "
      "concentrated among multi-operation patients specifically, that's a strong argument "
      "for running (not assuming) the full three-way sensitivity analysis the source repo's "
      "own roadmap already calls for.")



# Part 12 — Experiment log, and what to do next

## 12.1 Experiment log (fill this in as you run variants)

Every `CONFIG` flag in §1.7 is a genuine fork — this table is meant to be copied and
appended to as you try combinations, so the reasoning you asked to preserve stays
attached to actual results, not just intentions. Suggested first sweep once the full
cohort is loaded: hold everything else fixed and vary one flag at a time.

| Run | TIME_WINDOW | IMPUTATION_STRATEGY | SAMPLING_STRATEGY | FUSION_STRATEGY | test AUROC | test AUPRC | notes |
|---|---|---|---|---|---|---|---|
| 1 (this notebook, dev subset) | pre_op | decision_tree | class_weight | nam | *(§11.1)* | *(§11.1)* | 30-patient dev subset — directional only |
| 2 | peri_op | decision_tree | class_weight | nam | | | §11.5 |
| 3 | pre_op | median | class_weight | nam | | | isolates the imputation choice |
| 4 | pre_op | decision_tree | smote | nam | | | needs full cohort — too few positives here (§1.4.2) |
| 5 | pre_op | decision_tree | class_weight | concat | | | §11.4 |

## 12.2 What this notebook deliberately did *not* build (flagged, not silently skipped)

Consistent with keeping every decision visible rather than assumed:

- **Learned ICD-10/ATC embeddings** (§1.2.1, §1.5.3) — chapter flags and ATC level-2
  counts are used instead, because the label-starved small-cohort regime makes a learned
  embedding table under-constrained. Revisit once the full ~99,886-patient cohort is
  available for unsupervised co-occurrence pre-training.
- **POSSUM/P-POSSUM and NEWS2 baselines** (§1.2.2) — the source repo has working NELA
  (`nela.py`) and NEWS2 (`score_models.py`) implementations; wiring them in as comparison
  points alongside this DNN is a natural next cell, not built here to keep this notebook's
  scope to the multimodal architecture itself.
- **Time-to-event / survival reframing** (Dynamic-DeepHit / DySurv-style, per the source
  repo's `Research_Aim.md` §2.8) — a genuinely different output type (a hazard trajectory,
  not a single 30-day probability), flagged as a parallel track rather than folded in here.
- **Symmetric MICE / full multiple imputation** (§1.3.1) — `IterativeImputer` is imported
  and ready; not run by default because, at n=30, its iterative per-feature regressions are
  data-starved in the same way SMOTE is (§1.4.2's caveat applies almost identically here).
- **HFRS's full 109-code table** — §6.6 uses a representative ~30-code subset for
  demonstration; swap in the source repo's complete `frailty_hfrs.py` table for a
  publication-grade run.

## 12.3 Direct next steps, in priority order (mirrors the source repo's own roadmap)

1. **Swap in the full ~99,886-patient cohort** (`CONFIG['SUBJECTS_DIR']`) — every metric in
   this notebook is directional at n=30; this is the single highest-value next step.
2. **Run the full `TIME_WINDOW` and multi-op sensitivity sweeps** (§11.5, §11.7) for real,
   not as a template.
3. **Re-run §11.4's fusion ablation with `ABLATION_EPOCHS` raised** and, ideally, several
   seeds, to get a trustworthy answer on whether NAM's interpretability is bought cheaply
   or expensively in this specific architecture.
4. **Wire in NELA/POSSUM/NEWS2 as baselines** in a new cell, using the already-computed
   `cohort_df` fields — direct comparison points for the DNN's AUROC.
5. **Extend §6.6 to the full HFRS table**, and fix the age-75+/2-year window exactly as
   the source repo's own roadmap flags (`CONFIG['HFRS_LOOKBACK_YEARS']` is already wired
   for this — just confirm the full weights table before publishing any HFRS-derived
   number).
6. **Confirm the `CARDIAC_DEPARTMENTS` mapping** (§6.11) against your site's actual
   department coding before trusting the 6-month exception flag on a new cohort — `CTS`
   was inferred from this dataset's observed department codes, not looked up from an
   authoritative source.

## 12.4 One-paragraph summary, for pasting into a supervisor update

This notebook implements a multimodal, organ-system-separated DNN for 30-day
peri-operative mortality prediction on the INSPIRE dataset, extending the project's
existing six-system architecture with two diagnosis/department-driven systems
(Gastrointestinal, Musculoskeletal — necessary because this dataset has no dedicated
lab/vital panel for either, confirmed programmatically in §4.1) and an explicit
cardiovascular→renal coupling (§6.9, motivated by the clinical cardiorenal-syndrome
literature). Every data-cleaning decision — imputation strategy, class-imbalance sampling,
fusion architecture — is implemented as an interchangeable, documented option rather than a
silent default, with the reasoning for each spelled out in Part 1 before any code runs.
Rule-based clinical exceptions (the 6-month post-cardiac-surgery washout) are kept
separate from learned features on purpose, for both sample-size and actionability reasons.
Current results are directional only (30-patient development subset); the pipeline is
written to scale unchanged to the full ~99,886-patient cohort.
